In [1]:
import os
import gc
import math
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from model_tools.optimization.anti_overfitting_tuner import AntiOverfittingTuner
from model_tools.optimization.experiment_pipeline import run_experiment_batch
from sklearn.linear_model import LogisticRegression
from model_tools.optimization.objectives import KSObjective,AUCObjective
from model_tools.evaluation.delivery_report import ModelDeliveryReport

# 分数写入
from model_tools.utils import write_model_scores
from model_tools.utils import whos
from model_tools.utils import read_model_scores

In [2]:
def Prob2Score(prob, basePoint, PDO):
    y = np.log(prob/(1-prob))
    return basePoint+PDO/np.log(2)*(-y)


def AdjustScore(prob, min_score, max_score, basePoint, PDO):
    if prob <= 0 or prob >= 1:
        return 0

    score = Prob2Score(prob, basePoint, PDO)
    if score < min_score:
        score = min_score
    if score > max_score:
        score = max_score

    score_new = int(round((score-min_score)*600.0/(max_score - min_score) + 300,0))
    if score_new < 300:
        score_new = 300
    if score_new > 900:
        score_new = 900
    return int(score_new)

# label

In [11]:
df_label = pd.read_csv('/Users/mayongzhi/Job/Job_xyd/数据/蜜蜂分/2-建模样本_v10-update-补充华通星选y-新增b066至b069-a077分.csv')
df_label.head()

,1-阿里反欺诈v2标品,B068_prob,B069_prob,apply_date,b058_bh_prob,b058_bh_prob_new_export,b066_bh_model_prob,b066_prob,b066_score,b067_bh_prob,...,蜜蜂分_中利率版8_3_proba_a075,蜜蜂分_中利率版_1,蜜蜂分_中利率版_1_proba,蜜蜂分_高利率版10_proba(B058),蜜蜂分_高利率版5_2,蜜蜂分_高利率版5_2_proba,蜜蜂分_高利率版6_2,蜜蜂分_高利率版6_2_proba,蜜蜂分_高利率版7_2,蜜蜂分_高利率版7_2_proba
0,NaN,NaN,NaN,20220910,0.108168,0.108168,0.030210,0.368135,702.0,NaN,...,NaN,790.0,0.163839,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,20220910,0.081587,0.081587,0.033117,0.399353,692.0,NaN,...,NaN,757.0,0.191322,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,20220930,0.000000,0.000000,0.009372,0.116548,814.0,NaN,...,NaN,720.0,0.225587,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,20220910,0.066016,0.066016,0.023768,0.591837,634.0,NaN,...,NaN,564.0,0.412577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,20220924,0.067988,0.067988,0.024111,0.369848,701.0,NaN,...,NaN,694.0,0.252113,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_label.shape

(6461455, 50)

In [5]:
df_label.columns

Index(['1-阿里反欺诈v2标品', 'B068_prob', 'B069_prob', 'apply_date', 'b058_bh_prob',
       'b058_bh_prob_new_export', 'b066_bh_model_prob', 'b066_prob',
       'b066_score', 'b067_bh_prob', 'b067_prob', 'bh_imbank_proba(B058)',
       'c031_bh_prob', 'enc_method', 'flag', 'high_sample_type',
       'hl_jianpu_score_v1', 'hl_yuxin_score_v3_a', 'hl_yuxin_score_v3_b',
       'hl_yuxin_score_v3_c', 'id_number', 'mid_sample_type', 'mobile',
       'model_risk_v9_gtongyong_score', 'model_risk_v9_gxiaojin_score',
       'partner_code', 'partner_name', 'pric_type_code', 'pric_type_desc',
       'test_id', 'ym_A077', '蜜蜂分_中利率版2_2', '蜜蜂分_中利率版2_2_proba', '蜜蜂分_中利率版6_2',
       '蜜蜂分_中利率版6_2_proba', '蜜蜂分_中利率版6_3_proba', '蜜蜂分_中利率版7_2',
       '蜜蜂分_中利率版7_2_proba', '蜜蜂分_中利率版8_1_proba_a073', '蜜蜂分_中利率版8_2_proba_a074',
       '蜜蜂分_中利率版8_3_proba_a075', '蜜蜂分_中利率版_1', '蜜蜂分_中利率版_1_proba',
       '蜜蜂分_高利率版10_proba(B058)', '蜜蜂分_高利率版5_2', '蜜蜂分_高利率版5_2_proba',
       '蜜蜂分_高利率版6_2', '蜜蜂分_高利率版6_2_proba', '蜜蜂分_高利率版7_2', '

# embedding

In [41]:
df_embeddings = pd.read_csv('/Users/mayongzhi/Job/Job_xyd/数据/客户测建模/华通/回溯结果-华通银行（阿里的结果与原样本的顺序一致）/embedding.csv')
print(df_embeddings.shape)
df_embeddings.head()

(128769, 86)


,serie_no,id_card_md5,phonenum_md5(param),app_date,code,hxe1,hxe10,hxe11,hxe12,hxe13,...,hxe74,hxe75,hxe76,hxe77,hxe78,hxe79,hxe8,hxe80,hxe9,hxs1
0,18257,baf799226f4ecc5e6e0155c51cec94ce,0eff48907cbf6c40e1631941d344a123,20260331,0,0.000000,0.0,0.144249,0.0,0.0,...,0.382729,0.0,0.511470,0.0,0.0,0.0,0.0,0.0,0.288778,0.055253
1,1863,2ba1bb299ebba228bd46f85f3c7e4835,107cf7ab8f4371239c326bfd91ab76cc,20260404,0,0.025855,0.0,0.144249,0.0,0.0,...,0.423831,0.0,0.708697,0.0,0.0,0.0,0.0,0.0,0.372183,0.045963
2,42351,c706ac557deafcdad6b2650c5f1b48a9,1d9606a31ea40cdc406640b6bab13034,20260310,0,0.000000,0.0,0.122617,0.0,0.0,...,0.328287,0.0,0.448501,0.0,0.0,0.0,0.0,0.0,0.266015,0.059765
3,14423,e5d7023d318bdb39b1556849266e4d2d,1e23d36c81dc673db857661f62ba701a,20260402,0,0.415383,0.0,0.000000,0.0,0.0,...,0.353615,0.0,0.545343,0.0,0.0,0.0,0.0,0.0,0.302708,0.059050
4,3061,41849b7da94584575f67ffd3f36ed2b9,210dfeec094cf2c5b7980e82a845ffc9,20260326,0,0.254720,0.0,0.222043,0.0,0.0,...,0.442660,0.0,0.819071,0.0,0.0,0.0,0.0,0.0,0.419072,0.031940


# V4

In [7]:
df_v4 = pd.DataFrame()
v4_file_path = '/Users/mayongzhi/Job/Job_xyd/数据/客户测建模/华通/回溯结果-华通银行（阿里的结果与原样本的顺序一致）/平台特征V4/'
for file in os.listdir(v4_file_path):
    tmp_df = pd.read_csv(v4_file_path+file)
    df_v4 = pd.concat([df_v4, tmp_df], axis=0)

print(df_v4.shape)
del tmp_df
gc.collect()
whos(globals(), top=5)

(128769, 267)
        Variable       Type          Shape  Memory(MB)
0       df_label  DataFrame  (6461455, 50)     6481.21
1          df_v4  DataFrame  (128769, 267)      283.19
2  df_embeddings  DataFrame   (128769, 86)      104.38
3             _5      Index          (50,)        0.01
4             _3  DataFrame        (5, 50)        0.01


,Variable,Type,Shape,Memory(MB)
0,df_label,DataFrame,"(6461455, 50)",6481.21
1,df_v4,DataFrame,"(128769, 267)",283.19
2,df_embeddings,DataFrame,"(128769, 86)",104.38
3,_5,Index,"(50,)",0.01
4,_3,DataFrame,"(5, 50)",0.01


In [8]:
print(df_v4.shape)
df_v4.head()

(128769, 267)


,serie_no,id_card_md5,phonenum_md5(param),app_date,code,hce1,hce10,hce11,hce12,hce13,...,hcu90,hcu91,hcu92,hcu93,hcu94,hcu95,hcu96,hcu97,hcu98,hcu99
0,115271,07da4c3fd092ded002b05f9fc5aa5bc6,46ba7127eb066a294a0b2650121ec2dc,20260303,0,25.0,1.0,1.0,1.0,26.0,...,0.4052,-0.5925,0.3472,0.4038,0.1163,0.2480,0.1927,0.2868,0.5805,0.7129
1,115485,73c5997777637b50d6404c0b47fe67c0,600d511e67898e75f62f029e8684e361,20260307,0,1.0,12.0,8.0,13.0,1.0,...,0.1363,-0.5467,0.3967,0.1198,-0.2310,0.7244,0.6689,0.0868,-0.4255,-1.2142
2,1762,b87cddbd1c1a3915ad54d74ba4b1da14,63eaf8e8acf8bb1f50eb2f682776a8be,20260411,0,1.0,23.0,1.0,1.0,1.0,...,0.9347,0.3533,0.8628,-0.2677,0.2849,-0.3632,0.8087,0.2362,-0.1343,-0.6054
3,85625,91c415b92a9623d66eb18ad81d75cbf9,678666ecc4d120dca97c3cabd2b8384e,20260309,0,1.0,1.0,21.0,12.0,1.0,...,1.5560,0.8355,1.3419,-0.8287,0.8122,-0.9390,1.1026,0.7011,0.1740,-0.7112
4,100601,883a49f67572dd74f5ceaf6041ee051f,71477d17dd9e3e726eb56df0b2d638da,20260304,0,19.0,1.0,19.0,1.0,19.0,...,-0.0781,-1.0140,-0.5541,0.8859,-0.2389,0.9382,0.0613,0.4289,0.5781,0.3156


# V3

In [9]:
df_v3 = pd.read_csv('/Users/mayongzhi/Job/Job_xyd/数据/客户测建模/华通/回溯结果-华通银行（阿里的结果与原样本的顺序一致）/V3.csv')
print(df_v3.shape)
df_v3.head()

(128769, 283)


,serie_no,id_card_md5,phonenum_md5(param),app_date,code,F1,F10,F100,F101,F102,...,zf03,zf04,zf05,zf06,zf07,zf08,zf09,zf10,zf11,zf12
0,112574,d0cbaef1155a8dc254676780ddf559c7,3ad6a5a2a8ea1053a999ccaabb8f6634,20260310,0,0.261534,0.146508,-0.628364,0.261071,-0.623853,...,0.456545,0.426282,0.437528,0.411844,0.427579,0.458387,0.451413,0.433768,0.443950,0.428845
1,26176,e6f48b600fc0df6786c4042287aa82ce,48fc0b54bf7d98cc6df3b0ec67996f4a,20260404,0,0.275184,0.146508,-0.628364,0.514357,-0.644022,...,0.469830,0.483494,0.469509,0.446273,0.462855,0.489139,0.450895,0.452956,0.404786,0.454153
2,102825,2ef594a4b164b06bcb85380ad7d716a8,5aadde4aa6a8d1d742eb2b23eb5131ec,20260304,0,0.474636,0.234870,-0.628364,0.305006,-0.644022,...,0.435237,0.442429,0.450786,0.438774,0.475516,0.449667,0.450875,0.450514,0.435617,0.445040
3,8647,1de9305f9ef1c7b5fdaa5d07eb0ae933,a485b96a031d60cf21a3d07fbc924390,20260308,0,0.221125,0.146508,-0.628364,0.201143,-0.644022,...,0.451709,0.408556,0.523679,0.440101,0.510474,0.479021,0.477343,0.443368,0.438408,0.490325
4,1992,aafd25625f1cb124fa0f6951c87da5e1,54f38bc84542555b759e826ae7bef1ab,20260407,0,0.247603,0.146508,-0.628364,0.407497,-0.644022,...,0.430500,0.444788,0.502083,0.461896,0.479571,0.454140,0.445551,0.456885,0.431374,0.459664


In [11]:
df_v3['month']= df_v3['app_date'].astype(str).str[0:6]
df_v3['month'].value_counts()

month
202603    61842
202604    55514
202602    11251
202605      162
Name: count, dtype: int64

In [12]:
df_v3['partner_name'] =df_v3['month'].map(
    {'202602':'华通-2026-02',
    '202603':'华通-2026-03',
    '202604':'华通-2026-04至05',
    '202605':'华通-2026-04至05'}
    )
df_v3['mid_sample_type'] = df_v3['partner_name'].map(
    {'华通-2026-02':'1-train',
    '华通-2026-03':'1-train',
    '华通-2026-04至05':'3-oot'}
    )

In [13]:
print(df_v3.shape)
df_v3 = df_v3.merge(df_label[df_label['partner_name']=='华通银行-星选'][['id_number','flag']],left_on='id_card_md5',right_on='id_number',how='inner')

(128769, 286)


In [14]:
df_v3.head()

,serie_no,id_card_md5,phonenum_md5(param),app_date,code,F1,F10,F100,F101,F102,...,zf08,zf09,zf10,zf11,zf12,month,partner_name,mid_sample_type,id_number,flag
0,112574,d0cbaef1155a8dc254676780ddf559c7,3ad6a5a2a8ea1053a999ccaabb8f6634,20260310,0,0.261534,0.146508,-0.628364,0.261071,-0.623853,...,0.458387,0.451413,0.433768,0.443950,0.428845,202603,华通-2026-03,1-train,d0cbaef1155a8dc254676780ddf559c7,0.0
1,26176,e6f48b600fc0df6786c4042287aa82ce,48fc0b54bf7d98cc6df3b0ec67996f4a,20260404,0,0.275184,0.146508,-0.628364,0.514357,-0.644022,...,0.489139,0.450895,0.452956,0.404786,0.454153,202604,华通-2026-04至05,3-oot,e6f48b600fc0df6786c4042287aa82ce,0.0
2,102825,2ef594a4b164b06bcb85380ad7d716a8,5aadde4aa6a8d1d742eb2b23eb5131ec,20260304,0,0.474636,0.234870,-0.628364,0.305006,-0.644022,...,0.449667,0.450875,0.450514,0.435617,0.445040,202603,华通-2026-03,1-train,2ef594a4b164b06bcb85380ad7d716a8,0.0
3,8647,1de9305f9ef1c7b5fdaa5d07eb0ae933,a485b96a031d60cf21a3d07fbc924390,20260308,0,0.221125,0.146508,-0.628364,0.201143,-0.644022,...,0.479021,0.477343,0.443368,0.438408,0.490325,202603,华通-2026-03,1-train,1de9305f9ef1c7b5fdaa5d07eb0ae933,0.0
4,1992,aafd25625f1cb124fa0f6951c87da5e1,54f38bc84542555b759e826ae7bef1ab,20260407,0,0.247603,0.146508,-0.628364,0.407497,-0.644022,...,0.454140,0.445551,0.456885,0.431374,0.459664,202604,华通-2026-04至05,3-oot,aafd25625f1cb124fa0f6951c87da5e1,0.0


# 建模

## V3

In [16]:
feas = df_v3.columns[5:-5].to_list()
feas

['F1',
 'F10',
 'F100',
 'F101',
 'F102',
 'F103',
 'F104',
 'F105',
 'F106',
 'F107',
 'F108',
 'F109',
 'F11',
 'F110',
 'F111',
 'F112',
 'F113',
 'F114',
 'F115',
 'F116',
 'F117',
 'F118',
 'F119',
 'F12',
 'F120',
 'F121',
 'F122',
 'F123',
 'F124',
 'F125',
 'F126',
 'F127',
 'F128',
 'F13',
 'F14',
 'F15',
 'F16',
 'F17',
 'F18',
 'F19',
 'F2',
 'F20',
 'F21',
 'F22',
 'F23',
 'F24',
 'F25',
 'F26',
 'F27',
 'F28',
 'F29',
 'F3',
 'F30',
 'F31',
 'F32',
 'F33',
 'F34',
 'F35',
 'F36',
 'F37',
 'F38',
 'F39',
 'F4',
 'F40',
 'F41',
 'F42',
 'F43',
 'F44',
 'F45',
 'F46',
 'F47',
 'F48',
 'F49',
 'F5',
 'F50',
 'F51',
 'F52',
 'F53',
 'F54',
 'F55',
 'F56',
 'F57',
 'F58',
 'F59',
 'F6',
 'F60',
 'F61',
 'F62',
 'F63',
 'F64',
 'F65',
 'F66',
 'F67',
 'F68',
 'F69',
 'F7',
 'F70',
 'F71',
 'F72',
 'F73',
 'F74',
 'F75',
 'F76',
 'F77',
 'F78',
 'F79',
 'F8',
 'F80',
 'F81',
 'F82',
 'F83',
 'F84',
 'F85',
 'F86',
 'F87',
 'F88',
 'F89',
 'F9',
 'F90',
 'F91',
 'F92',
 'F93',
 'F9

In [17]:
df_v3['flag'].value_counts()

flag
0.0    125109
1.0      3660
Name: count, dtype: int64

In [41]:
tuner.optimize_anti_overfitting??

Signature:
tuner.optimize_anti_overfitting(
    X,
    y,
    objective_function: Union[model_tools.optimization.objectives.ObjectiveFunction, NoneType] = None,
    n_trials: int = 100,
    cv_folds: int = 5,
    n_jobs: int = 1,
    show_progress_bar: bool = True,
    weight_col: Union[str, NoneType] = None,
) -> Dict[str, Any]
Source:   
    def optimize_anti_overfitting(
        self,
        X,
        y,
        objective_function: Optional[ObjectiveFunction] = None,
        n_trials: int = 100,
        cv_folds: int = 5,
        n_jobs: int = 1,
        show_progress_bar: bool = True,
        weight_col: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        执行防过拟合优化（使用纯交叉验证）

        Args:
            X: 特征数据
            y: 目标变量
            objective_function: 目标函数，如None则使用AUC
            n_trials: 试验次数
            cv_folds: 交叉验证折数
            n_jobs: 并行数
            show_progress_bar: 是否显示进度条
            weight_col: 权重列名，如果传入则使用该列作为样本权重

        Returns:
            

In [43]:
AntiOverfittingTuner??

Init signature:
AntiOverfittingTuner(
    model_class,
    model_type: str,
    direction: str = 'maximize',
    study_name: Union[str, NoneType] = None,
    max_auc_gap: float = 0.05,
    max_ks_gap: float = 0.03,
    overfitting_penalty_weight: float = 2.0,
    use_regularization_focused_space: bool = True,
    early_stopping_patience: int = 10,
)
Source:        
class AntiOverfittingTuner(HyperparameterTuner):
    """
    防过拟合超参数调优器

    核心功能:
    1. 验证集性能监控
    2. 过拟合惩罚机制
    3. 正则化导向的参数空间
    4. 早停机制
    """

    def __init__(
        self,
        model_class,
        model_type: str,
        direction: str = 'maximize',
        study_name: Optional[str] = None,
        # 防过拟合参数
        max_auc_gap: float = 0.05,
        max_ks_gap: float = 0.03,
        overfitting_penalty_weight: float = 2.0,
        use_regularization_focused_space: bool = True,
        early_stopping_patience: int = 10
    ):
        super().__init__(
            model_class=model_class,
            model_typ

In [19]:
tuner =  AntiOverfittingTuner(
    model_class=lgb.LGBMClassifier,
    model_type='lgb',
    direction='maximize'
)

params_grid_lgb = tuner.optimize_anti_overfitting(
    df_v3[df_v3['mid_sample_type']=='1-train'][feas],
    df_v3[df_v3['mid_sample_type']=='1-train']['flag'],
    n_trials=30
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [20]:
pd.DataFrame(params_grid_lgb['performance_history']).to_csv('params_log_v3.csv',index=False)

In [21]:
params_grid_lgb_v3 = pd.read_csv('params_log_v3.csv')
params_grid_lgb_v3 = params_grid_lgb_v3[params_grid_lgb_v3['is_overfitting']==False]
params_grid_lgb_v3

,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
7,7,"{'boosting_type': 'dart', 'num_leaves': 13, 'm...",0.535170,0.505627,0.505627,0.535170,0.505627,0.069302,0.039824,0.029543,0.029478,False,0.0,0.005446,0.009694,0.005446,0.009694
14,14,"{'boosting_type': 'dart', 'num_leaves': 7, 'ma...",0.560986,0.539097,0.539097,0.560986,0.539097,0.094459,0.075284,0.021889,0.019174,False,0.0,0.014355,0.005531,0.014355,0.005531
27,27,"{'boosting_type': 'dart', 'num_leaves': 7, 'ma...",0.463255,0.477035,0.477035,0.463255,0.477035,0.010328,0.015606,0.013780,0.005278,False,0.0,0.009071,0.018204,0.009071,0.018204


In [22]:
import ast
candidate_params = params_grid_lgb_v3['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]
candidate_params

[{'boosting_type': 'dart',
  'num_leaves': 13,
  'max_depth': 4,
  'learning_rate': 0.032356740642956634,
  'n_estimators': 779,
  'subsample_for_bin': 57721,
  'min_split_gain': 0.27864646423661144,
  'min_child_weight': 4.295992790539128,
  'min_child_samples': 44,
  'colsample_bytree': 0.6507132052319281,
  'reg_alpha': 0.039319117678150585,
  'reg_lambda': 16.786595382063055,
  'random_state': 42,
  'n_jobs': -1,
  'verbosity': -1,
  'objective': 'binary',
  'metric': 'binary_logloss',
  'is_unbalance': False,
  'subsample': 0.8665668653650511,
  'subsample_freq': 2,
  'drop_rate': 0.3668260108198112,
  'max_drop': 6,
  'skip_drop': 0.6323058305935795,
  'xgboost_dart_mode': True},
 {'boosting_type': 'dart',
  'num_leaves': 7,
  'max_depth': 3,
  'learning_rate': 0.013846699331327199,
  'n_estimators': 1224,
  'subsample_for_bin': 149746,
  'min_split_gain': 0.390838548424289,
  'min_child_weight': 0.22662669367988195,
  'min_child_samples': 35,
  'colsample_bytree': 0.618943369112

In [24]:
df_v3.columns

Index(['serie_no', 'id_card_md5', 'phonenum_md5(param)', 'app_date', 'code',
       'F1', 'F10', 'F100', 'F101', 'F102',
       ...
       'zf08', 'zf09', 'zf10', 'zf11', 'zf12', 'month', 'partner_name',
       'mid_sample_type', 'id_number', 'flag'],
      dtype='object', length=288)

In [25]:
df_v3['apply_date_new'] = pd.to_datetime(df_v3['app_date'], format='%Y%m%d')
df_v3['apply_date_new'].head()

0   2026-03-10
1   2026-04-04
2   2026-03-04
3   2026-03-08
4   2026-04-07
Name: apply_date_new, dtype: datetime64[ns]

In [26]:
params_perf = run_experiment_batch(
    lgb.LGBMClassifier,
    df_v3[df_v3['mid_sample_type']=='1-train'],
    df_v3[df_v3['mid_sample_type']=='3-oot'],
    feas,
    'flag',
    candidate_params,
    report_dir = 'report_v3/',
    ledger_path = 'params_perf_v3.csv',
    sample_type_col = 'mid_sample_type',
    date_col = 'apply_date_new'
    
)

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_v3/anti_overfitting-t0-20260707170359-400ad7f3.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_v3/anti_overfitting-t1-20260707170404-1fe1e42b.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_v3/anti_overfitting-t2-20260707170411-b730ff82.xlsx


In [ ]:
params = {'boosting_type': 'dart',
  'num_leaves': 5,
  'max_depth': 4,
  'learning_rate': 0.012356740642956634,
  'n_estimators': 100,
  'min_split_gain': 0.27864646423661144,
  'min_child_weight': 0.295992790539128,
  'colsample_bytree': 0.6507132052319281,
  'reg_alpha': 5.039319117678150585,
  'reg_lambda': 6.786595382063055,
  'random_state': 42,
  'n_jobs': -1,
  'verbosity': -1,
  'objective': 'binary',
  'metric': 'binary_logloss',
  'is_unbalance': True,
  'subsample': 0.8665668653650511,
  'subsample_freq': 2,
  'drop_rate': 0.5668260108198112,
  'max_drop': 6,
  'skip_drop': 0.6323058305935795,
  'xgboost_dart_mode': True}
params = {'boosting_type': 'dart',
  'num_leaves': 6,
  'max_depth': 4,
  'learning_rate': 0.032356740642956634,
  'n_estimators': 50,
  'min_split_gain': 0.27864646423661144,
  'min_child_weight': 4.295992790539128,
  'min_child_samples': 44,
  'colsample_bytree': 0.6507132052319281,
  'reg_alpha': 0.039319117678150585,
  'reg_lambda': 16.786595382063055,
  'random_state': 42,
  'n_jobs': -1,
  'verbosity': -1,
  'objective': 'binary',
  'metric': 'binary_logloss',
  'is_unbalance': True,
  'subsample': 0.8665668653650511,
  'subsample_freq': 2,
  'drop_rate': 0.3668260108198112,
  'max_drop': 6,
  'skip_drop': 0.6323058305935795,
  'xgboost_dart_mode': True},
model = lgb.LGBMClassifier(**params)

model.fit(df_v3[df_v3['mid_sample_type']=='1-train'][feas_embeddings_v4], df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='1-train']['flag'])
df_embeddings_v4['prob_embeddings_v4'] = model.predict_proba(df_embeddings_v4[feas_embeddings_v4])[:,1]

mdpr = ModelDeliveryReport(
    df_embeddings_v4,
    'flag',
    'prob_embeddings_v4',
    'apply_date_new',
    'partner_name',
)
mdpr.generate_model_performance()

## V4 + embedding

In [27]:
df_embeddings.columns

Index(['serie_no', 'id_card_md5', 'phonenum_md5(param)', 'app_date', 'code',
       'hxe1', 'hxe10', 'hxe11', 'hxe12', 'hxe13', 'hxe14', 'hxe15', 'hxe16',
       'hxe17', 'hxe18', 'hxe19', 'hxe2', 'hxe20', 'hxe21', 'hxe22', 'hxe23',
       'hxe24', 'hxe25', 'hxe26', 'hxe27', 'hxe28', 'hxe29', 'hxe3', 'hxe30',
       'hxe31', 'hxe32', 'hxe33', 'hxe34', 'hxe35', 'hxe36', 'hxe37', 'hxe38',
       'hxe39', 'hxe4', 'hxe40', 'hxe41', 'hxe42', 'hxe43', 'hxe44', 'hxe45',
       'hxe46', 'hxe47', 'hxe48', 'hxe49', 'hxe5', 'hxe50', 'hxe51', 'hxe52',
       'hxe53', 'hxe54', 'hxe55', 'hxe56', 'hxe57', 'hxe58', 'hxe59', 'hxe6',
       'hxe60', 'hxe61', 'hxe62', 'hxe63', 'hxe64', 'hxe65', 'hxe66', 'hxe67',
       'hxe68', 'hxe69', 'hxe7', 'hxe70', 'hxe71', 'hxe72', 'hxe73', 'hxe74',
       'hxe75', 'hxe76', 'hxe77', 'hxe78', 'hxe79', 'hxe8', 'hxe80', 'hxe9',
       'hxs1'],
      dtype='object')

In [28]:
df_v4.columns

Index(['serie_no', 'id_card_md5', 'phonenum_md5(param)', 'app_date', 'code',
       'hce1', 'hce10', 'hce11', 'hce12', 'hce13',
       ...
       'hcu90', 'hcu91', 'hcu92', 'hcu93', 'hcu94', 'hcu95', 'hcu96', 'hcu97',
       'hcu98', 'hcu99'],
      dtype='object', length=267)

In [29]:
df_embeddings.merge(df_v4.drop(columns='code'),on='serie_no').shape

(128769, 351)

In [31]:
df_embeddings.merge(df_v4.drop(columns='code'),on=['serie_no','id_card_md5', 'phonenum_md5(param)', 'app_date']).columns

Index(['serie_no', 'id_card_md5', 'phonenum_md5(param)', 'app_date', 'code',
       'hxe1', 'hxe10', 'hxe11', 'hxe12', 'hxe13',
       ...
       'hcu90', 'hcu91', 'hcu92', 'hcu93', 'hcu94', 'hcu95', 'hcu96', 'hcu97',
       'hcu98', 'hcu99'],
      dtype='object', length=348)

In [36]:
df_embeddings_v4 = df_embeddings.merge(df_v4.drop(columns='code'),on=['serie_no','id_card_md5', 'phonenum_md5(param)', 'app_date'])
feas_embeddings_v4 = df_embeddings_v4.columns[5:].to_list()
len(feas_embeddings_v4)

343

In [37]:
df_embeddings_v4['month']= df_embeddings_v4['app_date'].astype(str).str[0:6]
df_embeddings_v4['month'].value_counts()

month
202603    61842
202604    55514
202602    11251
202605      162
Name: count, dtype: int64

In [38]:
df_embeddings_v4['partner_name'] =df_embeddings_v4['month'].map(
    {'202602':'华通-2026-02',
    '202603':'华通-2026-03',
    '202604':'华通-2026-04至05',
    '202605':'华通-2026-04至05'}
    )
df_embeddings_v4['mid_sample_type'] = df_embeddings_v4['partner_name'].map(
    {'华通-2026-02':'1-train',
    '华通-2026-03':'1-train',
    '华通-2026-04至05':'3-oot'}
    )

In [39]:
print(df_embeddings_v4.shape)
df_embeddings_v4 = df_embeddings_v4.merge(df_label[df_label['partner_name']=='华通银行-星选'][['id_number','flag']],left_on='id_card_md5',right_on='id_number',how='inner')
df_embeddings_v4.head()

(128769, 351)


,serie_no,id_card_md5,phonenum_md5(param),app_date,code,hxe1,hxe10,hxe11,hxe12,hxe13,...,hcu95,hcu96,hcu97,hcu98,hcu99,month,partner_name,mid_sample_type,id_number,flag
0,18257,baf799226f4ecc5e6e0155c51cec94ce,0eff48907cbf6c40e1631941d344a123,20260331,0,0.000000,0.0,0.144249,0.0,0.0,...,0.2617,0.3304,0.2334,0.0805,-0.4599,202603,华通-2026-03,1-train,baf799226f4ecc5e6e0155c51cec94ce,0.0
1,1863,2ba1bb299ebba228bd46f85f3c7e4835,107cf7ab8f4371239c326bfd91ab76cc,20260404,0,0.025855,0.0,0.144249,0.0,0.0,...,-0.3223,0.7351,0.3830,0.0228,-0.5502,202604,华通-2026-04至05,3-oot,2ba1bb299ebba228bd46f85f3c7e4835,0.0
2,42351,c706ac557deafcdad6b2650c5f1b48a9,1d9606a31ea40cdc406640b6bab13034,20260310,0,0.000000,0.0,0.122617,0.0,0.0,...,0.5116,0.9448,0.7116,-0.0313,-1.6431,202603,华通-2026-03,1-train,c706ac557deafcdad6b2650c5f1b48a9,0.0
3,14423,e5d7023d318bdb39b1556849266e4d2d,1e23d36c81dc673db857661f62ba701a,20260402,0,0.415383,0.0,0.000000,0.0,0.0,...,0.4156,1.2965,0.5496,0.2758,-0.0433,202604,华通-2026-04至05,3-oot,e5d7023d318bdb39b1556849266e4d2d,0.0
4,3061,41849b7da94584575f67ffd3f36ed2b9,210dfeec094cf2c5b7980e82a845ffc9,20260326,0,0.254720,0.0,0.222043,0.0,0.0,...,-0.6088,1.0829,0.3245,-0.2280,-0.8750,202603,华通-2026-03,1-train,41849b7da94584575f67ffd3f36ed2b9,0.0


In [40]:
tuner =  AntiOverfittingTuner(
    model_class=lgb.LGBMClassifier,
    model_type='lgb',
    direction='maximize'
)

params_grid_lgb = tuner.optimize_anti_overfitting(
    df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='1-train'][feas_embeddings_v4],
    df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='1-train']['flag'],
    n_trials=30
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [42]:
pd.DataFrame(params_grid_lgb['performance_history']).to_csv('params_log_embedding_v4.csv',index=False)

In [43]:
params_grid_lgb_embedding_v4 = pd.read_csv('params_log_embedding_v4.csv')
# params_grid_lgb_embedding_v4 = params_grid_lgb_embedding_v4[params_grid_lgb_embedding_v4['is_overfitting']==False]
params_grid_lgb_embedding_v4

,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
0,0,"{'boosting_type': 'dart', 'num_leaves': 43, 'm...",0.783801,0.549055,-0.439155,0.783801,0.549055,0.435203,0.095844,0.234746,0.339359,True,0.988210,0.004618,0.008963,0.004618,0.008963
1,1,"{'boosting_type': 'goss', 'num_leaves': 15, 'm...",0.903998,0.540706,-1.171236,0.903998,0.540706,0.649045,0.076366,0.363292,0.572679,True,1.711942,0.001028,0.007215,0.001028,0.007215
2,2,"{'boosting_type': 'goss', 'num_leaves': 15, 'm...",0.818731,0.549463,-0.590618,0.818731,0.549463,0.471477,0.090705,0.269268,0.380772,True,1.140081,0.003142,0.006589,0.003142,0.006589
3,3,"{'boosting_type': 'dart', 'num_leaves': 93, 'm...",0.960158,0.539803,-1.554309,0.960158,0.539803,0.785864,0.079163,0.420355,0.706701,True,2.094112,0.002080,0.007234,0.002080,0.007234
4,4,"{'boosting_type': 'gbdt', 'num_leaves': 15, 'm...",0.995190,0.529869,-2.008305,0.995190,0.529869,0.942105,0.058340,0.465321,0.883765,True,2.538173,0.000335,0.005828,0.000335,0.005828
5,5,"{'boosting_type': 'dart', 'num_leaves': 63, 'm...",0.314208,0.454439,0.273978,0.314208,0.454439,0.000183,0.004660,0.140230,0.004477,True,0.180460,0.008161,0.011382,0.008161,0.011382
6,6,"{'boosting_type': 'goss', 'num_leaves': 127, '...",0.916738,0.536039,-1.242795,0.916738,0.536039,0.663093,0.074375,0.380699,0.588718,True,1.778834,0.004396,0.006637,0.004396,0.006637
7,7,"{'boosting_type': 'dart', 'num_leaves': 13, 'm...",0.536616,0.515337,0.515337,0.536616,0.515337,0.071201,0.052020,0.021279,0.019181,False,0.000000,0.006439,0.019709,0.006439,0.019709
8,8,"{'boosting_type': 'dart', 'num_leaves': 14, 'm...",0.869119,0.546454,-0.903351,0.869119,0.546454,0.566302,0.084064,0.322665,0.482238,True,1.449805,0.003137,0.004618,0.003137,0.004618
9,9,"{'boosting_type': 'gbdt', 'num_leaves': 7, 'ma...",0.989024,0.522168,-1.977620,0.989024,0.522168,0.915696,0.052659,0.466857,0.863037,True,2.499787,0.000376,0.008237,0.000376,0.008237


In [ ]:
import ast
candidate_params = params_grid_lgb_embedding_v4['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]
candidate_params

[{'boosting_type': 'dart',
  'num_leaves': 13,
  'max_depth': 4,
  'learning_rate': 0.032356740642956634,
  'n_estimators': 779,
  'subsample_for_bin': 57721,
  'min_split_gain': 0.27864646423661144,
  'min_child_weight': 4.295992790539128,
  'min_child_samples': 44,
  'colsample_bytree': 0.6507132052319281,
  'reg_alpha': 0.039319117678150585,
  'reg_lambda': 16.786595382063055,
  'random_state': 42,
  'n_jobs': -1,
  'verbosity': -1,
  'objective': 'binary',
  'metric': 'binary_logloss',
  'is_unbalance': False,
  'subsample': 0.8665668653650511,
  'subsample_freq': 2,
  'drop_rate': 0.3668260108198112,
  'max_drop': 6,
  'skip_drop': 0.6323058305935795,
  'xgboost_dart_mode': True},
 {'boosting_type': 'dart',
  'num_leaves': 7,
  'max_depth': 3,
  'learning_rate': 0.013846699331327199,
  'n_estimators': 1224,
  'subsample_for_bin': 149746,
  'min_split_gain': 0.390838548424289,
  'min_child_weight': 0.22662669367988195,
  'min_child_samples': 35,
  'colsample_bytree': 0.618943369112

In [ ]:
params_grid_lgb_embedding_v4.columns

Index(['serie_no', 'id_card_md5', 'phonenum_md5(param)', 'app_date', 'code',
       'F1', 'F10', 'F100', 'F101', 'F102',
       ...
       'zf08', 'zf09', 'zf10', 'zf11', 'zf12', 'month', 'partner_name',
       'mid_sample_type', 'id_number', 'flag'],
      dtype='object', length=288)

In [ ]:
params_grid_lgb_embedding_v4['apply_date_new'] = pd.to_datetime(params_grid_lgb_embedding_v4['app_date'], format='%Y%m%d')
params_grid_lgb_embedding_v4['apply_date_new'].head()

0   2026-03-10
1   2026-04-04
2   2026-03-04
3   2026-03-08
4   2026-04-07
Name: apply_date_new, dtype: datetime64[ns]

In [ ]:
params_perf = run_experiment_batch(
    lgb.LGBMClassifier,
    params_grid_lgb_embedding_v4[params_grid_lgb_embedding_v4['mid_sample_type']=='1-train'],
    params_grid_lgb_embedding_v4[params_grid_lgb_embedding_v4['mid_sample_type']=='3-oot'],
    feas,
    'flag',
    candidate_params,
    report_dir = 'report_embedding_v4/',
    ledger_path = 'params_log_embedding_v4.csv',
    sample_type_col = 'mid_sample_type',
    date_col = 'apply_date_new'
    
)

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_v3/anti_overfitting-t0-20260707170359-400ad7f3.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_v3/anti_overfitting-t1-20260707170404-1fe1e42b.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_v3/anti_overfitting-t2-20260707170411-b730ff82.xlsx


In [223]:
params = {'boosting_type': 'dart', 'num_leaves': 7, 'max_depth': 3, 'learning_rate': 0.002260914149876286, 
'n_estimators': 100, 'min_split_gain': 0.10374114228057249, 'min_child_weight': 0.41651374331523994, 
'min_child_samples': 61, 'colsample_bytree': 0.7426335695527173, 'reg_alpha': 0.0006885458259604218, 'reg_lambda': 0.24219256423539123, 
'random_state': 42, 'n_jobs': -1, 'verbosity': -1, 'objective': 'binary', 'metric': 'binary_logloss', 
'is_unbalance': True, 'subsample': 0.8252604112678665, 'subsample_freq': 1, 'drop_rate': 0.1426912961739955, 'max_drop': 2, 
'skip_drop': 0.727328849205757, 'xgboost_dart_mode': True}
model = lgb.LGBMClassifier(**params)

model.fit(df_embeddings_v4[selected_features_v4], df_embeddings_v4['flag'])
df_embeddings_v4['prob_embeddings_v4'] = model.predict_proba(df_embeddings_v4[selected_features_v4])[:,1]


In [225]:
mdpr = ModelDeliveryReport(
    df_embeddings_v4[df_embeddings_v4['hxs1'].notnull()],
    'flag',
    'prob_embeddings_v4',
    'apply_date_new',
    'partner_name',
)
mdpr.generate_model_performance()

,样本类型,Good,Bad,Total,Bad_Rate,KS,AUC,TOP10_lift,PSI
0,all,120601,3539,124140,0.0285,0.1155,0.5804,1.5682,0.0000
1,华通-2026-03,57759,1907,59666,0.0320,0.1102,0.5738,1.5114,0.0000
2,华通-2026-04至05,52438,1151,53589,0.0215,0.1200,0.5849,1.5818,0.0000
3,华通-2026-02,10404,481,10885,0.0442,0.1273,0.5750,1.5628,0.0000


In [226]:
joblib.dump(model, 'models/overfit_v4.pkl')

['models/overfit_v4.pkl']

In [227]:

skf = StratifiedKFold(n_splits=4,shuffle=True,random_state=20)
train_auc = []
test_auc = []
for train_idx, test_idx in skf.split(df_embeddings_v4[selected_features_v4], df_embeddings_v4['flag']):
    X_train, X_test = df_embeddings_v4[selected_features_v4].iloc[train_idx], df_embeddings_v4[selected_features_v4].iloc[test_idx]
    y_train, y_test = df_embeddings_v4['flag'].iloc[train_idx], df_embeddings_v4['flag'].iloc[test_idx]
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    train_pred = model.predict_proba(X_train)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]
    train_auc.append(roc_auc_score(y_train, train_pred))
    test_auc.append(roc_auc_score(y_test, test_pred))
    if abs(train_auc[-1] - test_auc[-1]) < 0.02:
        joblib.dump(model, 'models/model_embedding_v4_not_overfit.pkl')
    print(f'Train AUC: {train_auc[-1]:.4f}, Test AUC: {test_auc[-1]:.4f}')
    
np.mean(train_auc), np.mean(test_auc)

Train AUC: 0.5881, Test AUC: 0.5490
Train AUC: 0.5882, Test AUC: 0.5456
Train AUC: 0.5829, Test AUC: 0.5623
Train AUC: 0.5855, Test AUC: 0.5518


(0.5861699752888456, 0.5521868814633355)

In [229]:
df_embeddings_v4.columns

Index(['serie_no', 'id_card_md5', 'phonenum_md5(param)', 'app_date', 'code',
       'hxe1', 'hxe10', 'hxe11', 'hxe12', 'hxe13',
       ...
       'hcu96', 'hcu97', 'hcu98', 'hcu99', 'month', 'partner_name',
       'mid_sample_type', 'id_number', 'flag', 'prob_embeddings_v4'],
      dtype='object', length=354)

In [231]:
df_embeddings_v4[['serie_no', 'prob_embeddings_v4']].to_csv('preds_v4_overfit.csv', index=False)

In [44]:
ModelDeliveryReport??

Init signature:
ModelDeliveryReport(
    data: pandas.core.frame.DataFrame,
    target_col: str = 'target',
    score_col: Union[str, List[str]] = 'score',
    date_col: str = 'date',
    sample_type_col: str = 'sample_type',
    score_na_strategy: str = 'intersection',
)
Source:        
class ModelDeliveryReport:
    """模型交付报告生成器"""

    def __init__(self, data: pd.DataFrame, target_col: str = 'target',
                 score_col: Union[str, List[str]] = 'score', date_col: str = 'date',
                 sample_type_col: str = 'sample_type',
                 score_na_strategy: str = 'intersection'):
        """
        初始化报告生成器

        Args:
            data: 包含预测结果的数据框
            target_col: 目标变量列名
            score_col: 模型分数列名，支持两种形式：
                       - str: 单个分数列（原有行为）
                       - List[str]: 多个分数列，对比多个分数的表现
            date_col: 日期列名
            sample_type_col: 样本类型列名 (all, train, test, oot)
            score_na_strategy: 多分数模式下的缺失处理策略：
                       -

In [54]:
mdpr = ModelDeliveryReport(
    df_embeddings_v4[df_embeddings_v4['hxs1'].notnull()],
    'flag',
    'hxs1',
    'apply_date_new',
    'mid_sample_type',
)
mdpr.generate_model_performance()

,样本类型,Good,Bad,Total,Bad_Rate,KS,AUC,TOP10_lift,PSI
0,all,120601,3539,124140,0.0285,0.0643,0.5410,1.2885,0.0000
1,1-train,68163,2388,70551,0.0338,0.0597,0.5352,1.2858,0.0000
2,3-oot,52438,1151,53589,0.0215,0.0716,0.5446,1.1989,0.0000


In [53]:
df_embeddings_v4['hxs1'].isnull().sum()

4629

In [68]:
params = {'boosting_type': 'dart',
  'num_leaves': 5,
  'max_depth': 3,
  'learning_rate': 0.012356740642956634,
  'n_estimators': 100,
  'min_split_gain': 0.27864646423661144,
  'min_child_weight': 0.295992790539128,
  'colsample_bytree': 0.6507132052319281,
  'reg_alpha': 0.039319117678150585,
  'reg_lambda': 1.786595382063055,
  'random_state': 42,
  'n_jobs': -1,
  'verbosity': -1,
  'objective': 'binary',
  'metric': 'binary_logloss',
  'is_unbalance': True,
  'subsample': 0.4665668653650511,
  'subsample_freq': 2,
  'drop_rate': 0.5668260108198112,
  'max_drop': 6,
  'skip_drop': 0.6323058305935795,
  'xgboost_dart_mode': True}
model = lgb.LGBMClassifier(**params)

model.fit(df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='1-train'][feas_embeddings_v4], df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='1-train']['flag'])
df_embeddings_v4['prob_embeddings_v4'] = model.predict_proba(df_embeddings_v4[feas_embeddings_v4])[:,1]

mdpr = ModelDeliveryReport(
    df_embeddings_v4,
    'flag',
    'prob_embeddings_v4',
    'apply_date_new',
    'partner_name',
)
mdpr.generate_model_performance()

,样本类型,Good,Bad,Total,Bad_Rate,KS,AUC,TOP10_lift,PSI
0,all,125109,3660,128769,0.0284,0.1484,0.6052,1.7295,0.0000
1,华通-2026-03,59876,1966,61842,0.0318,0.1751,0.6254,1.7495,0.0000
2,华通-2026-04至05,54478,1198,55676,0.0215,0.0928,0.5548,1.4607,0.0000
3,华通-2026-02,10755,496,11251,0.0441,0.1817,0.6250,1.9357,0.0000


In [69]:
params = {'boosting_type': 'dart',
  'num_leaves': 5,
  'max_depth': 4,
  'learning_rate': 0.012356740642956634,
  'n_estimators': 100,
  'min_split_gain': 0.27864646423661144,
  'min_child_weight': 0.295992790539128,
  'colsample_bytree': 0.6507132052319281,
  'reg_alpha': 5.039319117678150585,
  'reg_lambda': 6.786595382063055,
  'random_state': 42,
  'n_jobs': -1,
  'verbosity': -1,
  'objective': 'binary',
  'metric': 'binary_logloss',
  'is_unbalance': True,
  'subsample': 0.4665668653650511,
  'subsample_freq': 2,
  'drop_rate': 0.5668260108198112,
  'max_drop': 6,
  'skip_drop': 0.6323058305935795,
  'xgboost_dart_mode': True}
model = lgb.LGBMClassifier(**params)

model.fit(df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='1-train'][feas_embeddings_v4], df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='1-train']['flag'])
df_embeddings_v4['prob_embeddings_v4'] = model.predict_proba(df_embeddings_v4[feas_embeddings_v4])[:,1]

mdpr = ModelDeliveryReport(
    df_embeddings_v4,
    'flag',
    'prob_embeddings_v4',
    'apply_date_new',
    'partner_name',
)
mdpr.generate_model_performance()

,样本类型,Good,Bad,Total,Bad_Rate,KS,AUC,TOP10_lift,PSI
0,all,125109,3660,128769,0.0284,0.1424,0.6016,1.6940,0.0000
1,华通-2026-03,59876,1966,61842,0.0318,0.1687,0.6197,1.7343,0.0000
2,华通-2026-04至05,54478,1198,55676,0.0215,0.0865,0.5548,1.5191,0.0000
3,华通-2026-02,10755,496,11251,0.0441,0.1841,0.6204,1.6937,0.0000


# 特征筛选 后建模

In [ ]:
"""
单特征Lift分箱分析与可视化
用于建模前的特征筛选
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


class FeatureLiftAnalyzer:
    """单特征Lift分析器"""
    
    def __init__(self, df: pd.DataFrame, features: List[str], target: str, 
                 max_bins: int = 10, min_samples: int = 30):
        """
        初始化
        
        Parameters:
        -----------
        df : DataFrame
            数据集
        features : List[str]
            特征列表
        target : str
            目标变量名（二分类）
        max_bins : int
            最大分箱数，默认10
        min_samples : int
            每个分箱最小样本数，默认30
        """
        self.df = df.copy()
        self.features = features
        self.target = target
        self.max_bins = max_bins
        self.min_samples = min_samples
        self.lift_results = {}
        self.summary_table = None
        
    def _auto_bin(self, series: pd.Series) -> pd.Series:
        """
        自动分箱
        - 连续型特征：等频分箱
        - 离散型特征：保持原值（如果类别数<=max_bins），否则按频数合并
        """
        # 去除缺失值
        df_lr = series.dropna()
        
        if len(df_lr) == 0:
            return series
        
        # 判断是否为数值型
        if pd.api.types.is_numeric_dtype(series):
            unique_count = df_lr.nunique()
            
            if unique_count <= 2:
                # 二值变量，不分箱
                return series
            elif unique_count <= self.max_bins:
                # 离散数值，尝试等频分箱但保留原值
                try:
                    # 尝试按等频分箱
                    binned = pd.qcut(series, q=min(unique_count, self.max_bins), 
                                      duplicates='drop', precision=2)
                    return binned
                except:
                    return series
            else:
                # 连续数值，等频分箱
                try:
                    # 动态调整分箱数，确保每箱有足够样本
                    n_samples = len(df_lr)
                    adjusted_bins = min(self.max_bins, int(n_samples / self.min_samples))
                    adjusted_bins = max(2, adjusted_bins)
                    
                    binned = pd.qcut(series, q=adjusted_bins, duplicates='drop', precision=2)
                    return binned
                except:
                    return series
        else:
            # 非数值型，按类别分组
            unique_count = df_lr.nunique()
            if unique_count <= self.max_bins:
                return series
            else:
                # 类别过多，按频数合并低频类别
                value_counts = series.value_counts()
                top_categories = value_counts.head(self.max_bins - 1).index.tolist()
                return series.where(series.isin(top_categories), 'Other')
    
    def _calculate_lift_single_feature(self, feature: str) -> Dict:
        """
        计算单个特征的Lift和IV值
        
        Returns:
        --------
        Dict: 包含分箱详情、Lift值、IV值等
        """
        # 分箱
        binned_feature = self._auto_bin(self.df[feature])
        
        # 计算每个分箱的统计量
        grouped = self.df.groupby(binned_feature, dropna=False).agg({
            self.target: ['sum', 'count']
        })
        grouped.columns = ['bad', 'total']
        grouped = grouped.reset_index()
        grouped.columns = ['bin', 'bad', 'total']
        
        # 计算整体bad rate
        total_bad = self.df[self.target].sum()
        total_count = len(self.df)
        overall_bad_rate = total_bad / total_count
        
        # 计算每个分箱的指标
        grouped['good'] = grouped['total'] - grouped['bad']
        grouped['bad_rate'] = grouped['bad'] / grouped['total']
        grouped['lift'] = grouped['bad_rate'] / overall_bad_rate
        grouped['lift'] = grouped['lift'].replace([np.inf, -np.inf], np.nan)
        
        # 计算WOE和IV
        grouped['pct_bad'] = grouped['bad'] / total_bad
        grouped['pct_good'] = grouped['good'] / (total_count - total_bad)
        grouped['woe'] = np.log(grouped['pct_bad'] / grouped['pct_good'])
        grouped['iv'] = (grouped['pct_bad'] - grouped['pct_good']) * grouped['woe']
        
        # 计算总IV值
        iv_value = grouped['iv'].sum()
        
        # 计算最大Lift和Lift范围
        max_lift = grouped['lift'].max()
        min_lift = grouped['lift'].min()
        lift_range = max_lift - min_lift
        
        return {
            'feature': feature,
            'binned_data': grouped,
            'iv_value': iv_value,
            'max_lift': max_lift,
            'min_lift': min_lift,
            'lift_range': lift_range,
            'n_bins': len(grouped),
            'missing_rate': self.df[feature].isna().sum() / len(self.df)
        }
    
    def analyze(self) -> pd.DataFrame:
        """
        对所有特征进行Lift分析
        
        Returns:
        --------
        DataFrame: 特征汇总表
        """
        print(f"开始分析 {len(self.features)} 个特征...")
        
        summary_data = []
        
        for i, feature in enumerate(self.features, 1):
            try:
                result = self._calculate_lift_single_feature(feature)
                self.lift_results[feature] = result
                
                summary_data.append({
                    '特征名': feature,
                    'IV值': round(result['iv_value'], 4),
                    '最大Lift': round(result['max_lift'], 4),
                    '最小Lift': round(result['min_lift'], 4),
                    'Lift范围': round(result['lift_range'], 4),
                    '分箱数': result['n_bins'],
                    '缺失率': round(result['missing_rate'], 4)
                })
                
                if i % 10 == 0:
                    print(f"已处理 {i}/{len(self.features)} 个特征")
                    
            except Exception as e:
                print(f"特征 {feature} 处理失败: {str(e)}")
                continue
        
        # 生成汇总表
        self.summary_table = pd.DataFrame(summary_data)
        
        # 按IV值降序排序
        self.summary_table = self.summary_table.sort_values('IV值', ascending=False)
        self.summary_table = self.summary_table.reset_index(drop=True)
        
        print(f"\n分析完成！共成功分析 {len(self.summary_table)} 个特征")
        
        return self.summary_table
    
    def plot_feature_lift(self, feature: str, figsize: Tuple[int, int] = (10, 6)):
        """
        绘制单个特征的Lift图
        
        Parameters:
        -----------
        feature : str
            特征名
        figsize : Tuple[int, int]
            图形大小
        """
        if feature not in self.lift_results:
            print(f"特征 {feature} 未在分析结果中")
            return
        
        result = self.lift_results[feature]
        binned_data = result['binned_data']
        
        # 创建图形
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
        
        # 左图：Lift柱状图
        bars = ax1.bar(range(len(binned_data)), binned_data['lift'], 
                       color='steelblue', alpha=0.7, edgecolor='black')
        ax1.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='Baseline (Lift=1)')
        ax1.set_xlabel('Bin', fontsize=12)
        ax1.set_ylabel('Lift', fontsize=12)
        ax1.set_title(f'{feature}\nIV={result["iv_value"]:.4f}', fontsize=13, fontweight='bold')
        ax1.legend(loc='best')
        ax1.grid(axis='y', alpha=0.3)
        
        # 设置x轴标签
        bin_labels = [str(x) if len(str(x)) < 15 else str(x)[:12]+'...' 
                      for x in binned_data['bin']]
        ax1.set_xticks(range(len(binned_data)))
        ax1.set_xticklabels(bin_labels, rotation=45, ha='right', fontsize=9)
        
        # 在柱子上标注Lift值
        for i, (bar, lift) in enumerate(zip(bars, binned_data['lift'])):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{lift:.2f}',
                    ha='center', va='bottom', fontsize=9)
        
        # 右图：Bad Rate和样本量
        ax2_twin = ax2.twinx()
        
        # 柱状图：样本量
        bars2 = ax2.bar(range(len(binned_data)), binned_data['total'], 
                        color='lightgray', alpha=0.5, edgecolor='black', label='样本量')
        
        # 折线图：Bad Rate
        line = ax2_twin.plot(range(len(binned_data)), binned_data['bad_rate'], 
                            color='darkorange', marker='o', linewidth=2, 
                            markersize=8, label='Bad Rate')
        
        ax2.set_xlabel('Bin', fontsize=12)
        ax2.set_ylabel('样本量', fontsize=12, color='gray')
        ax2_twin.set_ylabel('Bad Rate', fontsize=12, color='darkorange')
        ax2.set_title('样本分布与Bad Rate', fontsize=13, fontweight='bold')
        
        ax2.set_xticks(range(len(binned_data)))
        ax2.set_xticklabels(bin_labels, rotation=45, ha='right', fontsize=9)
        
        ax2.tick_params(axis='y', labelcolor='gray')
        ax2_twin.tick_params(axis='y', labelcolor='darkorange')
        ax2.grid(axis='y', alpha=0.3)
        
        # 添加图例
        lines1, labels1 = ax2.get_legend_handles_labels()
        lines2, labels2 = ax2_twin.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
        
        plt.tight_layout()
        plt.show()
    
    def plot_top_features(self, top_n: int = 10, figsize: Tuple[int, int] = (16, 10)):
        """
        绘制Top N特征的Lift图
        
        Parameters:
        -----------
        top_n : int
            显示前N个特征
        figsize : Tuple[int, int]
            图形大小
        """
        if self.summary_table is None:
            print("请先运行 analyze() 方法")
            return
        
        top_features = self.summary_table.head(top_n)['特征名'].tolist()
        
        n_cols = 2
        n_rows = (top_n + 1) // 2
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
        axes = axes.flatten()
        
        for idx, feature in enumerate(top_features):
            if feature not in self.lift_results:
                continue
            
            result = self.lift_results[feature]
            binned_data = result['binned_data']
            ax = axes[idx]
            
            # 绘制Lift柱状图
            bars = ax.bar(range(len(binned_data)), binned_data['lift'], 
                         color='steelblue', alpha=0.7, edgecolor='black')
            ax.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5)
            
            ax.set_title(f'{feature}\nIV={result["iv_value"]:.4f}', 
                        fontsize=11, fontweight='bold')
            ax.set_ylabel('Lift', fontsize=10)
            ax.grid(axis='y', alpha=0.3)
            
            # 简化x轴标签
            bin_labels = [str(x) if len(str(x)) < 10 else str(x)[:8]+'...' 
                          for x in binned_data['bin']]
            ax.set_xticks(range(len(binned_data)))
            ax.set_xticklabels(bin_labels, rotation=45, ha='right', fontsize=8)
        
        # 隐藏多余的子图
        for idx in range(len(top_features), len(axes)):
            axes[idx].axis('off')
        
        plt.suptitle(f'Top {top_n} 特征Lift分析', fontsize=16, fontweight='bold', y=1.00)
        plt.tight_layout()
        # 
        plt.show()
    
    def get_feature_recommendation(self, iv_threshold: float = 0.02, 
                                   lift_range_threshold: float = 0.5) -> pd.DataFrame:
        """
        根据IV值和Lift范围给出特征筛选建议
        
        Parameters:
        -----------
        iv_threshold : float
            IV值阈值，默认0.02
        lift_range_threshold : float
            Lift范围阈值，默认0.5
        
        Returns:
        --------
        DataFrame: 特征筛选建议表
        """
        if self.summary_table is None:
            print("请先运行 analyze() 方法")
            return None
        
        recommendation = self.summary_table.copy()
        
        # IV值评级
        recommendation['IV评级'] = recommendation['IV值'].apply(
            lambda x: '强' if x >= 0.5 else ('中' if x >= 0.1 else ('弱' if x >= 0.02 else '无用'))
        )
        
        # Lift范围评级
        recommendation['Lift区分度'] = recommendation['Lift范围'].apply(
            lambda x: '高' if x >= 1.0 else ('中' if x >= 0.5 else '低')
        )
        
        # 综合建议
        def get_suggestion(row):
            if row['IV值'] >= 0.1 and row['Lift范围'] >= 0.5:
                return '保留'
            elif row['IV值'] >= 0.02 and row['Lift范围'] >= 0.3:
                return '考虑保留'
            else:
                return '建议剔除'
        
        recommendation['建议'] = recommendation.apply(get_suggestion, axis=1)
        
        return recommendation[['特征名', 'IV值', 'IV评级', 'Lift范围', 'Lift区分度', '建议']]
    
    def export_to_excel(self, output_path: str = 'feature_lift_analysis.xlsx'):
        """
        将所有特征的Lift分析结果导出到Excel
        
        Parameters:
        -----------
        output_path : str
            Excel文件路径
        
        Excel结构：
        - Sheet1: 特征汇总表（所有特征的IV值、Lift范围等）
        - Sheet2: 特征筛选建议
        - Sheet3: 所有特征Lift对比（所有特征的分箱信息合并到一个Sheet，特征间用不同颜色区分）
        """
        if self.summary_table is None:
            print("请先运行 analyze() 方法")
            return
        
        print(f"开始导出到Excel: {output_path}")
        
        # 创建Excel writer
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            # Sheet1: 特征汇总表
            summary_sheet = self.summary_table.copy()
            summary_sheet.to_excel(writer, sheet_name='特征汇总', index=False)
            print("  ✓ 特征汇总表已写入")
            
            # Sheet2: 特征筛选建议
            recommendation = self.get_feature_recommendation()
            recommendation.to_excel(writer, sheet_name='筛选建议', index=False)
            print("  ✓ 筛选建议表已写入")
            
            # Sheet3: 所有特征Lift对比（合并所有特征到一个Sheet）
            print(f"  正在合并 {len(self.lift_results)} 个特征的详细分箱信息...")
            
            all_features_detail = []
            feature_row_info = []  # 记录每个特征的起始行和结束行
            
            # 按IV值降序处理特征
            current_row = 0  # 当前行计数器
            for i, (feature, result) in enumerate(self.lift_results.items(), 1):
                start_row = current_row
                
                # 准备详细数据
                detail_df = result['binned_data'].copy()
                n_rows = len(detail_df)  # 该特征的行数
                
                # 添加汇总信息
                detail_df['特征名'] = feature
                detail_df['IV值'] = round(result['iv_value'], 4)
                detail_df['最大Lift'] = round(result['max_lift'], 4)
                detail_df['最小Lift'] = round(result['min_lift'], 4)
                detail_df['Lift范围'] = round(result['lift_range'], 4)
                detail_df['缺失率'] = round(result['missing_rate'], 4)
                
                # 调整列顺序
                cols = ['特征名', 'bin', 'bad', 'good', 'total', 'bad_rate', 'lift', 
                        'woe', 'iv', 'IV值', '最大Lift', '最小Lift', 'Lift范围', '缺失率']
                detail_df = detail_df[cols]
                
                # 重命名列
                detail_df.columns = ['特征名', '分箱', 'Bad数', 'Good数', '样本量', 
                                    'Bad Rate', 'Lift', 'WOE', 'IV贡献', 
                                    '总IV值', '最大Lift', '最小Lift', 'Lift范围', '缺失率']
                
                all_features_detail.append(detail_df)
                
                # 更新结束行
                end_row = current_row + n_rows - 1
                feature_row_info.append((feature, start_row, end_row))
                
                # 更新当前行计数器
                current_row += n_rows
                
                if i % 20 == 0:
                    print(f"    已处理 {i}/{len(self.lift_results)} 个特征")
            
            # 合并所有特征
            all_features_df = pd.concat(all_features_detail, axis=0, ignore_index=True)
            
            # 写入到Sheet3
            all_features_df.to_excel(writer, sheet_name='所有特征Lift对比', index=False)
            print(f"  ✓ 所有特征Lift对比表已写入（共 {len(all_features_df)} 行）")
            
            # 添加间隔颜色填充
            print(f"  正在添加间隔颜色...")
            from openpyxl.styles import PatternFill, Font
            
            worksheet = writer.sheets['所有特征Lift对比']
            
            # 定义两种填充颜色（浅色，方便阅读）
            fill_color_1 = PatternFill(start_color='E6F3FF', end_color='E6F3FF', fill_type='solid')  # 浅蓝色
            fill_color_2 = PatternFill(start_color='FFF9E6', end_color='FFF9E6', fill_type='solid')  # 浅黄色
            
            # 为每个特征的行添加颜色
            # feature_row_info记录的是DataFrame中的行索引（从0开始）
            # Excel行号 = DataFrame行索引 + 2（因为Excel从1开始，且第1行是表头）
            for idx, (feature, start_row, end_row) in enumerate(feature_row_info):
                # 交替使用两种颜色
                fill_color = fill_color_1 if idx % 2 == 0 else fill_color_2
                
                # 填充该特征的所有行
                # start_row和end_row是DataFrame的索引，从0开始
                # Excel行号 = DataFrame索引 + 2
                for row_idx in range(start_row, end_row + 1):
                    excel_row_num = row_idx + 2  # 转换为Excel行号
                    for col_num in range(1, len(all_features_df.columns) + 1):
                        worksheet.cell(row=excel_row_num, column=col_num).fill = fill_color
            
            # 设置表头样式
            header_font = Font(bold=True)
            header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')  # 深蓝色表头
            header_font_white = Font(bold=True, color='FFFFFF')
            for col_num in range(1, len(all_features_df.columns) + 1):
                worksheet.cell(row=1, column=col_num).font = header_font_white
                worksheet.cell(row=1, column=col_num).fill = header_fill
            
            print(f"  ✓ 间隔颜色已添加")
        
        print(f"\n✅ Excel导出成功: {output_path}")
        print(f"\n文件包含:")
        print(f"  - Sheet 1: 特征汇总表（所有特征的IV值、Lift等指标）")
        print(f"  - Sheet 2: 特征筛选建议（保留/剔除建议）")
        print(f"  - Sheet 3: 所有特征Lift对比（{len(self.lift_results)}个特征的分箱详情，特征间用不同颜色区分）")


def run_feature_analysis(df: pd.DataFrame, features: List[str], target: str,
                        dataset_name: str = "数据集", max_bins: int = 10,
                        top_n_plot: int = 10, export_excel: bool = True,
                        excel_path: str = None) -> Tuple[FeatureLiftAnalyzer, pd.DataFrame]:
    """
    执行完整的特征Lift分析流程
    
    Parameters:
    -----------
    df : DataFrame
        数据集
    features : List[str]
        特征列表
    target : str
        目标变量名
    dataset_name : str
        数据集名称（用于显示）
    max_bins : int
        最大分箱数
    top_n_plot : int
        绘制前N个特征的图表（设置为0则不绘图）
    export_excel : bool
        是否导出Excel，默认True
    excel_path : str
        Excel文件路径，默认None时自动生成
    
    Returns:
    --------
    Tuple: (analyzer对象, 汇总表)
    """
    print("="*80)
    print(f"{dataset_name} - 特征Lift分析")
    print("="*80)
    
    # 初始化分析器
    analyzer = FeatureLiftAnalyzer(df, features, target, max_bins=max_bins)
    
    # 执行分析
    summary_table = analyzer.analyze()
    
    # 显示汇总表
    print("\n" + "="*80)
    print("特征汇总表（按IV值降序）")
    print("="*80)
    print(summary_table.to_string(index=False))
    
    # 显示Top特征（如果top_n_plot > 0）
    if top_n_plot > 0:
        print("\n" + "="*80)
        print(f"Top {top_n_plot} 特征详情")
        print("="*80)
        analyzer.plot_top_features(top_n=top_n_plot)
    
    # 生成筛选建议
    print("\n" + "="*80)
    print("特征筛选建议")
    print("="*80)
    recommendation = analyzer.get_feature_recommendation()
    print(recommendation.to_string(index=False))
    
    # 统计建议分布
    print("\n建议统计:")
    print(recommendation['建议'].value_counts())
    
    # 导出Excel
    if export_excel:
        if excel_path is None:
            # 自动生成文件名
            safe_name = dataset_name.replace(' ', '_').replace('+', '_')
            excel_path = f'feature_lift_{safe_name}.xlsx'
        
        analyzer.export_to_excel(excel_path)
    
    return analyzer, summary_table

In [86]:
# 导入模块
# from feature_lift_analysis import run_feature_analysis


analyzer_v3, summary_v3 = run_feature_analysis(
    df=df_v3[df_v3['mid_sample_type']=='1-train'],
    features=feas,  # 你的V3特征列表
    target='flag',
    dataset_name='V3数据集',
    top_n_plot=0,  # 设置为0不绘图
    export_excel=True,
    excel_path='feature_lift_v3.xlsx'  # 指定Excel文件名
)

# V4+Embedding数据集分析
analyzer_v4, summary_v4 = run_feature_analysis(
    df=df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='1-train'],
    features=feas_embeddings_v4,  # 你的V4+Embedding特征列表
    target='flag',
    dataset_name='V4+Embedding数据集',
    top_n_plot=0,  # 设置为0不绘图
    export_excel=True,
    excel_path='feature_lift_v4_embeddings.xlsx'  # 指定Excel文件名
)



V3数据集 - 特征Lift分析
开始分析 278 个特征...
已处理 10/278 个特征
已处理 20/278 个特征
已处理 30/278 个特征
已处理 40/278 个特征
已处理 50/278 个特征
已处理 60/278 个特征
已处理 70/278 个特征
已处理 80/278 个特征
已处理 90/278 个特征
已处理 100/278 个特征
已处理 110/278 个特征
已处理 120/278 个特征
已处理 130/278 个特征
已处理 140/278 个特征
已处理 150/278 个特征
已处理 160/278 个特征
已处理 170/278 个特征
已处理 180/278 个特征
已处理 190/278 个特征
已处理 200/278 个特征
已处理 210/278 个特征
已处理 220/278 个特征
已处理 230/278 个特征
已处理 240/278 个特征
已处理 250/278 个特征
已处理 260/278 个特征
已处理 270/278 个特征

分析完成！共成功分析 278 个特征

特征汇总表（按IV值降序）
 特征名    IV值  最大Lift  最小Lift  Lift范围  分箱数    缺失率
gt38    inf  1.0018  0.0000  1.0018    6 0.9974
gt65    inf  1.0003  0.0000  1.0003    6 0.9993
gl12    inf  1.1580  0.0000  1.1580    3 0.9572
gt64    inf  1.8120  0.0000  1.8120    6 0.9957
gt34    inf  1.4081  0.0000  1.4081   10 0.9658
 F91 0.0342  1.3121  0.7463  0.5658   11 0.0312
 F78 0.0318  1.2912  0.7756  0.5155   11 0.0312
 F94 0.0312  1.2451  0.7715  0.4736   10 0.0312
F114 0.0306  1.2912  0.7336  0.5575   11 0.0312
 F39 0.0302  1.2493  0.7505  

In [ ]:

# 3. 获取推荐保留的特征
recommendation_v3 = analyzer_v3.get_feature_recommendation()
selected_features_v3 = recommendation_v3[recommendation_v3['建议'].isin(['保留', '考虑保留'])]['特征名'].tolist()
len(selected_features_v3)

57

In [88]:
selected_features_v3

['gt38',
 'gt65',
 'gl12',
 'gt64',
 'gt34',
 'F91',
 'F78',
 'F94',
 'F114',
 'F39',
 'F16',
 'F29',
 'F4',
 'F118',
 'F30',
 'F38',
 'F47',
 'F19',
 'F9',
 'F8',
 'F35',
 'F104',
 'F23',
 'F126',
 'F1',
 'F62',
 'F28',
 'F84',
 'F36',
 'F105',
 'F70',
 'F127',
 'F103',
 'F71',
 'F69',
 'F32',
 'F73',
 'F40',
 'F74',
 'F90',
 'F85',
 'F59',
 'F124',
 'F37',
 'F116',
 'F88',
 'F21',
 'F98',
 'F113',
 'F93',
 'F83',
 'F5',
 'F111',
 'F77',
 'F52',
 'F86',
 'F10']

In [89]:
'hxs1' in feas_embeddings_v4

True

In [90]:
feas_embeddings_v4

['hxe1',
 'hxe10',
 'hxe11',
 'hxe12',
 'hxe13',
 'hxe14',
 'hxe15',
 'hxe16',
 'hxe17',
 'hxe18',
 'hxe19',
 'hxe2',
 'hxe20',
 'hxe21',
 'hxe22',
 'hxe23',
 'hxe24',
 'hxe25',
 'hxe26',
 'hxe27',
 'hxe28',
 'hxe29',
 'hxe3',
 'hxe30',
 'hxe31',
 'hxe32',
 'hxe33',
 'hxe34',
 'hxe35',
 'hxe36',
 'hxe37',
 'hxe38',
 'hxe39',
 'hxe4',
 'hxe40',
 'hxe41',
 'hxe42',
 'hxe43',
 'hxe44',
 'hxe45',
 'hxe46',
 'hxe47',
 'hxe48',
 'hxe49',
 'hxe5',
 'hxe50',
 'hxe51',
 'hxe52',
 'hxe53',
 'hxe54',
 'hxe55',
 'hxe56',
 'hxe57',
 'hxe58',
 'hxe59',
 'hxe6',
 'hxe60',
 'hxe61',
 'hxe62',
 'hxe63',
 'hxe64',
 'hxe65',
 'hxe66',
 'hxe67',
 'hxe68',
 'hxe69',
 'hxe7',
 'hxe70',
 'hxe71',
 'hxe72',
 'hxe73',
 'hxe74',
 'hxe75',
 'hxe76',
 'hxe77',
 'hxe78',
 'hxe79',
 'hxe8',
 'hxe80',
 'hxe9',
 'hxs1',
 'hce1',
 'hce10',
 'hce11',
 'hce12',
 'hce13',
 'hce14',
 'hce15',
 'hce16',
 'hce17',
 'hce18',
 'hce19',
 'hce2',
 'hce20',
 'hce21',
 'hce3',
 'hce4',
 'hce5',
 'hce6',
 'hce7',
 'hce8',
 'hce9',

In [91]:
# 3. 获取推荐保留的特征
recommendation_v4 = analyzer_v4.get_feature_recommendation()
selected_features_v4 = recommendation_v4[recommendation_v4['建议'].isin(['保留', '考虑保留'])]['特征名'].tolist()
len(selected_features_v4)

14

In [92]:
selected_features_v4

['hcs102',
 'hce16',
 'hce9',
 'hxe3',
 'hxe76',
 'hce15',
 'hce18',
 'hce7',
 'hce13',
 'hxe61',
 'hce4',
 'hce19',
 'hxs1',
 'hce11']

In [203]:
params =  {"boosting_type":"gbdt","colsample_bytree":0.8070438785116428,"is_unbalance":True,"learning_rate":0.0012692642404181964,
"max_depth":3,"metric":"binary_logloss","min_child_samples":69,"min_child_weight":0.013372501428914574,
"min_split_gain":0.012961068370405936,"n_estimators":50,"n_jobs":-1,"num_leaves":7,"objective":"binary",
"random_state":42,"reg_alpha":19.82826417723215,"reg_lambda":0.7277335835353113,"subsample":0.7927640638764053,
"subsample_for_bin":106937,"subsample_freq":7,"verbosity":-1}


model.fit(df_v3[selected_features_v3], df_v3['flag'])
df_v3['prob_v3'] = model.predict_proba(df_v3[selected_features_v3])[:,1]

mdpr = ModelDeliveryReport(
    df_v3,
    'flag',
    'prob_v3',
    'apply_date_new',
    'partner_name',
)
mdpr.generate_model_performance()

,样本类型,Good,Bad,Total,Bad_Rate,KS,AUC,TOP10_lift,PSI
0,all,125109,3660,128769,0.0284,0.1121,0.5802,1.5492,0.0000
1,华通-2026-03,59876,1966,61842,0.0318,0.1152,0.5794,1.5422,0.0000
2,华通-2026-04至05,54478,1198,55676,0.0215,0.1156,0.5820,1.6192,0.0000
3,华通-2026-02,10755,496,11251,0.0441,0.1155,0.5730,1.4393,0.0000


In [111]:
params =  {"boosting_type":"gbdt","colsample_bytree":0.8070438785116428,"is_unbalance":True,"learning_rate":0.0012692642404181964,
"max_depth":3,"metric":"binary_logloss","min_child_samples":69,"min_child_weight":0.013372501428914574,
"min_split_gain":0.012961068370405936,"n_estimators":50,"n_jobs":-1,"num_leaves":7,"objective":"binary",
"random_state":42,"reg_alpha":19.82826417723215,"reg_lambda":0.7277335835353113,"subsample":0.7927640638764053,
"subsample_for_bin":106937,"subsample_freq":7,"verbosity":-1}


model.fit(df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='1-train'][feas_embeddings_v4], df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='1-train']['flag'])
df_embeddings_v4['prob_embeddings_v4'] = model.predict_proba(df_embeddings_v4[feas_embeddings_v4])[:,1]

mdpr = ModelDeliveryReport(
    df_embeddings_v4,
    'flag',
    'prob_embeddings_v4',
    'apply_date_new',
    'partner_name',
)
mdpr.generate_model_performance()

,样本类型,Good,Bad,Total,Bad_Rate,KS,AUC,TOP10_lift,PSI
0,all,125109,3660,128769,0.0284,0.1424,0.6016,1.6940,0.0000
1,华通-2026-03,59876,1966,61842,0.0318,0.1687,0.6197,1.7343,0.0000
2,华通-2026-04至05,54478,1198,55676,0.0215,0.0865,0.5548,1.5191,0.0000
3,华通-2026-02,10755,496,11251,0.0441,0.1841,0.6204,1.6937,0.0000


In [112]:
tuner =  AntiOverfittingTuner(
    model_class=lgb.LGBMClassifier,
    model_type='lgb',
    direction='maximize'
)

params_grid_lgb = tuner.optimize_anti_overfitting(
    df_embeddings_v4[selected_features_v4],
    df_embeddings_v4['flag'],
    n_trials=30
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [113]:
pd.DataFrame(params_grid_lgb['performance_history']).to_csv('params_log_embedding_v4.csv',index=False)

In [114]:
params_grid_lgb_embedding_v4 = pd.read_csv('params_log_embedding_v4.csv')
params_grid_lgb_embedding_v4 = params_grid_lgb_embedding_v4[params_grid_lgb_embedding_v4['is_overfitting']==False]
params_grid_lgb_embedding_v4

,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
5,5,"{'boosting_type': 'dart', 'num_leaves': 63, 'm...",0.411528,0.451595,0.451595,0.411528,0.451595,0.000769,0.002708,0.040067,0.001939,False,0.0,0.004812,0.009433,0.004812,0.009433
7,7,"{'boosting_type': 'dart', 'num_leaves': 13, 'm...",0.525940,0.513436,0.513436,0.525940,0.513436,0.045410,0.039865,0.012504,0.005544,False,0.0,0.011799,0.009703,0.011799,0.009703
13,13,"{'boosting_type': 'dart', 'num_leaves': 31, 'm...",0.507019,0.501680,0.501680,0.507019,0.501680,0.066260,0.038475,0.005339,0.027786,False,0.0,0.086299,0.039054,0.086299,0.039054
14,14,"{'boosting_type': 'dart', 'num_leaves': 7, 'ma...",0.556283,0.545745,0.545745,0.556283,0.545745,0.087327,0.078535,0.010538,0.008793,False,0.0,0.008017,0.009535,0.008017,0.009535
20,20,"{'boosting_type': 'dart', 'num_leaves': 7, 'ma...",0.567333,0.552786,0.552786,0.567333,0.552786,0.097728,0.085766,0.014547,0.011962,False,0.0,0.002671,0.008008,0.002671,0.008008
21,21,"{'boosting_type': 'dart', 'num_leaves': 7, 'ma...",0.562979,0.546182,0.546182,0.562979,0.546182,0.092738,0.081459,0.016797,0.011279,False,0.0,0.003372,0.008251,0.003372,0.008251
22,22,"{'boosting_type': 'dart', 'num_leaves': 7, 'ma...",0.556381,0.543906,0.543906,0.556381,0.543906,0.087352,0.079905,0.012475,0.007446,False,0.0,0.002629,0.012934,0.002629,0.012934
23,23,"{'boosting_type': 'dart', 'num_leaves': 7, 'ma...",0.580403,0.554084,0.554084,0.580403,0.554084,0.116985,0.090999,0.026320,0.025986,False,0.0,0.001511,0.007181,0.001511,0.007181
25,25,"{'boosting_type': 'dart', 'num_leaves': 7, 'ma...",0.582490,0.554209,0.554209,0.582490,0.554209,0.118881,0.089299,0.028281,0.029582,False,0.0,0.001254,0.007022,0.001254,0.007022


In [115]:
import ast
candidate_params = params_grid_lgb_embedding_v4['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]
candidate_params

[{'boosting_type': 'dart',
  'num_leaves': 63,
  'max_depth': 6,
  'learning_rate': 0.004145024286040948,
  'n_estimators': 645,
  'subsample_for_bin': 163333,
  'min_split_gain': 0.22879816549162246,
  'min_child_weight': 0.0020319809838424984,
  'min_child_samples': 49,
  'colsample_bytree': 0.6564274505389015,
  'reg_alpha': 8.47920513131841,
  'reg_lambda': 1.9225178118966844,
  'random_state': 42,
  'n_jobs': -1,
  'verbosity': -1,
  'objective': 'binary',
  'metric': 'binary_logloss',
  'is_unbalance': False,
  'subsample': 0.88128522691469,
  'subsample_freq': 2,
  'drop_rate': 0.4473539092600891,
  'max_drop': 27,
  'skip_drop': 0.8074401551640625,
  'xgboost_dart_mode': True},
 {'boosting_type': 'dart',
  'num_leaves': 13,
  'max_depth': 4,
  'learning_rate': 0.032356740642956634,
  'n_estimators': 779,
  'subsample_for_bin': 57721,
  'min_split_gain': 0.27864646423661144,
  'min_child_weight': 4.295992790539128,
  'min_child_samples': 44,
  'colsample_bytree': 0.6507132052319

In [123]:
params_perf = run_experiment_batch(
    lgb.LGBMClassifier,
    df_embeddings_v4,
    df_embeddings_v4[df_embeddings_v4['mid_sample_type']=='3-oot'],
    selected_features_v4,
    'flag',
    candidate_params,
    report_dir = 'report_embedding_v4/',
    ledger_path = 'params_log_embedding_v4.csv',
    sample_type_col = 'mid_sample_type',
    date_col = 'apply_date_new'
    
)

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_embedding_v4/anti_overfitting-t0-20260708175708-1dec1a03.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_embedding_v4/anti_overfitting-t1-20260708175713-400ad7f3.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_embedding_v4/anti_overfitting-t2-20260708175717-61a17da1.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_embedding_v4/anti_overfitting-t3-20260708175722-1fe1e42b.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_embedding_v4/anti_overfitting-t4-20260708175728

In [146]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
params = {'boosting_type': 'dart', 'num_leaves': 5, 'max_depth': 3, 'learning_rate': 0.02260914149876286, 'n_estimators': 40, 
'min_split_gain': 0.10374114228057249, 'min_child_weight': 0.41651374331523994, 
'min_child_samples': 61, 'colsample_bytree': 0.7426335695527173, 'reg_alpha': 0.10006885458259604218, 
'reg_lambda': 1.24219256423539123, 'random_state': 42, 'n_jobs': -1, 'verbosity': -1, 'objective': 'binary', 
'metric': 'binary_logloss', 'is_unbalance': True, 'subsample': 0.8252604112678665, 'subsample_freq': 1, 'drop_rate': 0.1426912961739955, 'max_drop': 2, 'skip_drop': 0.727328849205757, 'xgboost_dart_mode': True}

skf = StratifiedKFold(n_splits=4,shuffle=True,random_state=20)
train_auc = []
test_auc = []
for train_idx, test_idx in skf.split(df_embeddings_v4[selected_features_v4], df_embeddings_v4['flag']):
    X_train, X_test = df_embeddings_v4[selected_features_v4].iloc[train_idx], df_embeddings_v4[selected_features_v4].iloc[test_idx]
    y_train, y_test = df_embeddings_v4['flag'].iloc[train_idx], df_embeddings_v4['flag'].iloc[test_idx]
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    # 预测训练集和验证集，计算auc

    train_pred = model.predict_proba(X_train)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]
    train_auc.append(roc_auc_score(y_train, train_pred))
    test_auc.append(roc_auc_score(y_test, test_pred))
    if abs(train_auc[-1] - test_auc[-1]) < 0.02:
        joblib.dump(model, 'models/model_embedding_v4_not_overfit.pkl')
    print(f'Train AUC: {train_auc[-1]:.4f}, Test AUC: {test_auc[-1]:.4f}')
    
np.mean(train_auc), np.mean(test_auc)

Train AUC: 0.5858, Test AUC: 0.5488
Train AUC: 0.5850, Test AUC: 0.5485
Train AUC: 0.5777, Test AUC: 0.5662
Train AUC: 0.5827, Test AUC: 0.5475


(0.5828017084887828, 0.5527515490490997)

In [147]:
model = joblib.load('models/model_embedding_v4_not_overfit.pkl')

model.fit(df_embeddings_v4[selected_features_v4], df_embeddings_v4['flag'])
df_embeddings_v4['prob_embeddings_v4'] = model.predict_proba(df_embeddings_v4[selected_features_v4])[:,1]

mdpr = ModelDeliveryReport(
    df_embeddings_v4,
    'flag',
    'prob_embeddings_v4',
    'apply_date_new',
    'partner_name',
)
mdpr.generate_model_performance()

,样本类型,Good,Bad,Total,Bad_Rate,KS,AUC,TOP10_lift,PSI
0,all,125109,3660,128769,0.0284,0.1139,0.5779,1.5209,0.0000
1,华通-2026-03,59876,1966,61842,0.0318,0.1087,0.5699,1.4800,0.0000
2,华通-2026-04至05,54478,1198,55676,0.0215,0.1226,0.5829,1.5517,0.0000
3,华通-2026-02,10755,496,11251,0.0441,0.1255,0.5777,1.4581,0.0000


In [149]:
df_embeddings_v4.head()

,serie_no,id_card_md5,phonenum_md5(param),app_date,code,hxe1,hxe10,hxe11,hxe12,hxe13,...,hcu96,hcu97,hcu98,hcu99,month,partner_name,mid_sample_type,id_number,flag,prob_embeddings_v4
0,18257,baf799226f4ecc5e6e0155c51cec94ce,0eff48907cbf6c40e1631941d344a123,20260331,0,0.000000,0.0,0.144249,0.0,0.0,...,0.3304,0.2334,0.0805,-0.4599,202603,华通-2026-03,1-train,baf799226f4ecc5e6e0155c51cec94ce,0.0,0.261673
1,1863,2ba1bb299ebba228bd46f85f3c7e4835,107cf7ab8f4371239c326bfd91ab76cc,20260404,0,0.025855,0.0,0.144249,0.0,0.0,...,0.7351,0.3830,0.0228,-0.5502,202604,华通-2026-04至05,3-oot,2ba1bb299ebba228bd46f85f3c7e4835,0.0,0.282888
2,42351,c706ac557deafcdad6b2650c5f1b48a9,1d9606a31ea40cdc406640b6bab13034,20260310,0,0.000000,0.0,0.122617,0.0,0.0,...,0.9448,0.7116,-0.0313,-1.6431,202603,华通-2026-03,1-train,c706ac557deafcdad6b2650c5f1b48a9,0.0,0.310997
3,14423,e5d7023d318bdb39b1556849266e4d2d,1e23d36c81dc673db857661f62ba701a,20260402,0,0.415383,0.0,0.000000,0.0,0.0,...,1.2965,0.5496,0.2758,-0.0433,202604,华通-2026-04至05,3-oot,e5d7023d318bdb39b1556849266e4d2d,0.0,0.338233
4,3061,41849b7da94584575f67ffd3f36ed2b9,210dfeec094cf2c5b7980e82a845ffc9,20260326,0,0.254720,0.0,0.222043,0.0,0.0,...,1.0829,0.3245,-0.2280,-0.8750,202603,华通-2026-03,1-train,41849b7da94584575f67ffd3f36ed2b9,0.0,0.269742


In [156]:
df_embeddings_v4.rename(columns={'test_id':'serie_no'},inplace=True)

In [157]:
df_embeddings_v4[['serie_no','id_card_md5','prob_embeddings_v4']].to_csv('ym_v4_embd_prob.csv',index=False)

In [158]:
df_embeddings_v4.shape

(128769, 354)

In [122]:
joblib.dump(model, 'models/model_embedding_v4.pkl')

['models/model_embedding_v4.pkl']

In [204]:
{'boosting_type': 'dart', 'num_leaves': 7, 'max_depth': 3, 'learning_rate': 0.002260914149876286, 'n_estimators': 1486, 
'subsample_for_bin': 127686, 'min_split_gain': 0.10374114228057249, 'min_child_weight': 0.41651374331523994, 
'min_child_samples': 61, 'colsample_bytree': 0.7426335695527173, 'reg_alpha': 0.0006885458259604218, 
'reg_lambda': 0.24219256423539123, 'random_state': 42, 'n_jobs': -1, 'verbosity': -1, 'objective': 'binary', 
'metric': 'binary_logloss', 'is_unbalance': False, 'subsample': 0.8252604112678665, 'subsample_freq': 1, 'drop_rate': 0.1426912961739955, 'max_drop': 2, 'skip_drop': 0.727328849205757, 'xgboost_dart_mode': True}


model.fit(df_embeddings_v4[feas_embeddings_v4], df_embeddings_v4['flag'])
df_embeddings_v4['prob_embeddings_v4'] = model.predict_proba(df_embeddings_v4[feas_embeddings_v4])[:,1]

mdpr = ModelDeliveryReport(
    df_embeddings_v4,
    'flag',
    'prob_embeddings_v4',
    'apply_date_new',
    'partner_name',
)
mdpr.generate_model_performance()

,样本类型,Good,Bad,Total,Bad_Rate,KS,AUC,TOP10_lift,PSI
0,all,125109,3660,128769,0.0284,0.1478,0.6054,1.7104,0.0000
1,华通-2026-03,59876,1966,61842,0.0318,0.1473,0.6020,1.6580,0.0000
2,华通-2026-04至05,54478,1198,55676,0.0215,0.1482,0.6067,1.7778,0.0000
3,华通-2026-02,10755,496,11251,0.0441,0.1515,0.5953,1.5526,0.0000


In [104]:
tuner =  AntiOverfittingTuner(
    model_class=lgb.LGBMClassifier,
    model_type='lgb',
    direction='maximize'
)

params_grid_lgb = tuner.optimize_anti_overfitting(
    df_v3[df_v3['mid_sample_type']=='1-train'][selected_features_v3],
    df_v3[df_v3['mid_sample_type']=='1-train']['flag'],
    n_trials=30
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [105]:
pd.DataFrame(params_grid_lgb['performance_history']).to_csv('params_log_v3.csv',index=False)

In [106]:
params_grid_lgb_v3 = pd.read_csv('params_log_v3.csv')
params_grid_lgb_v3 = params_grid_lgb_v3[params_grid_lgb_v3['is_overfitting']==False]
params_grid_lgb_v3

,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
7,7,"{'boosting_type': 'dart', 'num_leaves': 13, 'm...",0.533024,0.513419,0.513419,0.533024,0.513419,0.062856,0.044386,0.019605,0.018470,False,0.0,0.007385,0.007550,0.007385,0.007550
14,14,"{'boosting_type': 'dart', 'num_leaves': 7, 'ma...",0.552314,0.530318,0.530318,0.552314,0.530318,0.091753,0.069334,0.021996,0.022418,False,0.0,0.006947,0.016747,0.006947,0.016747
23,23,"{'boosting_type': 'dart', 'num_leaves': 15, 'm...",0.528592,0.510984,0.510984,0.528592,0.510984,0.061606,0.042166,0.017608,0.019440,False,0.0,0.005251,0.006105,0.005251,0.006105
24,24,"{'boosting_type': 'dart', 'num_leaves': 7, 'ma...",0.542934,0.519093,0.519093,0.542934,0.519093,0.080707,0.051291,0.023841,0.029417,False,0.0,0.002663,0.009824,0.002663,0.009824
25,25,"{'boosting_type': 'dart', 'num_leaves': 15, 'm...",0.461597,0.479358,0.479358,0.461597,0.479358,0.007256,0.014822,0.017760,0.007566,False,0.0,0.011124,0.010999,0.011124,0.010999


In [107]:
import ast
candidate_params = params_grid_lgb_v3['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]
candidate_params

[{'boosting_type': 'dart',
  'num_leaves': 13,
  'max_depth': 4,
  'learning_rate': 0.032356740642956634,
  'n_estimators': 779,
  'subsample_for_bin': 57721,
  'min_split_gain': 0.27864646423661144,
  'min_child_weight': 4.295992790539128,
  'min_child_samples': 44,
  'colsample_bytree': 0.6507132052319281,
  'reg_alpha': 0.039319117678150585,
  'reg_lambda': 16.786595382063055,
  'random_state': 42,
  'n_jobs': -1,
  'verbosity': -1,
  'objective': 'binary',
  'metric': 'binary_logloss',
  'is_unbalance': False,
  'subsample': 0.8665668653650511,
  'subsample_freq': 2,
  'drop_rate': 0.3668260108198112,
  'max_drop': 6,
  'skip_drop': 0.6323058305935795,
  'xgboost_dart_mode': True},
 {'boosting_type': 'dart',
  'num_leaves': 7,
  'max_depth': 3,
  'learning_rate': 0.013846699331327199,
  'n_estimators': 1224,
  'subsample_for_bin': 149746,
  'min_split_gain': 0.390838548424289,
  'min_child_weight': 0.22662669367988195,
  'min_child_samples': 35,
  'colsample_bytree': 0.618943369112

In [108]:
params_perf = run_experiment_batch(
    lgb.LGBMClassifier,
    df_v3[df_v3['mid_sample_type']=='1-train'],
    df_v3[df_v3['mid_sample_type']=='3-oot'],
    selected_features_v3,
    'flag',
    candidate_params,
    report_dir = 'report_embedding_v3/',
    ledger_path = 'params_log_v3.csv',
    sample_type_col = 'mid_sample_type',
    date_col = 'apply_date_new'
    
)

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_embedding_v3/anti_overfitting-t0-20260708164024-400ad7f3.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_embedding_v3/anti_overfitting-t1-20260708164028-1fe1e42b.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_embedding_v3/anti_overfitting-t2-20260708164032-3b0ccba6.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_embedding_v3/anti_overfitting-t3-20260708164036-68a6cb3d.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_embedding_v3/anti_overfitting-t4-20260708164040

# LR融合

In [232]:
al_df = pd.read_csv('ali_prob.csv')
al_df.head()

,serie_no,ali_88_prob_keep_oot_4_5m,ali_88_prob_no_oot,ali_88_prob_no_oot_gap_auc_002
0,1,0.034181,0.030180,0.033990
1,2,0.033811,0.028276,0.027606
2,3,0.035362,0.030181,0.031936
3,4,0.033315,0.027863,0.026289
4,5,0.034104,0.029549,0.031649


In [233]:
df_lr = df_embeddings_v4[['serie_no','prob_embeddings_v4','flag']].merge(al_df,on='serie_no')
df_lr.head()

,serie_no,prob_embeddings_v4,flag,ali_88_prob_keep_oot_4_5m,ali_88_prob_no_oot,ali_88_prob_no_oot_gap_auc_002
0,18257,0.098796,0.0,0.033285,0.024459,0.022148
1,1863,0.106963,0.0,0.033353,0.026158,0.024307
2,42351,0.117867,0.0,0.035180,0.029374,0.029832
3,14423,0.124353,0.0,0.033633,0.029441,0.032937
4,3061,0.101311,0.0,0.032668,0.027009,0.024874


In [234]:
df_lr.shape

(128769, 6)

In [57]:
def LogitTransformation(prob):
    if 0<prob<1:
        return np.log(prob/(1-prob))
    else:
        return 0


def Prob2Score(prob, basePoint, PDO):
    y = np.log(prob/(1-prob))
    return basePoint+PDO/np.log(2)*(-y)


def AdjustScore(prob, min_score, max_score, basePoint, PDO):
    if prob <= 0 or prob >= 1:
        return 0

    score = Prob2Score(prob, basePoint, PDO)
    if score < min_score:
        score = min_score
    if score > max_score:
        score = max_score

    score_new = int(round((score-min_score)*600.0/(max_score - min_score) + 300,0))
    if score_new < 300:
        score_new = 300
    if score_new > 900:
        score_new = 900
    return int(score_new)

In [235]:
def get_valid_prob(x):

    if x <= 0 or x >= 1:
        y = 0
    else:
        y = 1
    return y

sub_probs = ['prob_embeddings_v4','ali_88_prob_no_oot_gap_auc_002']
feas = []
for fea in sub_probs:
    df_lr[f'{fea}_fill'] = df_lr[fea].fillna(0)
    df_lr[f'{fea}_valid'] = df_lr[f'{fea}_fill'].apply(get_valid_prob)
    df_lr[f'{fea}_trans'] = df_lr[f'{fea}_fill'].apply(LogitTransformation)
    feas.append(f'{fea}_trans')
    feas.append(f'{fea}_valid')

In [246]:
tuner.optimize_anti_overfitting??

Signature:
tuner.optimize_anti_overfitting(
    X,
    y,
    objective_function: Union[model_tools.optimization.objectives.ObjectiveFunction, NoneType] = None,
    n_trials: int = 100,
    cv_folds: int = 5,
    n_jobs: int = 1,
    show_progress_bar: bool = True,
    weight_col: Union[str, NoneType] = None,
) -> Dict[str, Any]
Source:   
    def optimize_anti_overfitting(
        self,
        X,
        y,
        objective_function: Optional[ObjectiveFunction] = None,
        n_trials: int = 100,
        cv_folds: int = 5,
        n_jobs: int = 1,
        show_progress_bar: bool = True,
        weight_col: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        执行防过拟合优化（使用纯交叉验证）

        Args:
            X: 特征数据
            y: 目标变量
            objective_function: 目标函数，如None则使用AUC
            n_trials: 试验次数
            cv_folds: 交叉验证折数
            n_jobs: 并行数
            show_progress_bar: 是否显示进度条
            weight_col: 权重列名，如果传入则使用该列作为样本权重

        Returns:
            

In [ ]:


tuner = AntiOverfittingTuner(LogisticRegression,'lr')
params_grid_lr = tuner.optimize_anti_overfitting(
    df_lr[feas],
    df_lr['flag'],
    n_trials=30
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [237]:
pd.DataFrame(params_grid_lr['performance_history']).to_csv('params_log_lr_ov.csv',index=False)

In [238]:
params_grid_lr = pd.read_csv('params_log_lr_ov.csv')
params_grid_lr = params_grid_lr[params_grid_lr['is_overfitting']==False]
params_grid_lr

,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
0,0,"{'C': 15.741890047456641, 'penalty': 'l1', 'so...",0.622142,0.621958,0.621958,0.622142,0.621958,0.174384,0.182188,0.000183,0.007805,False,0.0,0.002089,0.008424,0.002089,0.008424
1,3,"{'C': 0.0015777663630582469, 'penalty': None, ...",0.622143,0.621962,0.621962,0.622143,0.621962,0.174471,0.182056,0.000181,0.007585,False,0.0,0.002089,0.008420,0.002089,0.008420
2,5,"{'C': 0.0005397956855996448, 'penalty': 'elast...",0.605261,0.605230,0.605230,0.605261,0.605230,0.150171,0.156914,0.000030,0.006743,False,0.0,0.002431,0.010061,0.002431,0.010061
3,10,"{'C': 0.000769639183009799, 'penalty': None, '...",0.622143,0.621962,0.621962,0.622143,0.621962,0.174471,0.182056,0.000181,0.007585,False,0.0,0.002089,0.008420,0.002089,0.008420
4,11,"{'C': 0.0005350071576967551, 'penalty': None, ...",0.622143,0.621962,0.621962,0.622143,0.621962,0.174471,0.182056,0.000181,0.007585,False,0.0,0.002089,0.008420,0.002089,0.008420
5,12,"{'C': 0.01009570399417486, 'penalty': None, 's...",0.622143,0.621962,0.621962,0.622143,0.621962,0.174471,0.182056,0.000181,0.007585,False,0.0,0.002089,0.008420,0.002089,0.008420
6,13,"{'C': 0.04254242912496109, 'penalty': None, 's...",0.622143,0.621962,0.621962,0.622143,0.621962,0.174471,0.182056,0.000181,0.007585,False,0.0,0.002089,0.008420,0.002089,0.008420
7,15,"{'C': 0.00010847199548376785, 'penalty': 'l2',...",0.613698,0.613657,0.613657,0.613698,0.613657,0.167183,0.173636,0.000041,0.006454,False,0.0,0.002180,0.009721,0.002180,0.009721
8,16,"{'C': 0.004961338420711863, 'penalty': None, '...",0.622143,0.621962,0.621962,0.622143,0.621962,0.174471,0.182056,0.000181,0.007585,False,0.0,0.002089,0.008420,0.002089,0.008420
9,17,"{'C': 1.1045487630580906, 'penalty': None, 'so...",0.622143,0.621962,0.621962,0.622143,0.621962,0.174471,0.182056,0.000181,0.007585,False,0.0,0.002089,0.008420,0.002089,0.008420


In [239]:

import ast
candidate_params = params_grid_lr['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]
candidate_params

[{'C': 15.741890047456641,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 641,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0015777663630582469,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 563,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0005397956855996448,
  'penalty': 'elasticnet',
  'solver': 'saga',
  'max_iter': 546,
  'random_state': 42,
  'l1_ratio': 0.034388521115218396,
  'n_jobs': -1},
 {'C': 0.000769639183009799,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 116,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0005350071576967551,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 137,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.01009570399417486,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 291,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.04254242912496109,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 942,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.00010847199548376785,
  'penalty': 'l2',
  'solver': 'saga',
  'ma

In [240]:
df_embeddings_v4.columns

Index(['serie_no', 'id_card_md5', 'phonenum_md5(param)', 'app_date', 'code',
       'hxe1', 'hxe10', 'hxe11', 'hxe12', 'hxe13',
       ...
       'hcu96', 'hcu97', 'hcu98', 'hcu99', 'month', 'partner_name',
       'mid_sample_type', 'id_number', 'flag', 'prob_embeddings_v4'],
      dtype='object', length=354)

In [242]:
df_lr = df_lr.merge(df_embeddings_v4[['serie_no','app_date','partner_name']],on='serie_no')

In [241]:
model_lr = LogisticRegression(**candidate_params[0])
model_lr.fit(
    df_lr[feas],
    df_lr['flag'],
)
df_lr['lr_ov'] = model_lr.predict_proba(df_lr[feas])[:,1]

In [181]:
df_lr.columns

Index(['serie_no', 'prob_embeddings_v4', 'flag', 'ali_88_prob_keep_oot_4_5m',
       'ali_88_prob_no_oot', 'prob_embeddings_v4_fill',
       'prob_embeddings_v4_valid', 'prob_embeddings_v4_trans',
       'ali_88_prob_no_oot_fill', 'ali_88_prob_no_oot_valid',
       'ali_88_prob_no_oot_trans', 'month', 'lr_no_oot', 'app_date'],
      dtype='object')

In [243]:
mdrp  = ModelDeliveryReport(
    df_lr,
    'flag',
    'lr_ov',
    'app_date',
    'partner_name',
)
mdrp.generate_full_report(save_path='report_lr/lr_report.xlsx')

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_lr/lr_report.xlsx


{'样本情况':      Month    Good   Bad   Total Bad_Rate
 0  1970-01  125109  3660  128769   0.0284
 1       合计  125109  3660  128769   0.0284,
 '模型效果':             样本类型    Good   Bad   Total Bad_Rate      KS     AUC TOP10_lift  \
 0            all  125109  3660  128769   0.0284  0.1735  0.6221     2.0300   
 1     华通-2026-03   59876  1966   61842   0.0318  0.1668  0.6143     1.9580   
 2  华通-2026-04至05   54478  1198   55676   0.0215  0.1759  0.6245     1.9948   
 3     华通-2026-02   10755   496   11251   0.0441  0.2075  0.6297     2.1776   
 
       PSI  
 0  0.0000  
 1  0.0000  
 2  0.0000  
 3  0.0000  ,
 '模型效果分箱':              样本类型                 分箱     好用户   坏用户     总用户   总用户占比   坏用户占比  \
 0             all    (0.0452, 0.203]   12134   743   12877  0.1000  0.2030   
 1             all   (0.0373, 0.0452]   12372   505   12877  0.1000  0.1380   
 2             all   (0.0326, 0.0373]   12442   435   12877  0.1000  0.1189   
 3             all    (0.029, 0.0326]   12490   387   12877  0.10

In [245]:
mdrp.generate_model_performance()

,样本类型,Good,Bad,Total,Bad_Rate,KS,AUC,TOP10_lift,PSI
0,all,125109,3660,128769,0.0284,0.1735,0.6221,2.0300,0.0000
1,华通-2026-03,59876,1966,61842,0.0318,0.1668,0.6143,1.9580,0.0000
2,华通-2026-04至05,54478,1198,55676,0.0215,0.1759,0.6245,1.9948,0.0000
3,华通-2026-02,10755,496,11251,0.0441,0.2075,0.6297,2.1776,0.0000


In [182]:
model_perf = ModelDeliveryReport(df_lr,'flag',['prob_embeddings_v4','ali_88_prob_no_oot','lr_no_oot'],'app_date','month','intersection')
model_perf_df = model_perf.generate_model_performance()
model_perf_df = model_perf_df.sort_values('样本类型',ascending=True).reset_index(drop=True)
model_perf_df = model_perf_df[[col for col in model_perf_df.columns if 'PSI_' not in col]]
model_perf_df.to_csv('model_perf_intersection.csv')
model_perf_df

,样本类型,Good,Bad,Total,Bad_Rate,KS_prob_embeddings_v4,AUC_prob_embeddings_v4,TOP10_lift_prob_embeddings_v4,KS_ali_88_prob_no_oot,AUC_ali_88_prob_no_oot,TOP10_lift_ali_88_prob_no_oot,KS_lr_no_oot,AUC_lr_no_oot,TOP10_lift_lr_no_oot
0,202602,10755,496,11251,0.0441,0.1255,0.5777,1.4581,0.1858,0.6115,1.9558,0.1964,0.6259,1.9357
1,202603,59876,1966,61842,0.0318,0.1087,0.5699,1.4800,0.1287,0.5878,1.6376,0.1487,0.6055,1.8156
2,202604,54317,1197,55514,0.0216,0.1225,0.5829,1.5538,0.1283,0.5929,1.7960,0.1687,0.6188,1.9463
3,202605,161,1,162,0.0062,0.7143,0.7143,0.0000,0.4783,0.4783,0.0000,0.6149,0.6149,0.0000
4,all,125109,3660,128769,0.0284,0.1139,0.5779,1.5209,0.1346,0.5935,1.7541,0.1620,0.6148,1.8798


In [185]:
# 蜜蜂标品
df_mifeng = pd.read_csv('华通银行_蜜蜂分.csv',encoding='gbk')
df_mifeng.head()

,serie_no,rn,cif_cust_no,make_loan_date,姓名,身份证,手机号,蜜蜂分A_070,蜜蜂分A_063,蜜蜂分A_064,蜜蜂分A_071,蜜蜂分A_077,蜜蜂分A_076,蜜蜂分A_077_子分1,蜜蜂分A_077_子分2,蜜蜂分B_062,蜜蜂分B_054,蜜蜂分B_059
0,1,4,100249100519,2026-03-06,474deb1a1a00f2df6fe1426af7aec00d,1e0968a3c477e2a4e5a5d68fb3712d74,fa4fc24bb7eb64ad38befd4c8bb753ae,574,610,586,600,528,598,486,590,523,663,0.023683
1,2,106,100109734289,2026-03-23,49121e96fad562a122ebf2d55106a367,3de08d3162b2aed10933cc74165808b7,b821b26916d2c64ff89872f40d858257,648,617,658,696,726,664,746,629,585,705,0.015142
2,3,114,100253118290,2026-04-05,812d0e8e2ba9281155307138861e674f,89d5fd9ad9ea658b847d6f449908a674,11b515f674b7a9639e1efd47acaddd63,518,616,510,667,748,670,797,633,503,623,0.033417
3,4,120,100220083085,2026-04-08,acc5115681d105bef92fdfe98b9d0ebb,4e861b04b6721d7c9e85421d2aa86018,4a18e4983795edc5c2d5a5d55f94dfbe,665,630,685,663,642,680,649,581,584,725,0.013513
4,5,176,100247676250,2026-02-28,04216d6f3d6e724010e5212c56b4a537,a343b5d96e944690c34ada406b395a4f,6079d651e4d5f0b422a5de7ed38b7410,606,601,646,533,522,518,0,599,539,693,0.024078


In [186]:
df_lr.head()

,serie_no,prob_embeddings_v4,flag,ali_88_prob_keep_oot_4_5m,ali_88_prob_no_oot,prob_embeddings_v4_fill,prob_embeddings_v4_valid,prob_embeddings_v4_trans,ali_88_prob_no_oot_fill,ali_88_prob_no_oot_valid,ali_88_prob_no_oot_trans,month,lr_no_oot,app_date
0,18257,0.261673,0.0,0.033285,0.024459,0.261673,1,-1.037294,0.024459,1,-3.686014,202603,0.013592,20260331
1,1863,0.282888,0.0,0.033353,0.026158,0.282888,1,-0.930182,0.026158,1,-3.617096,202604,0.020364,20260404
2,42351,0.310997,0.0,0.035180,0.029374,0.310997,1,-0.795461,0.029374,1,-3.497822,202603,0.037396,20260310
3,14423,0.338233,0.0,0.033633,0.029441,0.338233,1,-0.671176,0.029441,1,-3.495469,202604,0.046139,20260402
4,3061,0.269742,0.0,0.032668,0.027009,0.269742,1,-0.995933,0.027009,1,-3.584196,202603,0.020313,20260326


In [ ]:
pred_fea = 'prob_embeddings_v4'
df_lr['score_ym'] = df_lr[pred_fea].apply(lambda x: Prob2Score(x, basePoint=800, PDO=100) if pd.notnull(x) else x)
min_score = math.floor(df_lr['score_ym'].min())
max_score = math.ceil(df_lr['score_ym'].max())
df_lr['score_ym'] = df_lr.apply(lambda x: AdjustScore(x['score_ym'], min_score, max_score, 800, 100) if pd.notnull(x['score_ym']) else x, axis=1)

pred_fea = 'ali_88_prob_no_oot'
df_lr['score_ali_88'] = df_lr[pred_fea].apply(lambda x: Prob2Score(x, basePoint=800, PDO=100) if pd.notnull(x) else x)
min_score = math.floor(df_lr['score_ali_88'].min())
max_score = math.ceil(df_lr['score_ali_88'].max())
df_lr['score_ali_88'] = df_lr.apply(lambda x: AdjustScore(x['score_ali_88'], min_score, max_score, 800, 100) if pd.notnull(x['score_ali_88']) else x, axis=1)

,serie_no,prob_embeddings_v4,flag,ali_88_prob_keep_oot_4_5m,ali_88_prob_no_oot,prob_embeddings_v4_fill,prob_embeddings_v4_valid,prob_embeddings_v4_trans,ali_88_prob_no_oot_fill,ali_88_prob_no_oot_valid,ali_88_prob_no_oot_trans,month,lr_no_oot,app_date,score_ym,score_ali_88
0,18257,0.261673,0.0,0.033285,0.024459,0.261673,1,-1.037294,0.024459,1,-3.686014,202603,0.013592,20260331,949.649835,1331.779461
1,1863,0.282888,0.0,0.033353,0.026158,0.282888,1,-0.930182,0.026158,1,-3.617096,202604,0.020364,20260404,934.196964,1321.836652
2,42351,0.310997,0.0,0.035180,0.029374,0.310997,1,-0.795461,0.029374,1,-3.497822,202603,0.037396,20260310,914.760834,1304.629111
3,14423,0.338233,0.0,0.033633,0.029441,0.338233,1,-0.671176,0.029441,1,-3.495469,202604,0.046139,20260402,896.830289,1304.289526
4,3061,0.269742,0.0,0.032668,0.027009,0.269742,1,-0.995933,0.027009,1,-3.584196,202603,0.020313,20260326,943.682708,1317.090134


In [195]:
AdjustScore??

Signature: AdjustScore(prob, min_score, max_score, basePoint, PDO)
Docstring: <no docstring>
Source:   
def AdjustScore(prob, min_score, max_score, basePoint, PDO):
    if prob <= 0 or prob >= 1:
        return 0

    score = Prob2Score(prob, basePoint, PDO)
    if score < min_score:
        score = min_score
    if score > max_score:
        score = max_score

    score_new = int(round((score-min_score)*600.0/(max_score - min_score) + 300,0))
    if score_new < 300:
        score_new = 300
    if score_new > 900:
        score_new = 900
    return int(score_new)
File:      /var/folders/v1/ch71pmtn60dbr0m_t7s_ch1m0000gn/T/ipykernel_12802/2300116844.py
Type:      function

In [197]:
pred_fea = 'prob_embeddings_v4'
df_lr['score_ym_'] = df_lr[pred_fea].apply(lambda x: Prob2Score(x, basePoint=800, PDO=100) if pd.notnull(x) else x)
min_score = math.floor(df_lr['score_ym_'].min())
max_score = math.ceil(df_lr['score_ym_'].max())
df_lr['score_ym'] = df_lr['prob_embeddings_v4'].apply(lambda x: AdjustScore(x, min_score, max_score, 800, 100) if pd.notnull(x) else x)

pred_fea = 'ali_88_prob_no_oot'
df_lr['score_ali_88_'] = df_lr[pred_fea].apply(lambda x: Prob2Score(x, basePoint=800, PDO=100) if pd.notnull(x) else x)
min_score = math.floor(df_lr['score_ali_88_'].min())
max_score = math.ceil(df_lr['score_ali_88_'].max())
df_lr['score_ali_88'] = df_lr['ali_88_prob_no_oot'].apply(lambda x: AdjustScore(x, min_score, max_score, 800, 100) if pd.notnull(x) else x)
df_lr.head()

,serie_no,prob_embeddings_v4,flag,ali_88_prob_keep_oot_4_5m,ali_88_prob_no_oot,prob_embeddings_v4_fill,prob_embeddings_v4_valid,prob_embeddings_v4_trans,ali_88_prob_no_oot_fill,ali_88_prob_no_oot_valid,ali_88_prob_no_oot_trans,month,lr_no_oot,app_date,score_ym,score_ali_88,score_ym_,score_ali_88_
0,18257,0.261673,0.0,0.033285,0.024459,0.261673,1,-1.037294,0.024459,1,-3.686014,202603,0.013592,20260331,571,811,949.649835,1331.779461
1,1863,0.282888,0.0,0.033353,0.026158,0.282888,1,-0.930182,0.026158,1,-3.617096,202604,0.020364,20260404,518,773,934.196964,1321.836652
2,42351,0.310997,0.0,0.035180,0.029374,0.310997,1,-0.795461,0.029374,1,-3.497822,202603,0.037396,20260310,451,707,914.760834,1304.629111
3,14423,0.338233,0.0,0.033633,0.029441,0.338233,1,-0.671176,0.029441,1,-3.495469,202604,0.046139,20260402,389,706,896.830289,1304.289526
4,3061,0.269742,0.0,0.032668,0.027009,0.269742,1,-0.995933,0.027009,1,-3.584196,202603,0.020313,20260326,551,755,943.682708,1317.090134


In [196]:
min_score, max_score

(1198, 1355)

In [ ]:
df_lr.merge(df_mifeng)

In [199]:
df_mifeng = df_mifeng.merge(df_lr[['serie_no','flag','month','app_date','score_ym',	'score_ali_88']],on='serie_no')

In [200]:
df_mifeng.columns

Index(['serie_no', 'rn', 'cif_cust_no', 'make_loan_date', '姓名', '身份证', '手机号',
       '蜜蜂分A_070', '蜜蜂分A_063', '蜜蜂分A_064', '蜜蜂分A_071', '蜜蜂分A_077', '蜜蜂分A_076',
       '蜜蜂分A_077_子分1', '蜜蜂分A_077_子分2', '蜜蜂分B_062', '蜜蜂分B_054', '蜜蜂分B_059',
       'flag', 'month', 'app_date', 'score_ym', 'score_ali_88'],
      dtype='object')

In [202]:
from itertools import combinations
all_feas = ['蜜蜂分A_070', '蜜蜂分A_063', '蜜蜂分A_064', '蜜蜂分A_071', '蜜蜂分A_077', 
'蜜蜂分A_076','蜜蜂分A_077_子分1', '蜜蜂分A_077_子分2', '蜜蜂分B_062', '蜜蜂分B_054', '蜜蜂分B_059',
'score_ym', 'score_ali_88']

for fea1, fea2 in combinations(all_feas, 2):
    feas = [fea1, fea2]
    print(f"正在训练: {fea1} + {fea2}")
    tuner = AntiOverfittingTuner(LogisticRegression, 'lr')
    params_grid_lr = tuner.optimize_anti_overfitting(
        df_mifeng[feas],
        df_mifeng['flag'],
        n_trials=30
    )
    
    csv_name = f'params_log_lr_{fea1}_{fea2}.csv'
    pd.DataFrame(params_grid_lr['performance_history']).to_csv(csv_name, index=False)
    try:
        params_grid_lr_df = pd.read_csv(csv_name)
    except Exception:
        print(f"  跳过 {fea1} + {fea2}，读取CSV失败")
        continue
    params_grid_lr_df = params_grid_lr_df[params_grid_lr_df['is_overfitting'] == False]
    
    if len(params_grid_lr_df) == 0:
        print(f"  跳过 {fea1} + {fea2}，无非过拟合参数")
        continue
    candidate_params = params_grid_lr_df['params'].to_list()
    candidate_params = [ast.literal_eval(s) for s in candidate_params]
        
    model_lr = LogisticRegression(**candidate_params[0])
    model_lr.fit(
        df_mifeng[feas],
        df_mifeng['flag'],
    )
        
    col_name = f'lr_{fea1}_{fea2}'
    df_mifeng[col_name] = model_lr.predict_proba(df_mifeng[feas])[:, 1]
    print(f"  完成，结果列: {col_name}")


正在训练: 蜜蜂分A_070 + 蜜蜂分A_063


  0%|          | 0/30 [00:00<?, ?it/s]

  完成，结果列: lr_蜜蜂分A_070_蜜蜂分A_063
正在训练: 蜜蜂分A_070 + 蜜蜂分A_064


  0%|          | 0/30 [00:00<?, ?it/s]

  完成，结果列: lr_蜜蜂分A_070_蜜蜂分A_064
正在训练: 蜜蜂分A_070 + 蜜蜂分A_071


  0%|          | 0/30 [00:00<?, ?it/s]

... (147 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

  跳过 蜜蜂分B_059 + score_ali_88，读取CSV失败
正在训练: score_ym + score_ali_88


  0%|          | 0/30 [00:00<?, ?it/s]

  完成，结果列: lr_score_ym_score_ali_88


In [205]:
lr_probs = [col for col in df_mifeng.columns if 'lr_' in col and 'prob' not in col]
lr_probs

['lr_蜜蜂分A_070_蜜蜂分A_063',
 'lr_蜜蜂分A_070_蜜蜂分A_064',
 'lr_蜜蜂分A_070_蜜蜂分A_071',
 'lr_蜜蜂分A_070_蜜蜂分A_077',
 'lr_蜜蜂分A_070_蜜蜂分A_076',
 'lr_蜜蜂分A_070_蜜蜂分A_077_子分1',
 'lr_蜜蜂分A_070_蜜蜂分A_077_子分2',
 'lr_蜜蜂分A_070_蜜蜂分B_062',
 'lr_蜜蜂分A_070_蜜蜂分B_054',
 'lr_蜜蜂分A_070_score_ym',
 'lr_蜜蜂分A_070_score_ali_88',
 'lr_蜜蜂分A_063_蜜蜂分A_064',
 'lr_蜜蜂分A_063_蜜蜂分A_071',
 'lr_蜜蜂分A_063_蜜蜂分A_077',
 'lr_蜜蜂分A_063_蜜蜂分A_076',
 'lr_蜜蜂分A_063_蜜蜂分A_077_子分1',
 'lr_蜜蜂分A_063_蜜蜂分A_077_子分2',
 'lr_蜜蜂分A_063_蜜蜂分B_062',
 'lr_蜜蜂分A_063_蜜蜂分B_054',
 'lr_蜜蜂分A_063_score_ym',
 'lr_蜜蜂分A_063_score_ali_88',
 'lr_蜜蜂分A_064_蜜蜂分A_071',
 'lr_蜜蜂分A_064_蜜蜂分A_077',
 'lr_蜜蜂分A_064_蜜蜂分A_076',
 'lr_蜜蜂分A_064_蜜蜂分A_077_子分1',
 'lr_蜜蜂分A_064_蜜蜂分A_077_子分2',
 'lr_蜜蜂分A_064_蜜蜂分B_062',
 'lr_蜜蜂分A_064_蜜蜂分B_054',
 'lr_蜜蜂分A_064_score_ym',
 'lr_蜜蜂分A_064_score_ali_88',
 'lr_蜜蜂分A_071_蜜蜂分A_077',
 'lr_蜜蜂分A_071_蜜蜂分A_076',
 'lr_蜜蜂分A_071_蜜蜂分A_077_子分1',
 'lr_蜜蜂分A_071_蜜蜂分A_077_子分2',
 'lr_蜜蜂分A_071_蜜蜂分B_062',
 'lr_蜜蜂分A_071_蜜蜂分B_054',
 'lr_蜜蜂分A_071_score_ym',
 'lr_蜜蜂分A_071_score_ali_88',
 '

In [ ]:
lr_probs + ['score_ym', 'score_ali_88']

In [206]:
model_perf = ModelDeliveryReport(df_mifeng,'flag',lr_probs + ['score_ym', 'score_ali_88'],'app_date','month','intersection')
model_perf_df = model_perf.generate_model_performance()
model_perf_df = model_perf_df.sort_values('样本类型',ascending=True).reset_index(drop=True)
model_perf_df = model_perf_df[[col for col in model_perf_df.columns if 'PSI_' not in col]]
model_perf_df.to_csv('model_perf_intersection.csv')
model_perf_df

,样本类型,Good,Bad,Total,Bad_Rate,KS_lr_蜜蜂分A_070_蜜蜂分A_063,AUC_lr_蜜蜂分A_070_蜜蜂分A_063,TOP10_lift_lr_蜜蜂分A_070_蜜蜂分A_063,KS_lr_蜜蜂分A_070_蜜蜂分A_064,AUC_lr_蜜蜂分A_070_蜜蜂分A_064,...,TOP10_lift_lr_蜜蜂分B_054_score_ali_88,KS_lr_score_ym_score_ali_88,AUC_lr_score_ym_score_ali_88,TOP10_lift_lr_score_ym_score_ali_88,KS_score_ym,AUC_score_ym,TOP10_lift_score_ym,KS_score_ali_88,AUC_score_ali_88,TOP10_lift_score_ali_88
0,202602,10755,496,11251,0.0441,0.1510,0.5831,1.3106,0.1506,0.5886,...,1.9760,0.1954,0.6217,1.8550,0.0003,0.4223,0.7682,0.0000,0.3884,0.4760
1,202603,59876,1966,61842,0.0318,0.0971,0.5652,1.4240,0.1182,0.5769,...,1.6783,0.1488,0.6036,1.7142,0.0000,0.4302,0.5733,0.0002,0.4121,0.5544
2,202604,54317,1197,55514,0.0216,0.1277,0.5799,1.6038,0.1434,0.5912,...,1.8711,0.1688,0.6182,1.8628,0.0000,0.4171,0.3787,0.0000,0.4072,0.6327
3,202605,161,1,162,0.0062,0.9565,0.9565,9.5294,0.8199,0.8199,...,0.0000,0.6335,0.6335,0.0000,0.2795,0.2826,0.0000,0.5217,0.5280,0.0000
4,all,125109,3660,128769,0.0284,0.1137,0.5750,1.5058,0.1293,0.5856,...,1.7978,0.1599,0.6132,1.7869,0.0000,0.4221,0.5179,0.0000,0.4064,0.5555


# baidu

In [6]:
df_bd_md5 = pd.read_csv('/Users/mayongzhi/Job/Job_xyd/数据/客户测建模/华通/小雨点0714/1-bdy-md5_report/report/子分V4.0L测试结果_20260714.csv')
df_bd_sha256 = pd.read_csv('/Users/mayongzhi/Job/Job_xyd/数据/客户测建模/华通/小雨点0714/1-bdy-sha256_report/report/子分V4.0L测试结果_20260715.csv')
print(df_bd_md5.shape, df_bd_sha256.shape)
display(df_bd_md5.head(), df_bd_sha256.head())

(506601, 565) (2196253, 565)


,test_id,mobile,apply_date,enc_method,RL4006N0001,RL4006N0002,RL4006N0003,RL4006N0004,RL4006N0005,RL4002N0001,...,RL4001N0185,RL4001N0186,RL4001N0187,RL4001N0188,RL4001N0189,RL4001N0190,RL4001N0191,RL4001N0192,RL4001N0193,RL4001N0194
0,4491256,7e411d405b2b02af0ba63ed08417d8f4,20240717,md5,1.000000,0.453358,1.000000,0.849005,1.000000,0.115330,...,0.0555,0.0803,0.0402,0.1133,0.1893,0.2,0.390773,0.390773,0.215675,0.565872
1,4491258,bc47ece38f6cc62da8501e366b29b8bd,20240705,md5,0.896014,0.742479,0.527342,0.258269,0.009554,NaN,...,0.0404,0.0550,0.0127,0.0210,0.1862,0.0,0.073860,0.073860,0.073860,0.073860
2,4491259,db4ad8c8a88e12b0156c7ce8bc74cec0,20240715,md5,NaN,NaN,NaN,NaN,NaN,0.154212,...,0.0268,0.0544,0.0297,0.0707,0.1349,0.0,0.294892,0.294892,0.294892,0.294892
3,4491266,5238b849d58b526d057dd9349d08f663,20240722,md5,0.972645,0.451079,0.533517,0.258269,0.015393,0.135983,...,0.0294,0.0451,0.0061,0.0093,0.1360,0.2,0.046666,0.046666,0.039130,0.054203
4,4491282,2978be9b1688b31db9500a17e5deaf51,20240720,md5,0.855856,0.799909,0.648553,0.486115,0.107749,0.086284,...,0.0410,0.0629,0.0161,0.0275,0.1865,0.2,0.128850,0.128850,0.039130,0.218569


,test_id,mobile,apply_date,enc_method,RL4006N0001,RL4006N0002,RL4006N0003,RL4006N0004,RL4006N0005,RL4002N0001,...,RL4001N0185,RL4001N0186,RL4001N0187,RL4001N0188,RL4001N0189,RL4001N0190,RL4001N0191,RL4001N0192,RL4001N0193,RL4001N0194
0,-905190,f43d9e92c8c4500a31f80e3029d2bc080d5eb560dbeff5...,20241202,sha256,0.916030,0.692799,0.000000,0.421872,0.003715,0.237591,...,0.0262,0.0412,0.0082,0.0288,0.1124,0.0,0.192262,0.192262,0.192262,0.192262
1,-905189,dea95482bc6c843974225f711a27bb910da36eee748680...,20241130,sha256,0.822497,0.872379,0.570001,0.160641,0.055732,0.050257,...,0.0604,0.0842,0.0234,0.0581,0.2605,0.0,0.175157,0.175157,0.175157,0.175157
2,-905188,9a7165286adc219ab7c1c588c24d82b7aa395f7b0475ba...,20241121,sha256,0.985401,0.524005,0.810926,0.938574,0.324841,0.130542,...,0.0173,0.0318,0.0065,0.0137,0.0778,0.0,0.082543,0.082543,0.082543,0.082543
3,-905187,d6566823079807523d7ee817a6684c1dca5e1527eafdfb...,20240725,sha256,0.928500,0.679125,0.996915,0.421137,0.150212,NaN,...,0.0308,0.0493,0.0208,0.0636,0.1542,0.0,0.102802,0.102802,0.102802,0.102802
4,-905186,89077e35f4a0b2539fb5c4c8096eaaf8a736525ff9f790...,20241104,sha256,NaN,NaN,NaN,NaN,NaN,0.161806,...,0.0420,0.0582,0.0311,0.1417,0.1660,0.0,0.160686,0.160686,0.160686,0.160686


In [7]:
df_label.columns

Index(['1-阿里反欺诈v2标品', 'B068_prob', 'B069_prob', 'apply_date', 'b058_bh_prob',
       'b058_bh_prob_new_export', 'b066_bh_model_prob', 'b066_prob',
       'b066_score', 'b067_bh_prob', 'b067_prob', 'bh_imbank_proba(B058)',
       'c031_bh_prob', 'enc_method', 'flag', 'high_sample_type',
       'hl_jianpu_score_v1', 'hl_yuxin_score_v3_a', 'hl_yuxin_score_v3_b',
       'hl_yuxin_score_v3_c', 'id_number', 'mid_sample_type', 'mobile',
       'model_risk_v9_gtongyong_score', 'model_risk_v9_gxiaojin_score',
       'partner_code', 'partner_name', 'pric_type_code', 'pric_type_desc',
       'test_id', 'ym_A077', '蜜蜂分_中利率版2_2', '蜜蜂分_中利率版2_2_proba', '蜜蜂分_中利率版6_2',
       '蜜蜂分_中利率版6_2_proba', '蜜蜂分_中利率版6_3_proba', '蜜蜂分_中利率版7_2',
       '蜜蜂分_中利率版7_2_proba', '蜜蜂分_中利率版8_1_proba_a073', '蜜蜂分_中利率版8_2_proba_a074',
       '蜜蜂分_中利率版8_3_proba_a075', '蜜蜂分_中利率版_1', '蜜蜂分_中利率版_1_proba',
       '蜜蜂分_高利率版10_proba(B058)', '蜜蜂分_高利率版5_2', '蜜蜂分_高利率版5_2_proba',
       '蜜蜂分_高利率版6_2', '蜜蜂分_高利率版6_2_proba', '蜜蜂分_高利率版7_2', '

In [8]:
df_label['partner_name'].value_counts()

partner_name
海尔(借呗)                    496269
海尔(借呗贷中)                  453769
lexin_type1               391797
lexin_type3               333067
weipingfubang_highirr     308859
海尔(自营贷中)                  274100
海尔(京东贷中)                  250571
海尔(字节星选贷中)                236132
海尔(蚂蚁API贷中)               234715
众安小贷复贷常规                  204681
维新复贷                      168452
海尔(字节智选贷中)                155120
量化派VCC-fpd30_flag         152657
海尔(字节星选)                  143744
海尔(蚂蚁)                    141757
lexin_type4               130714
华通银行-星选                   128769
量化派现金-fpd30_flag          127742
lexin_type2               123679
恒昌客群C                     110187
恒昌_old                    105960
海尔(借呗拒绝其他通过)              101217
量化派VCC-mob3_30_flag        98385
量化派现金-mob3_30_flag         91762
hc_A2_mob6                 91242
hc_A2_mob3                 90703
weipingfubang_lowerirr     86380
你我贷                        78571
lexin_type5                75861
洋钱罐                        706

In [12]:
# 华通银行-星选 
df_label = df_label[df_label['partner_name']=='华通银行-星选']
df_label.shape

(128769, 50)

In [13]:
df_bd_md5 = df_bd_md5.merge(df_label[['test_id','apply_date','flag']],how='inner')
print(df_bd_md5.shape)

(128769, 566)


In [15]:
df_bd_md5.head()

,test_id,mobile,apply_date,enc_method,RL4006N0001,RL4006N0002,RL4006N0003,RL4006N0004,RL4006N0005,RL4002N0001,...,RL4001N0186,RL4001N0187,RL4001N0188,RL4001N0189,RL4001N0190,RL4001N0191,RL4001N0192,RL4001N0193,RL4001N0194,flag
0,6577808,fa4fc24bb7eb64ad38befd4c8bb753ae,20260306,md5,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0450,0.0257,0.0926,0.1514,NaN,NaN,NaN,NaN,NaN,1.0
1,6577809,b821b26916d2c64ff89872f40d858257,20260323,md5,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0473,0.0102,0.0271,0.1317,0.0,0.329103,0.329103,0.329103,0.329103,1.0
2,6577810,11b515f674b7a9639e1efd47acaddd63,20260405,md5,0.936395,0.635825,0.280602,0.0,0.002123,0.269247,...,0.0518,0.0061,0.0110,0.1360,0.0,0.065177,0.065177,0.065177,0.065177,1.0
3,6577811,4a18e4983795edc5c2d5a5d55f94dfbe,20260408,md5,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0471,0.0244,0.0944,0.1372,NaN,NaN,NaN,NaN,NaN,1.0
4,6577812,6079d651e4d5f0b422a5de7ed38b7410,20260228,md5,NaN,NaN,NaN,NaN,NaN,0.321472,...,0.0676,0.0424,0.1609,0.2052,NaN,NaN,NaN,NaN,NaN,1.0


In [14]:
del df_bd_sha256
gc.collect()

61

In [16]:
df_bd_md5['month']= df_bd_md5['apply_date'].astype(str).str[0:6]

df_bd_md5['partner_name'] =df_bd_md5['month'].map(
    {'202602':'华通-2026-02',
    '202603':'华通-2026-03',
    '202604':'华通-2026-04至05',
    '202605':'华通-2026-04至05'}
    )
df_bd_md5['mid_sample_type'] = df_bd_md5['partner_name'].map(
    {'华通-2026-02':'1-train',
    '华通-2026-03':'1-train',
    '华通-2026-04至05':'3-oot'}
    )


In [17]:
df_bd_md5['apply_date_new'] = pd.to_datetime(df_bd_md5['apply_date'], format='%Y%m%d')
df_bd_md5['apply_date_new'].head()

0   2026-03-06
1   2026-03-23
2   2026-04-05
3   2026-04-08
4   2026-02-28
Name: apply_date_new, dtype: datetime64[ns]

In [20]:
# 除去 test_id	mobile	apply_date	enc_method flag 是feas
feas = df_bd_md5.drop(columns=['test_id','mobile','apply_date','enc_method','flag', 'month',
'partner_name','mid_sample_type','apply_date_new']).columns.to_list()

In [21]:
feas

['RL4006N0001',
 'RL4006N0002',
 'RL4006N0003',
 'RL4006N0004',
 'RL4006N0005',
 'RL4002N0001',
 'RL4002N0002',
 'RL4002N0003',
 'RL4002N0004',
 'RL4002N0005',
 'RL4002N0006',
 'RL4002N0007',
 'RL4002N0008',
 'RL4002N0009',
 'RL4002N0010',
 'RL4002N0011',
 'RL4002N0012',
 'RL4002N0013',
 'RL4002N0014',
 'RL4002N0015',
 'RL4002N0016',
 'RL4002N0017',
 'RL4002N0018',
 'RL4002N0021',
 'RL4002N0022',
 'RL4002N0023',
 'RL4002N0027',
 'RL4002N0028',
 'RL4002N0030',
 'RL4002N0031',
 'RL4002N0032',
 'RL4002N0034',
 'RL4002N0035',
 'RL4002N0036',
 'RL4002N0037',
 'RL4002N0038',
 'RL4002N0039',
 'RL4002N0040',
 'RL4002N0041',
 'RL4002N0042',
 'RL4002N0043',
 'RL4002N0044',
 'RL4002N0045',
 'RL4002N0046',
 'RL4002N0047',
 'RL4002N0048',
 'RL4002N0049',
 'RL4002N0050',
 'RL4002N0051',
 'RL4002N0052',
 'RL4005N0005',
 'RL4005N0007',
 'RL4005N0011',
 'RL4005N0013',
 'RL4005N0014',
 'RL4005N0026',
 'RL4005N0047',
 'RL4005N0048',
 'RL4005N0054',
 'RL4005N0055',
 'RL4005N0056',
 'RL4004N0001',
 'RL4004

In [23]:
tuner =  AntiOverfittingTuner(
    model_class=lgb.LGBMClassifier,
    model_type='lgb',
    direction='maximize'
)

params_grid_lr = tuner.optimize_anti_overfitting(
        df_bd_md5[feas],
        df_bd_md5['flag'],
        n_trials=30
    )
pd.DataFrame(params_grid_lr['performance_history']).to_csv(f'params_log_bd_lgb_nooot.csv', index=False)
# 读取最优参数
params_grid_lr_sub = pd.read_csv(f'params_log_bd_lgb_nooot.csv')
params_grid_lr_sub = params_grid_lr_sub[params_grid_lr_sub['is_overfitting']==False]

candidate_params = params_grid_lr_sub['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]

# 评估
params_perf = run_experiment_batch(
    lgb.LGBMClassifier,
    df_bd_md5[feas],
    df_bd_md5['flag'],
    feas,
    'flag',
    candidate_params,
    report_dir = f'report_bd_lgb_nooot/',
    ledger_path = f'params_perf_bd_lgb_nooot.csv',
    sample_type_col = 'partner_name',
    date_col = 'apply_date_new'
)

  0%|          | 0/30 [00:00<?, ?it/s]

NameError: name 'ast' is not defined

In [25]:
run_experiment_batch??

Signature:
run_experiment_batch(
    model_class: 'Any',
    train_data: 'pd.DataFrame',
    oot_data: 'pd.DataFrame',
    feature_cols: 'Sequence[str]',
    target_col: 'str',
    candidate_params: 'List[Dict[str, Any]]',
    report_dir: 'Union[str, Path]',
    ledger_path: 'Union[str, Path]',
    study_name: 'str' = 'anti_overfitting',
    score_col: 'str' = 'score',
    sample_type_col: 'str' = 'sample_type',
    date_col: 'str' = 'date',
    report_features: 'Optional[List[str]]' = None,
    report_feature_names: 'Optional[Dict[str, str]]' = None,
    bin_method: 'str' = 'quantile',
    importance_type: 'str' = 'gain',
) -> 'pd.DataFrame'
Source:   
def run_experiment_batch(
    model_class: Any,
    train_data: pd.DataFrame,
    oot_data: pd.DataFrame,
    feature_cols: Sequence[str],
    target_col: str,
    candidate_params: List[Dict[str, Any]],
    report_dir: Union[str, Path],
    ledger_path: Union[str, Path],
    study_name: str = "anti_overfitting",
    score_col: str = "s

In [42]:
whos(globals(),top=5)

        Variable       Type          Shape  Memory(MB)
0      df_bd_md5  DataFrame  (128769, 571)      602.69
1       df_label  DataFrame   (128769, 50)      127.59
2  df_embeddings  DataFrame   (128769, 86)      104.38
3       df_ym_al  DataFrame    (128769, 6)       15.84
4   df_ymv4_prob  DataFrame    (128769, 3)       12.89


,Variable,Type,Shape,Memory(MB)
0,df_bd_md5,DataFrame,"(128769, 571)",602.69
1,df_label,DataFrame,"(128769, 50)",127.59
2,df_embeddings,DataFrame,"(128769, 86)",104.38
3,df_ym_al,DataFrame,"(128769, 6)",15.84
4,df_ymv4_prob,DataFrame,"(128769, 3)",12.89


In [26]:
df_bd_md5.columns

Index(['test_id', 'mobile', 'apply_date', 'enc_method', 'RL4006N0001',
       'RL4006N0002', 'RL4006N0003', 'RL4006N0004', 'RL4006N0005',
       'RL4002N0001',
       ...
       'RL4001N0190', 'RL4001N0191', 'RL4001N0192', 'RL4001N0193',
       'RL4001N0194', 'flag', 'month', 'partner_name', 'mid_sample_type',
       'apply_date_new'],
      dtype='object', length=570)

In [29]:
import ast
candidate_params = params_grid_lr_sub['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]


params_perf = run_experiment_batch(
    lgb.LGBMClassifier,
    df_bd_md5,
    df_bd_md5[df_bd_md5['mid_sample_type']=='3-oot'],
    feas,
    'flag',
    candidate_params,
    report_dir = f'report_bd_lgb_nooot/',
    ledger_path = f'params_perf_bd_lgb_nooot.csv',
    sample_type_col = 'partner_name',
    date_col = 'apply_date_new'
)

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_bd_lgb_nooot/anti_overfitting-t0-20260716111619-305e3c46.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_bd_lgb_nooot/anti_overfitting-t1-20260716111641-871e72d4.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_bd_lgb_nooot/anti_overfitting-t2-20260716111703-31b146d4.xlsx
✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ TOP10变量有效性分析完成
✓ 变量有效性按月拆分分析完成
✓ TOP10变量稳定性分析完成
✓ 单特征时间分布分析完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_bd_lgb_nooot/anti_overfitting-t3-20260716111723-6435b5fe.xlsx


In [30]:
candidate_params[-1]

{'boosting_type': 'dart',
 'num_leaves': 7,
 'max_depth': 3,
 'learning_rate': 0.0017374724748800178,
 'n_estimators': 1316,
 'subsample_for_bin': 118504,
 'min_split_gain': 0.7500761540623417,
 'min_child_weight': 0.008051998227368915,
 'min_child_samples': 20,
 'colsample_bytree': 0.678795288420621,
 'reg_alpha': 0.0003402124689189236,
 'reg_lambda': 1.356205880483939,
 'random_state': 42,
 'n_jobs': -1,
 'verbosity': -1,
 'objective': 'binary',
 'metric': 'binary_logloss',
 'is_unbalance': False,
 'subsample': 0.647794493736707,
 'subsample_freq': 5,
 'drop_rate': 0.4276524681434952,
 'max_drop': 3,
 'skip_drop': 0.03579182004480519,
 'xgboost_dart_mode': True}

In [ ]:
model_bd_lgb = lgb.LGBMClassifier(**candidate_params[-1])
model_bd_lgb.fit(
    df_bd_md5[feas],
    df_bd_md5['flag']
)
df_bd_md5['bd_lgb_NoOot_prob'] = model_bd_lgb.predict_proba(df_bd_md5[feas])[:,1]


In [32]:
joblib.dump(model_bd_lgb, 'models/bd_lgb_NoOot.pkl')

['models/bd_lgb_NoOot.pkl']

In [33]:
df_bd_md5.head()

,test_id,mobile,apply_date,enc_method,RL4006N0001,RL4006N0002,RL4006N0003,RL4006N0004,RL4006N0005,RL4002N0001,...,RL4001N0191,RL4001N0192,RL4001N0193,RL4001N0194,flag,month,partner_name,mid_sample_type,apply_date_new,bd_lgb_NoOot_prob
0,6577808,fa4fc24bb7eb64ad38befd4c8bb753ae,20260306,md5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,202603,华通-2026-03,1-train,2026-03-06,0.967041
1,6577809,b821b26916d2c64ff89872f40d858257,20260323,md5,NaN,NaN,NaN,NaN,NaN,NaN,...,0.329103,0.329103,0.329103,0.329103,1.0,202603,华通-2026-03,1-train,2026-03-23,0.967190
2,6577810,11b515f674b7a9639e1efd47acaddd63,20260405,md5,0.936395,0.635825,0.280602,0.0,0.002123,0.269247,...,0.065177,0.065177,0.065177,0.065177,1.0,202604,华通-2026-04至05,3-oot,2026-04-05,0.966381
3,6577811,4a18e4983795edc5c2d5a5d55f94dfbe,20260408,md5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,202604,华通-2026-04至05,3-oot,2026-04-08,0.967026
4,6577812,6079d651e4d5f0b422a5de7ed38b7410,20260228,md5,NaN,NaN,NaN,NaN,NaN,0.321472,...,NaN,NaN,NaN,NaN,1.0,202602,华通-2026-02,1-train,2026-02-28,0.967160


In [44]:
df_bd_md5['mobile'].nunique()

128769

In [34]:
df_al_prob = pd.read_csv('/Users/mayongzhi/Job/Job_xyd/code/华通银行/ali_prob.csv')
df_al_prob.head()

,serie_no,ali_88_prob_keep_oot_4_5m,ali_88_prob_no_oot,ali_88_prob_no_oot_gap_auc_002
0,1,0.034181,0.030180,0.033990
1,2,0.033811,0.028276,0.027606
2,3,0.035362,0.030181,0.031936
3,4,0.033315,0.027863,0.026289
4,5,0.034104,0.029549,0.031649


In [ ]:
df_ymv4_prob = pd.read_csv('/Users/mayongzhi/Job/Job_xyd/code/华通银行/ym_v4_embd_prob.csv')
df_ymv4_prob.head()

,serie_no,id_card_md5,prob_embeddings_v4
0,18257,baf799226f4ecc5e6e0155c51cec94ce,0.261673
1,1863,2ba1bb299ebba228bd46f85f3c7e4835,0.282888
2,42351,c706ac557deafcdad6b2650c5f1b48a9,0.310997
3,14423,e5d7023d318bdb39b1556849266e4d2d,0.338233
4,3061,41849b7da94584575f67ffd3f36ed2b9,0.269742


In [38]:
df_ymv4_prob['id_card_md5'].nunique()

128769

In [39]:
df_ym_al = df_ymv4_prob.merge(df_al_prob,on='serie_no')
df_ym_al.head()

,serie_no,id_card_md5,prob_embeddings_v4,ali_88_prob_keep_oot_4_5m,ali_88_prob_no_oot,ali_88_prob_no_oot_gap_auc_002
0,18257,baf799226f4ecc5e6e0155c51cec94ce,0.261673,0.033285,0.024459,0.022148
1,1863,2ba1bb299ebba228bd46f85f3c7e4835,0.282888,0.033353,0.026158,0.024307
2,42351,c706ac557deafcdad6b2650c5f1b48a9,0.310997,0.035180,0.029374,0.029832
3,14423,e5d7023d318bdb39b1556849266e4d2d,0.338233,0.033633,0.029441,0.032937
4,3061,41849b7da94584575f67ffd3f36ed2b9,0.269742,0.032668,0.027009,0.024874


In [43]:
df_embeddings.columns

Index(['serie_no', 'id_card_md5', 'phonenum_md5(param)', 'app_date', 'code',
       'hxe1', 'hxe10', 'hxe11', 'hxe12', 'hxe13', 'hxe14', 'hxe15', 'hxe16',
       'hxe17', 'hxe18', 'hxe19', 'hxe2', 'hxe20', 'hxe21', 'hxe22', 'hxe23',
       'hxe24', 'hxe25', 'hxe26', 'hxe27', 'hxe28', 'hxe29', 'hxe3', 'hxe30',
       'hxe31', 'hxe32', 'hxe33', 'hxe34', 'hxe35', 'hxe36', 'hxe37', 'hxe38',
       'hxe39', 'hxe4', 'hxe40', 'hxe41', 'hxe42', 'hxe43', 'hxe44', 'hxe45',
       'hxe46', 'hxe47', 'hxe48', 'hxe49', 'hxe5', 'hxe50', 'hxe51', 'hxe52',
       'hxe53', 'hxe54', 'hxe55', 'hxe56', 'hxe57', 'hxe58', 'hxe59', 'hxe6',
       'hxe60', 'hxe61', 'hxe62', 'hxe63', 'hxe64', 'hxe65', 'hxe66', 'hxe67',
       'hxe68', 'hxe69', 'hxe7', 'hxe70', 'hxe71', 'hxe72', 'hxe73', 'hxe74',
       'hxe75', 'hxe76', 'hxe77', 'hxe78', 'hxe79', 'hxe8', 'hxe80', 'hxe9',
       'hxs1'],
      dtype='object')

In [46]:
df_bd_md5.columns

Index(['test_id', 'mobile', 'apply_date', 'enc_method', 'RL4006N0001',
       'RL4006N0002', 'RL4006N0003', 'RL4006N0004', 'RL4006N0005',
       'RL4002N0001',
       ...
       'RL4001N0191', 'RL4001N0192', 'RL4001N0193', 'RL4001N0194', 'flag',
       'month', 'partner_name', 'mid_sample_type', 'apply_date_new',
       'bd_lgb_NoOot_prob'],
      dtype='object', length=571)

In [47]:
df_bd_md5.merge(df_embeddings[['serie_no','phonenum_md5(param)']],left_on='mobile',right_on='phonenum_md5(param)').shape

(128769, 573)

In [48]:
df_bd_md5 = df_bd_md5.merge(df_embeddings[['serie_no','phonenum_md5(param)']],left_on='mobile',right_on='phonenum_md5(param)')
df_bd_md5.head()

,test_id,mobile,apply_date,enc_method,RL4006N0001,RL4006N0002,RL4006N0003,RL4006N0004,RL4006N0005,RL4002N0001,...,RL4001N0193,RL4001N0194,flag,month,partner_name,mid_sample_type,apply_date_new,bd_lgb_NoOot_prob,serie_no,phonenum_md5(param)
0,6577808,fa4fc24bb7eb64ad38befd4c8bb753ae,20260306,md5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,202603,华通-2026-03,1-train,2026-03-06,0.967041,1,fa4fc24bb7eb64ad38befd4c8bb753ae
1,6577809,b821b26916d2c64ff89872f40d858257,20260323,md5,NaN,NaN,NaN,NaN,NaN,NaN,...,0.329103,0.329103,1.0,202603,华通-2026-03,1-train,2026-03-23,0.967190,2,b821b26916d2c64ff89872f40d858257
2,6577810,11b515f674b7a9639e1efd47acaddd63,20260405,md5,0.936395,0.635825,0.280602,0.0,0.002123,0.269247,...,0.065177,0.065177,1.0,202604,华通-2026-04至05,3-oot,2026-04-05,0.966381,3,11b515f674b7a9639e1efd47acaddd63
3,6577811,4a18e4983795edc5c2d5a5d55f94dfbe,20260408,md5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,202604,华通-2026-04至05,3-oot,2026-04-08,0.967026,4,4a18e4983795edc5c2d5a5d55f94dfbe
4,6577812,6079d651e4d5f0b422a5de7ed38b7410,20260228,md5,NaN,NaN,NaN,NaN,NaN,0.321472,...,NaN,NaN,1.0,202602,华通-2026-02,1-train,2026-02-28,0.967160,5,6079d651e4d5f0b422a5de7ed38b7410


In [51]:
df_bd_md5.columns

Index(['test_id', 'mobile', 'apply_date', 'enc_method', 'RL4006N0001',
       'RL4006N0002', 'RL4006N0003', 'RL4006N0004', 'RL4006N0005',
       'RL4002N0001',
       ...
       'RL4001N0192', 'RL4001N0193', 'RL4001N0194', 'flag', 'month',
       'partner_name', 'mid_sample_type', 'apply_date_new',
       'bd_lgb_NoOot_prob', 'serie_no'],
      dtype='object', length=572)

In [49]:
df_bd_md5.drop(columns=['phonenum_md5(param)'],inplace=True)

In [53]:
df_ym_al = df_ym_al.merge(df_bd_md5[['serie_no','bd_lgb_NoOot_prob','test_id','flag', 'month',
       'partner_name', 'mid_sample_type', 'apply_date_new']],on='serie_no')

In [54]:
df_ym_al.columns

Index(['serie_no', 'id_card_md5', 'prob_embeddings_v4',
       'ali_88_prob_keep_oot_4_5m', 'ali_88_prob_no_oot',
       'ali_88_prob_no_oot_gap_auc_002', 'bd_lgb_NoOot_prob', 'test_id',
       'flag', 'month', 'partner_name', 'mid_sample_type', 'apply_date_new'],
      dtype='object')

In [55]:
df_lr = df_ym_al

In [58]:
def get_valid_prob(x):

    if x <= 0 or x >= 1:
        y = 0
    else:
        y = 1
    return y

sub_probs = ['bd_lgb_NoOot_prob','ali_88_prob_no_oot_gap_auc_002']
feas = []
for fea in sub_probs:
    df_lr[f'{fea}_fill'] = df_lr[fea].fillna(0)
    df_lr[f'{fea}_valid'] = df_lr[f'{fea}_fill'].apply(get_valid_prob)
    df_lr[f'{fea}_trans'] = df_lr[f'{fea}_fill'].apply(LogitTransformation)
    feas.append(f'{fea}_trans')
    feas.append(f'{fea}_valid')

tuner = AntiOverfittingTuner(LogisticRegression,'lr')
params_grid_lr = tuner.optimize_anti_overfitting(
    df_lr[feas],
    df_lr['flag'],
    n_trials=30
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [59]:
pd.DataFrame(params_grid_lr['performance_history']).to_csv('params_log_lr_bd_al.csv',index=False)

In [60]:
params_grid_lr = pd.read_csv('params_log_lr_bd_al.csv')
params_grid_lr = params_grid_lr[params_grid_lr['is_overfitting']==False]
params_grid_lr

,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
0,0,"{'C': 15.741890047456641, 'penalty': 'l1', 'so...",0.622548,0.622489,0.622489,0.622548,0.622489,0.176689,0.185143,0.000059,0.008453,False,0.0,0.003019,0.012210,0.003019,0.012210
1,3,"{'C': 0.0015777663630582469, 'penalty': None, ...",0.622476,0.622431,0.622431,0.622476,0.622431,0.176818,0.185681,0.000045,0.008863,False,0.0,0.003004,0.012178,0.003004,0.012178
2,5,"{'C': 0.0005397956855996448, 'penalty': 'elast...",0.602766,0.602780,0.602780,0.602766,0.602780,0.145518,0.155941,0.000014,0.010423,False,0.0,0.002509,0.010058,0.002509,0.010058
3,10,"{'C': 65.6481761175346, 'penalty': 'l1', 'solv...",0.622859,0.622805,0.622805,0.622859,0.622805,0.177229,0.184633,0.000053,0.007404,False,0.0,0.003104,0.012468,0.003104,0.012468
4,11,"{'C': 83.67826428540977, 'penalty': 'l1', 'sol...",0.622865,0.622812,0.622812,0.622865,0.622812,0.177265,0.184316,0.000053,0.007051,False,0.0,0.003110,0.012485,0.003110,0.012485
5,12,"{'C': 94.42207016006594, 'penalty': 'l1', 'sol...",0.622865,0.622812,0.622812,0.622865,0.622812,0.177263,0.184332,0.000053,0.007068,False,0.0,0.003110,0.012485,0.003110,0.012485
6,13,"{'C': 1.7206300727024646, 'penalty': 'l1', 'so...",0.621426,0.621343,0.621343,0.621426,0.621343,0.175151,0.187038,0.000083,0.011887,False,0.0,0.002952,0.011756,0.002952,0.011756
7,15,"{'C': 0.20519059010204016, 'penalty': 'l2', 's...",0.604782,0.604797,0.604797,0.604782,0.604797,0.149880,0.157866,0.000016,0.007986,False,0.0,0.002509,0.010159,0.002509,0.010159
8,16,"{'C': 55.76242278032358, 'penalty': 'l1', 'sol...",0.622807,0.622756,0.622756,0.622807,0.622756,0.177194,0.186053,0.000051,0.008859,False,0.0,0.003078,0.012379,0.003078,0.012379
9,17,"{'C': 1.947856459923636, 'penalty': 'l1', 'sol...",0.621490,0.621425,0.621425,0.621490,0.621425,0.175110,0.186764,0.000065,0.011654,False,0.0,0.002946,0.011787,0.002946,0.011787


In [62]:

import ast
candidate_params = params_grid_lr['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]
candidate_params

[{'C': 15.741890047456641,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 641,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0015777663630582469,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 563,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0005397956855996448,
  'penalty': 'elasticnet',
  'solver': 'saga',
  'max_iter': 546,
  'random_state': 42,
  'l1_ratio': 0.034388521115218396,
  'n_jobs': -1},
 {'C': 65.6481761175346,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 939,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 83.67826428540977,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 975,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 94.42207016006594,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 971,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 1.7206300727024646,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 970,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.20519059010204016,
  'penalty': 'l2',
  'solver': 'saga',
  'max_iter': 187,


In [63]:
candidate_params[-5]

{'C': 0.2661426674667985,
 'penalty': None,
 'solver': 'saga',
 'max_iter': 992,
 'random_state': 42,
 'n_jobs': -1}

In [73]:
model_lr_AlYm = LogisticRegression(**candidate_params[-5])
model_lr_AlYm.fit(
    df_lr[feas],
    df_lr['flag'],
)
df_lr['lr_BdAl_prob'] = model_lr_AlYm.predict_proba(df_lr[feas])[:,1]

In [65]:
df_lr.head()

,serie_no,id_card_md5,prob_embeddings_v4,ali_88_prob_keep_oot_4_5m,ali_88_prob_no_oot,ali_88_prob_no_oot_gap_auc_002,bd_lgb_NoOot_prob,test_id,flag,month,partner_name,mid_sample_type,apply_date_new,bd_lgb_NoOot_prob_fill,bd_lgb_NoOot_prob_valid,bd_lgb_NoOot_prob_trans,ali_88_prob_no_oot_gap_auc_002_fill,ali_88_prob_no_oot_gap_auc_002_valid,ali_88_prob_no_oot_gap_auc_002_trans,lr_AlYm_prob
0,18257,baf799226f4ecc5e6e0155c51cec94ce,0.261673,0.033285,0.024459,0.022148,0.967202,6596064,0.0,202603,华通-2026-03,1-train,2026-03-31,0.967202,1,3.384038,0.022148,1,-3.787588,0.023921
1,1863,2ba1bb299ebba228bd46f85f3c7e4835,0.282888,0.033353,0.026158,0.024307,0.966569,6579670,0.0,202604,华通-2026-04至05,3-oot,2026-04-04,0.966569,1,3.364262,0.024307,1,-3.692369,0.015992
2,42351,c706ac557deafcdad6b2650c5f1b48a9,0.310997,0.035180,0.029374,0.029832,0.967229,6620158,0.0,202603,华通-2026-03,1-train,2026-03-10,0.967229,1,3.384902,0.029832,1,-3.481894,0.042635
3,14423,e5d7023d318bdb39b1556849266e4d2d,0.338233,0.033633,0.029441,0.032937,0.967193,6592230,0.0,202604,华通-2026-04至05,3-oot,2026-04-02,0.967193,1,3.383744,0.032937,1,-3.379678,0.049506
4,3061,41849b7da94584575f67ffd3f36ed2b9,0.269742,0.032668,0.027009,0.024874,0.966862,6580868,0.0,202603,华通-2026-03,1-train,2026-03-26,0.966862,1,3.373371,0.024874,1,-3.668763,0.021791


In [99]:
mdrp.generate_full_report??

Signature:
mdrp.generate_full_report(
    features: List[str] = None,
    feature_names: Dict[str, str] = None,
    save_path: str = None,
    bin_method: str = 'quantile',
    model: Any = None,
    importance_type: str = 'gain',
    experiment_meta: Union[Dict[str, Any], NoneType] = None,
) -> Dict[str, pandas.core.frame.DataFrame]
Source:   
    def generate_full_report(self, features: List[str] = None,
                           feature_names: Dict[str, str] = None,
                           save_path: str = None,
                           bin_method: str = 'quantile',
                           model: Any = None,
                           importance_type: str = 'gain',
                           experiment_meta: Optional[Dict[str, Any]] = None) -> Dict[str, pd.DataFrame]:
        """
        生成完整的模型交付报告

        Args:
            features: 特征列名列表
            feature_names: 特征中文名映射
            save_path: 保存路径（可选）
            bin_method: 分箱方式，'quantile'(等频分箱)或'equal'(等距分箱)
      

In [74]:
mdrp  = ModelDeliveryReport(
    df_lr,
    'flag',
    'lr_BdAl_prob',
    'apply_date_new',
    'partner_name',
)
mdrp.generate_full_report(save_path='report_lr/lr_BdAl_report.xlsx')

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_lr/lr_BdAl_report.xlsx


{'样本情况':      Month    Good   Bad   Total Bad_Rate
 0  2026-02   10755   496   11251   0.0441
 1  2026-03   59876  1966   61842   0.0318
 2  2026-04   54317  1197   55514   0.0216
 3  2026-05     161     1     162   0.0062
 4       合计  125109  3660  128769   0.0284,
 '模型效果':             样本类型    Good   Bad   Total Bad_Rate      KS     AUC TOP10_lift  \
 0            all  125109  3660  128769   0.0284  0.1763  0.6229     2.0218   
 1     华通-2026-03   59876  1966   61842   0.0318  0.1801  0.6191     1.9224   
 2  华通-2026-04至05   54478  1198   55676   0.0215  0.1763  0.6249     2.1034   
 3     华通-2026-02   10755   496   11251   0.0441  0.1660  0.6182     2.0163   
 
       PSI  
 0  0.0000  
 1  0.0000  
 2  0.0000  
 3  0.0000  ,
 '模型效果分箱':              样本类型                分箱     好用户   坏用户     总用户   总用户占比   坏用户占比  \
 0             all   (0.0455, 0.322]   12137   740   12877  0.1000  0.2022   
 1             all  (0.0376, 0.0455]   12385   492   12877  0.1000  0.1344   
 2             all

In [96]:
mdrp  = ModelDeliveryReport(
    df_lr,
    'flag',
    'lr_BdAl_prob',
    'apply_date_new',
    'partner_name',
)
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/BdAl/')['lift_details'][0]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,3093,34.0,3059.0,0.010993,0.345779,0.006-0.013,3093,34.0,0.010993,0.345779,0.017294,华通-2026-03
1,2,3092,57.0,3035.0,0.018435,0.579876,0.013-0.015,6185,91.0,0.014713,0.462809,0.046287,华通-2026-03
2,3,3092,52.0,3040.0,0.016818,0.529010,0.015-0.017,9277,143.0,0.015414,0.484874,0.072737,华通-2026-03
3,4,3092,60.0,3032.0,0.019405,0.610396,0.017-0.018,12369,203.0,0.016412,0.516252,0.103255,华通-2026-03
4,5,3092,78.0,3014.0,0.025226,0.793515,0.018-0.019,15461,281.0,0.018175,0.571701,0.142930,华通-2026-03
5,6,3092,68.0,3024.0,0.021992,0.691782,0.019-0.021,18553,349.0,0.018811,0.591713,0.177518,华通-2026-03
6,7,3092,83.0,3009.0,0.026843,0.844381,0.021-0.022,21645,432.0,0.019958,0.627807,0.219736,华通-2026-03
7,8,3092,75.0,3017.0,0.024256,0.762995,0.022-0.023,24737,507.0,0.020496,0.644705,0.257884,华通-2026-03
8,9,3092,80.0,3012.0,0.025873,0.813862,0.023-0.025,27829,587.0,0.021093,0.663499,0.298576,华通-2026-03
9,10,3092,86.0,3006.0,0.027814,0.874901,0.025-0.026,30921,673.0,0.021765,0.684639,0.342319,华通-2026-03


In [97]:
mdrp  = ModelDeliveryReport(
    df_lr,
    'flag',
    'lr_BdAl_prob',
    'apply_date_new',
    'partner_name',
)
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/BdAl/')['lift_details'][1]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,2784,25.0,2759.0,0.008980,0.417332,0.007-0.013,2784,25.0,0.008980,0.417332,0.020868,华通-2026-04至05
1,2,2784,27.0,2757.0,0.009698,0.450719,0.013-0.014,5568,52.0,0.009339,0.434026,0.043406,华通-2026-04至05
2,3,2784,40.0,2744.0,0.014368,0.667732,0.014-0.016,8352,92.0,0.011015,0.511928,0.076795,华通-2026-04至05
3,4,2784,36.0,2748.0,0.012931,0.600958,0.016-0.017,11136,128.0,0.011494,0.534185,0.106845,华通-2026-04至05
4,5,2783,18.0,2765.0,0.006468,0.300587,0.017-0.019,13919,146.0,0.010489,0.487479,0.121870,华通-2026-04至05
5,6,2784,48.0,2736.0,0.017241,0.801278,0.019-0.020,16703,194.0,0.011615,0.539782,0.161937,华通-2026-04至05
6,7,2784,37.0,2747.0,0.013290,0.617652,0.020-0.021,19487,231.0,0.011854,0.550907,0.192821,华通-2026-04至05
7,8,2784,50.0,2734.0,0.017960,0.834665,0.021-0.022,22271,281.0,0.012617,0.586378,0.234558,华通-2026-04至05
8,9,2783,66.0,2717.0,0.023715,1.102153,0.022-0.024,25054,347.0,0.013850,0.643671,0.289649,华通-2026-04至05
9,10,2784,49.0,2735.0,0.017601,0.817971,0.024-0.025,27838,396.0,0.014225,0.661102,0.330551,华通-2026-04至05


In [98]:
mdrp  = ModelDeliveryReport(
    df_lr,
    'flag',
    'lr_BdAl_prob',
    'apply_date_new',
    'partner_name',
)
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/BdAl/')['lift_details'][2]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,563,7.0,556.0,0.012433,0.282032,0.006-0.013,563,7.0,0.012433,0.282032,0.014113,华通-2026-02
1,2,563,11.0,552.0,0.019538,0.443194,0.013-0.015,1126,18.0,0.015986,0.362613,0.036290,华通-2026-02
2,3,562,18.0,544.0,0.032028,0.726517,0.015-0.017,1688,36.0,0.021327,0.483771,0.072581,华通-2026-02
3,4,563,14.0,549.0,0.024867,0.564065,0.017-0.018,2251,50.0,0.022212,0.503853,0.100806,华通-2026-02
4,5,562,20.0,542.0,0.035587,0.807241,0.018-0.019,2813,70.0,0.024884,0.564466,0.141129,华通-2026-02
5,6,563,16.0,547.0,0.028419,0.644646,0.019-0.021,3376,86.0,0.025474,0.577837,0.173387,华通-2026-02
6,7,562,18.0,544.0,0.032028,0.726517,0.021-0.022,3938,104.0,0.026409,0.599056,0.209677,华通-2026-02
7,8,563,17.0,546.0,0.030195,0.684936,0.022-0.023,4501,121.0,0.026883,0.609798,0.243952,华通-2026-02
8,9,562,28.0,534.0,0.049822,1.130137,0.023-0.025,5063,149.0,0.029429,0.667556,0.300403,华通-2026-02
9,10,563,29.0,534.0,0.051510,1.168420,0.025-0.026,5626,178.0,0.031639,0.717678,0.358871,华通-2026-02


In [100]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/BdAl/')['lift_details']

[    分箱   样本数   正样本数    负样本数      正样本率      Lift         概率范围  累积样本数  累积正样本数  \
 0    1  3093   34.0  3059.0  0.010993  0.345779  0.006-0.013   3093    34.0   
 1    2  3092   57.0  3035.0  0.018435  0.579876  0.013-0.015   6185    91.0   
 2    3  3092   52.0  3040.0  0.016818  0.529010  0.015-0.017   9277   143.0   
 3    4  3092   60.0  3032.0  0.019405  0.610396  0.017-0.018  12369   203.0   
 4    5  3092   78.0  3014.0  0.025226  0.793515  0.018-0.019  15461   281.0   
 5    6  3092   68.0  3024.0  0.021992  0.691782  0.019-0.021  18553   349.0   
 6    7  3092   83.0  3009.0  0.026843  0.844381  0.021-0.022  21645   432.0   
 7    8  3092   75.0  3017.0  0.024256  0.762995  0.022-0.023  24737   507.0   
 8    9  3092   80.0  3012.0  0.025873  0.813862  0.023-0.025  27829   587.0   
 9   10  3092   86.0  3006.0  0.027814  0.874901  0.025-0.026  30921   673.0   
 10  11  3092   94.0  2998.0  0.030401  0.956287  0.026-0.028  34013   767.0   
 11  12  3092   75.0  3017.0  0.024256  

### bd  ym

In [80]:
df_lr.columns

Index(['serie_no', 'id_card_md5', 'prob_embeddings_v4',
       'ali_88_prob_keep_oot_4_5m', 'ali_88_prob_no_oot',
       'ali_88_prob_no_oot_gap_auc_002', 'bd_lgb_NoOot_prob', 'test_id',
       'flag', 'month', 'partner_name', 'mid_sample_type', 'apply_date_new',
       'bd_lgb_NoOot_prob_fill', 'bd_lgb_NoOot_prob_valid',
       'bd_lgb_NoOot_prob_trans', 'ali_88_prob_no_oot_gap_auc_002_fill',
       'ali_88_prob_no_oot_gap_auc_002_valid',
       'ali_88_prob_no_oot_gap_auc_002_trans', 'lr_BdAl_prob'],
      dtype='object')

In [81]:
def get_valid_prob(x):

    if x <= 0 or x >= 1:
        y = 0
    else:
        y = 1
    return y

sub_probs = ['bd_lgb_NoOot_prob','prob_embeddings_v4']
feas = []
for fea in sub_probs:
    df_lr[f'{fea}_fill'] = df_lr[fea].fillna(0)
    df_lr[f'{fea}_valid'] = df_lr[f'{fea}_fill'].apply(get_valid_prob)
    df_lr[f'{fea}_trans'] = df_lr[f'{fea}_fill'].apply(LogitTransformation)
    feas.append(f'{fea}_trans')
    feas.append(f'{fea}_valid')

tuner = AntiOverfittingTuner(LogisticRegression,'lr')
params_grid_lr = tuner.optimize_anti_overfitting(
    df_lr[feas],
    df_lr['flag'],
    n_trials=30
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [82]:
pd.DataFrame(params_grid_lr['performance_history']).to_csv('params_log_lr_bd_ym.csv',index=False)

In [83]:
params_grid_lr = pd.read_csv('params_log_lr_bd_ym.csv')
params_grid_lr = params_grid_lr[params_grid_lr['is_overfitting']==False]
params_grid_lr

,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
0,0,"{'C': 15.741890047456641, 'penalty': 'l1', 'so...",0.606308,0.606060,0.606060,0.606308,0.606060,0.147426,0.154849,0.000248,0.007423,False,0.0,0.002974,0.011864,0.002974,0.011864
1,3,"{'C': 0.0015777663630582469, 'penalty': None, ...",0.606313,0.606063,0.606063,0.606313,0.606063,0.147637,0.155099,0.000250,0.007462,False,0.0,0.002974,0.011873,0.002974,0.011873
2,5,"{'C': 0.0005397956855996448, 'penalty': 'elast...",0.577906,0.577875,0.577875,0.577906,0.577875,0.114686,0.121386,0.000032,0.006700,False,0.0,0.002394,0.009586,0.002394,0.009586
3,10,"{'C': 0.000769639183009799, 'penalty': None, '...",0.600319,0.600124,0.600124,0.600319,0.600124,0.146406,0.152684,0.000194,0.006277,False,0.0,0.002697,0.010906,0.002697,0.010906
4,11,"{'C': 68.20019829930264, 'penalty': 'l1', 'sol...",0.606319,0.606059,0.606059,0.606319,0.606059,0.147519,0.156255,0.000260,0.008736,False,0.0,0.002973,0.011898,0.002973,0.011898
5,12,"{'C': 4.238271500303412, 'penalty': 'l1', 'sol...",0.606112,0.605847,0.605847,0.606112,0.605847,0.147181,0.155298,0.000265,0.008117,False,0.0,0.002979,0.011757,0.002979,0.011757
6,13,"{'C': 0.0778713991593174, 'penalty': None, 'so...",0.606306,0.606061,0.606061,0.606306,0.606061,0.147415,0.156265,0.000245,0.008850,False,0.0,0.002974,0.011921,0.002974,0.011921
7,15,"{'C': 0.04148386642515991, 'penalty': None, 's...",0.606306,0.606061,0.606061,0.606306,0.606061,0.147415,0.156265,0.000245,0.008850,False,0.0,0.002974,0.011921,0.002974,0.011921
8,16,"{'C': 0.030649534840788962, 'penalty': None, '...",0.606306,0.606061,0.606061,0.606306,0.606061,0.147415,0.156265,0.000245,0.008850,False,0.0,0.002974,0.011921,0.002974,0.011921
9,17,"{'C': 0.005946201558815915, 'penalty': 'l2', '...",0.578454,0.578429,0.578429,0.578454,0.578429,0.115665,0.122737,0.000025,0.007072,False,0.0,0.002369,0.009522,0.002369,0.009522


In [84]:

import ast
candidate_params = params_grid_lr['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]
candidate_params

[{'C': 15.741890047456641,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 641,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0015777663630582469,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 563,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0005397956855996448,
  'penalty': 'elasticnet',
  'solver': 'saga',
  'max_iter': 546,
  'random_state': 42,
  'l1_ratio': 0.034388521115218396,
  'n_jobs': -1},
 {'C': 0.000769639183009799,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 116,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 68.20019829930264,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 674,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 4.238271500303412,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 570,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0778713991593174,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 942,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.04148386642515991,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 9

In [85]:
candidate_params[1]

{'C': 0.0015777663630582469,
 'penalty': None,
 'solver': 'saga',
 'max_iter': 563,
 'random_state': 42,
 'n_jobs': -1}

In [86]:
model_lr_AlYm = LogisticRegression(**candidate_params[1])
model_lr_AlYm.fit(
    df_lr[feas],
    df_lr['flag'],
)
df_lr['lr_BdYm_prob'] = model_lr_AlYm.predict_proba(df_lr[feas])[:,1]

In [87]:
df_lr.head()

,serie_no,id_card_md5,prob_embeddings_v4,ali_88_prob_keep_oot_4_5m,ali_88_prob_no_oot,ali_88_prob_no_oot_gap_auc_002,bd_lgb_NoOot_prob,test_id,flag,month,...,bd_lgb_NoOot_prob_valid,bd_lgb_NoOot_prob_trans,ali_88_prob_no_oot_gap_auc_002_fill,ali_88_prob_no_oot_gap_auc_002_valid,ali_88_prob_no_oot_gap_auc_002_trans,lr_BdAl_prob,prob_embeddings_v4_fill,prob_embeddings_v4_valid,prob_embeddings_v4_trans,lr_BdYm_prob
0,18257,baf799226f4ecc5e6e0155c51cec94ce,0.261673,0.033285,0.024459,0.022148,0.967202,6596064,0.0,202603,...,1,3.384038,0.022148,1,-3.787588,0.023921,0.261673,1,-1.037294,0.030985
1,1863,2ba1bb299ebba228bd46f85f3c7e4835,0.282888,0.033353,0.026158,0.024307,0.966569,6579670,0.0,202604,...,1,3.364262,0.024307,1,-3.692369,0.015992,0.282888,1,-0.930182,0.019281
2,42351,c706ac557deafcdad6b2650c5f1b48a9,0.310997,0.035180,0.029374,0.029832,0.967229,6620158,0.0,202603,...,1,3.384902,0.029832,1,-3.481894,0.042635,0.310997,1,-0.795461,0.046057
3,14423,e5d7023d318bdb39b1556849266e4d2d,0.338233,0.033633,0.029441,0.032937,0.967193,6592230,0.0,202604,...,1,3.383744,0.032937,1,-3.379678,0.049506,0.338233,1,-0.671176,0.053547
4,3061,41849b7da94584575f67ffd3f36ed2b9,0.269742,0.032668,0.027009,0.024874,0.966862,6580868,0.0,202603,...,1,3.373371,0.024874,1,-3.668763,0.021791,0.269742,1,-0.995933,0.023405


In [88]:
mdrp  = ModelDeliveryReport(
    df_lr,
    'flag',
    'lr_BdYm_prob',
    'apply_date_new',
    'partner_name',
)
mdrp.generate_full_report(save_path='report_lr/lr_BdYm_report.xlsx')

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_lr/lr_BdYm_report.xlsx


{'样本情况':      Month    Good   Bad   Total Bad_Rate
 0  2026-02   10755   496   11251   0.0441
 1  2026-03   59876  1966   61842   0.0318
 2  2026-04   54317  1197   55514   0.0216
 3  2026-05     161     1     162   0.0062
 4       合计  125109  3660  128769   0.0284,
 '模型效果':             样本类型    Good   Bad   Total Bad_Rate      KS     AUC TOP10_lift  \
 0            all  125109  3660  128769   0.0284  0.1472  0.6063     1.8497   
 1     华通-2026-03   59876  1966   61842   0.0318  0.1464  0.6015     1.7648   
 2  华通-2026-04至05   54478  1198   55676   0.0215  0.1528  0.6115     1.9197   
 3     华通-2026-02   10755   496   11251   0.0441  0.1363  0.5900     1.7139   
 
       PSI  
 0  0.0000  
 1  0.0000  
 2  0.0000  
 3  0.0000  ,
 '模型效果分箱':              样本类型                               分箱     好用户   坏用户     总用户  \
 0             all                   (0.043, 0.317]   12200   677   12877   
 1             all                  (0.0366, 0.043]   12380   497   12877   
 2             all   

In [93]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/BdYm/')['lift_details'][0]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,3093,35.0,3058.0,0.011316,0.355949,0.005-0.014,3093,35.0,0.011316,0.355949,0.017803,华通-2026-03
1,2,3092,60.0,3032.0,0.019405,0.610396,0.014-0.017,6185,95.0,0.015360,0.483152,0.048321,华通-2026-03
2,3,3092,65.0,3027.0,0.021022,0.661262,0.017-0.018,9277,160.0,0.017247,0.542516,0.081384,华通-2026-03
3,4,3092,57.0,3035.0,0.018435,0.579876,0.018-0.019,12369,217.0,0.017544,0.551855,0.110376,华通-2026-03
4,5,3092,77.0,3015.0,0.024903,0.783342,0.019-0.021,15461,294.0,0.019016,0.598150,0.149542,华通-2026-03
5,6,3092,70.0,3022.0,0.022639,0.712129,0.021-0.022,18553,364.0,0.019619,0.617145,0.185148,华通-2026-03
6,7,3092,82.0,3010.0,0.026520,0.834208,0.022-0.023,21645,446.0,0.020605,0.648153,0.226857,华通-2026-03
7,8,3092,99.0,2993.0,0.032018,1.007154,0.023-0.024,24737,545.0,0.022032,0.693026,0.277213,华通-2026-03
8,9,3092,98.0,2994.0,0.031695,0.996980,0.024-0.025,27829,643.0,0.023105,0.726797,0.327060,华通-2026-03
9,10,3092,79.0,3013.0,0.025550,0.803688,0.025-0.027,30921,722.0,0.023350,0.734486,0.367243,华通-2026-03


In [94]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/BdYm/')['lift_details'][1]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,2784,14.0,2770.0,0.005029,0.233706,0.005-0.014,2784,14.0,0.005029,0.233706,0.011686,华通-2026-04至05
1,2,2784,34.0,2750.0,0.012213,0.567572,0.014-0.016,5568,48.0,0.008621,0.400639,0.040067,华通-2026-04至05
2,3,2784,30.0,2754.0,0.010776,0.500799,0.016-0.017,8352,78.0,0.009339,0.434026,0.065109,华通-2026-04至05
3,4,2784,40.0,2744.0,0.014368,0.667732,0.017-0.019,11136,118.0,0.010596,0.492452,0.098497,华通-2026-04至05
4,5,2783,37.0,2746.0,0.013295,0.617874,0.019-0.020,13919,155.0,0.011136,0.517529,0.129382,华通-2026-04至05
5,6,2784,45.0,2739.0,0.016164,0.751198,0.020-0.021,16703,200.0,0.011974,0.556476,0.166945,华通-2026-04至05
6,7,2784,61.0,2723.0,0.021911,1.018291,0.021-0.022,19487,261.0,0.013394,0.622453,0.217863,华通-2026-04至05
7,8,2784,46.0,2738.0,0.016523,0.767891,0.022-0.023,22271,307.0,0.013785,0.640634,0.256260,华通-2026-04至05
8,9,2783,56.0,2727.0,0.020122,0.935160,0.023-0.024,25054,363.0,0.014489,0.673350,0.303005,华通-2026-04至05
9,10,2784,67.0,2717.0,0.024066,1.118451,0.024-0.026,27838,430.0,0.015447,0.717863,0.358932,华通-2026-04至05


In [95]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/BdYm/')['lift_details'][2]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,563,11.0,552.0,0.019538,0.443194,0.007-0.015,563,11.0,0.019538,0.443194,0.022177,华通-2026-02
1,2,563,20.0,543.0,0.035524,0.805807,0.015-0.017,1126,31.0,0.027531,0.624500,0.062500,华通-2026-02
2,3,562,16.0,546.0,0.028470,0.645793,0.017-0.019,1688,47.0,0.027844,0.631589,0.094758,华通-2026-02
3,4,563,22.0,541.0,0.039076,0.886388,0.019-0.020,2251,69.0,0.030653,0.695317,0.139113,华通-2026-02
4,5,562,16.0,546.0,0.028470,0.645793,0.020-0.021,2813,85.0,0.030217,0.685423,0.171371,华通-2026-02
5,6,563,18.0,545.0,0.031972,0.725226,0.021-0.022,3376,103.0,0.030509,0.692061,0.207661,华通-2026-02
6,7,562,22.0,540.0,0.039146,0.887965,0.022-0.023,3938,125.0,0.031742,0.720019,0.252016,华通-2026-02
7,8,563,20.0,543.0,0.035524,0.805807,0.023-0.025,4501,145.0,0.032215,0.730749,0.292339,华通-2026-02
8,9,562,15.0,547.0,0.026690,0.605431,0.025-0.026,5063,160.0,0.031602,0.716839,0.322581,华通-2026-02
9,10,628,29.0,599.0,0.046178,1.047485,0.026-0.027,5691,189.0,0.033210,0.753325,0.381048,华通-2026-02


# tx al

In [101]:
df_tx = pd.read_csv('/Users/mayongzhi/Job/Job_xyd/数据/客户测建模/华通/小雨点数科-反欺诈v9g-20260713-12w.csv')
df_tx.head()

,serie_no,id_card_md5,phonenum_md5,app_date,other_infos,serie_no_orig,model_risk_v9_gtongyong_score,model_risk_v9_gxiaojin_score
0,1,1e0968a3c477e2a4e5a5d68fb3712d74,fa4fc24bb7eb64ad38befd4c8bb753ae,20260306,1,1,74.0,74.0
1,2,3de08d3162b2aed10933cc74165808b7,b821b26916d2c64ff89872f40d858257,20260323,1,2,46.0,42.0
2,3,89d5fd9ad9ea658b847d6f449908a674,11b515f674b7a9639e1efd47acaddd63,20260405,1,3,54.0,52.0
3,4,4e861b04b6721d7c9e85421d2aa86018,4a18e4983795edc5c2d5a5d55f94dfbe,20260408,1,4,58.0,53.0
4,5,a343b5d96e944690c34ada406b395a4f,6079d651e4d5f0b422a5de7ed38b7410,20260228,1,5,60.0,55.0


In [ ]:
df_lr.merge(df_tx[['serie_no','model_risk_v9_gtongyong_score',	'model_risk_v9_gxiaojin_score']],on='serie_no').shape

(128769, 26)

In [103]:
df_lr = df_lr.merge(df_tx[['serie_no','model_risk_v9_gtongyong_score',	'model_risk_v9_gxiaojin_score']],on='serie_no')

In [104]:
df_lr['model_risk_v9_gtongyong_score'] = df_lr['model_risk_v9_gtongyong_score'] / 100
df_lr['model_risk_v9_gxiaojin_score'] = df_lr['model_risk_v9_gxiaojin_score'] / 100
df_lr.head()

,serie_no,id_card_md5,prob_embeddings_v4,ali_88_prob_keep_oot_4_5m,ali_88_prob_no_oot,ali_88_prob_no_oot_gap_auc_002,bd_lgb_NoOot_prob,test_id,flag,month,...,ali_88_prob_no_oot_gap_auc_002_fill,ali_88_prob_no_oot_gap_auc_002_valid,ali_88_prob_no_oot_gap_auc_002_trans,lr_BdAl_prob,prob_embeddings_v4_fill,prob_embeddings_v4_valid,prob_embeddings_v4_trans,lr_BdYm_prob,model_risk_v9_gtongyong_score,model_risk_v9_gxiaojin_score
0,18257,baf799226f4ecc5e6e0155c51cec94ce,0.261673,0.033285,0.024459,0.022148,0.967202,6596064,0.0,202603,...,0.022148,1,-3.787588,0.023921,0.261673,1,-1.037294,0.030985,0.50,0.46
1,1863,2ba1bb299ebba228bd46f85f3c7e4835,0.282888,0.033353,0.026158,0.024307,0.966569,6579670,0.0,202604,...,0.024307,1,-3.692369,0.015992,0.282888,1,-0.930182,0.019281,0.57,0.55
2,42351,c706ac557deafcdad6b2650c5f1b48a9,0.310997,0.035180,0.029374,0.029832,0.967229,6620158,0.0,202603,...,0.029832,1,-3.481894,0.042635,0.310997,1,-0.795461,0.046057,0.54,0.55
3,14423,e5d7023d318bdb39b1556849266e4d2d,0.338233,0.033633,0.029441,0.032937,0.967193,6592230,0.0,202604,...,0.032937,1,-3.379678,0.049506,0.338233,1,-0.671176,0.053547,0.56,0.54
4,3061,41849b7da94584575f67ffd3f36ed2b9,0.269742,0.032668,0.027009,0.024874,0.966862,6580868,0.0,202603,...,0.024874,1,-3.668763,0.021791,0.269742,1,-0.995933,0.023405,0.72,0.69


In [105]:
sub_probs = ['model_risk_v9_gtongyong_score','ali_88_prob_no_oot_gap_auc_002']
feas = []
for fea in sub_probs:
    df_lr[f'{fea}_fill'] = df_lr[fea].fillna(0)
    df_lr[f'{fea}_valid'] = df_lr[f'{fea}_fill'].apply(get_valid_prob)
    df_lr[f'{fea}_trans'] = df_lr[f'{fea}_fill'].apply(LogitTransformation)
    feas.append(f'{fea}_trans')
    feas.append(f'{fea}_valid')

tuner = AntiOverfittingTuner(LogisticRegression,'lr')
params_grid_lr = tuner.optimize_anti_overfitting(
    df_lr[feas],
    df_lr['flag'],
    n_trials=30
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [106]:
pd.DataFrame(params_grid_lr['performance_history']).to_csv('params_log_lr_al_tx.csv',index=False)

In [107]:
params_grid_lr = pd.read_csv('params_log_lr_al_tx.csv')
params_grid_lr = params_grid_lr[params_grid_lr['is_overfitting']==False]
params_grid_lr

,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
0,0,"{'C': 15.741890047456641, 'penalty': 'l1', 'so...",0.612882,0.612758,0.612758,0.612882,0.612758,0.161523,0.168814,0.000124,0.007291,False,0.0,0.002299,0.009282,0.002299,0.009282
1,3,"{'C': 0.0015777663630582469, 'penalty': None, ...",0.612881,0.612758,0.612758,0.612881,0.612758,0.161525,0.168862,0.000124,0.007337,False,0.0,0.002299,0.009283,0.002299,0.009283
2,5,"{'C': 0.0005397956855996448, 'penalty': 'elast...",0.585120,0.585013,0.585013,0.585120,0.585013,0.128222,0.136536,0.000107,0.008314,False,0.0,0.002329,0.009307,0.002329,0.009307
3,10,"{'C': 65.6481761175346, 'penalty': 'l1', 'solv...",0.612882,0.612758,0.612758,0.612882,0.612758,0.161525,0.168838,0.000123,0.007313,False,0.0,0.002299,0.009283,0.002299,0.009283
4,11,"{'C': 83.67826428540977, 'penalty': 'l1', 'sol...",0.612882,0.612758,0.612758,0.612882,0.612758,0.161517,0.168838,0.000123,0.007321,False,0.0,0.002299,0.009283,0.002299,0.009283
5,12,"{'C': 94.42207016006594, 'penalty': 'l1', 'sol...",0.612882,0.612758,0.612758,0.612882,0.612758,0.161515,0.168838,0.000124,0.007323,False,0.0,0.002299,0.009283,0.002299,0.009283
6,13,"{'C': 2.1352376986091604, 'penalty': 'l1', 'so...",0.612859,0.612736,0.612736,0.612859,0.612736,0.161505,0.168766,0.000123,0.007261,False,0.0,0.002297,0.009277,0.002297,0.009277
7,15,"{'C': 0.8388063784216602, 'penalty': 'l2', 'so...",0.612883,0.612762,0.612762,0.612883,0.612762,0.161446,0.168827,0.000121,0.007381,False,0.0,0.002299,0.009268,0.002299,0.009268
8,16,"{'C': 0.4734498565795354, 'penalty': 'l2', 'so...",0.612887,0.612762,0.612762,0.612887,0.612762,0.161586,0.169040,0.000124,0.007453,False,0.0,0.002298,0.009256,0.002298,0.009256
9,17,"{'C': 0.09763576663581003, 'penalty': 'l2', 's...",0.612863,0.612725,0.612725,0.612863,0.612725,0.162396,0.170060,0.000138,0.007664,False,0.0,0.002299,0.009176,0.002299,0.009176


In [108]:

import ast
candidate_params = params_grid_lr['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]
candidate_params

[{'C': 15.741890047456641,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 641,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0015777663630582469,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 563,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0005397956855996448,
  'penalty': 'elasticnet',
  'solver': 'saga',
  'max_iter': 546,
  'random_state': 42,
  'l1_ratio': 0.034388521115218396,
  'n_jobs': -1},
 {'C': 65.6481761175346,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 939,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 83.67826428540977,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 975,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 94.42207016006594,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 958,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 2.1352376986091604,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 290,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.8388063784216602,
  'penalty': 'l2',
  'solver': 'saga',
  'max_iter': 688,
 

In [109]:
candidate_params[10]

{'C': 0.17823264111966045,
 'penalty': 'l2',
 'solver': 'saga',
 'max_iter': 765,
 'random_state': 42,
 'n_jobs': -1}

In [110]:
model_lr_AlYm = LogisticRegression(**candidate_params[10])
model_lr_AlYm.fit(
    df_lr[feas],
    df_lr['flag'],
)
df_lr['lr_AlTx_prob'] = model_lr_AlYm.predict_proba(df_lr[feas])[:,1]

In [111]:
mdrp  = ModelDeliveryReport(
    df_lr,
    'flag',
    'lr_AlTx_prob',
    'apply_date_new',
    'partner_name',
)
mdrp.generate_full_report(save_path='report_lr/lr_AlTx_report.xlsx')

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_lr/lr_AlTx_report.xlsx


{'样本情况':      Month    Good   Bad   Total Bad_Rate
 0  2026-02   10755   496   11251   0.0441
 1  2026-03   59876  1966   61842   0.0318
 2  2026-04   54317  1197   55514   0.0216
 3  2026-05     161     1     162   0.0062
 4       合计  125109  3660  128769   0.0284,
 '模型效果':             样本类型    Good   Bad   Total Bad_Rate      KS     AUC TOP10_lift  \
 0            all  125109  3660  128769   0.0284  0.1606  0.6129     1.9044   
 1     华通-2026-03   59876  1966   61842   0.0318  0.1468  0.6015     1.7597   
 2  华通-2026-04至05   54478  1198   55676   0.0215  0.1687  0.6150     2.0199   
 3     华通-2026-02   10755   496   11251   0.0441  0.1932  0.6322     2.0163   
 
       PSI  
 0  0.0000  
 1  0.0000  
 2  0.0000  
 3  0.0000  ,
 '模型效果分箱':              样本类型                分箱     好用户   坏用户     总用户   总用户占比   坏用户占比  \
 0             all    (0.043, 0.154]   12180   697   12877  0.1000  0.1904   
 1             all   (0.0366, 0.043]   12376   501   12877  0.1000  0.1369   
 2             all

In [112]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/AlTx/')['lift_details'][0]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,3093,45.0,3048.0,0.014549,0.457649,0.005-0.014,3093,45.0,0.014549,0.457649,0.022889,华通-2026-03
1,2,3092,58.0,3034.0,0.018758,0.590050,0.014-0.016,6185,103.0,0.016653,0.523839,0.052391,华通-2026-03
2,3,3092,63.0,3029.0,0.020375,0.640916,0.016-0.018,9277,166.0,0.017894,0.562860,0.084435,华通-2026-03
3,4,3092,66.0,3026.0,0.021345,0.671436,0.018-0.019,12369,232.0,0.018757,0.590002,0.118006,华通-2026-03
4,5,3092,59.0,3033.0,0.019082,0.600223,0.019-0.021,15461,291.0,0.018822,0.592046,0.148016,华通-2026-03
5,6,3092,70.0,3022.0,0.022639,0.712129,0.021-0.022,18553,361.0,0.019458,0.612059,0.183622,华通-2026-03
6,7,3092,78.0,3014.0,0.025226,0.793515,0.022-0.023,21645,439.0,0.020282,0.637980,0.223296,华通-2026-03
7,8,3092,84.0,3008.0,0.027167,0.854555,0.023-0.024,24737,523.0,0.021142,0.665051,0.266022,华通-2026-03
8,9,3092,88.0,3004.0,0.028461,0.895248,0.024-0.026,27829,611.0,0.021956,0.690627,0.310783,华通-2026-03
9,10,3092,111.0,2981.0,0.035899,1.129233,0.026-0.027,30921,722.0,0.023350,0.734486,0.367243,华通-2026-03


In [113]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/AlTx/')['lift_details'][1]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,2784,27.0,2757.0,0.009698,0.450719,0.006-0.014,2784,27.0,0.009698,0.450719,0.022538,华通-2026-04至05
1,2,2784,24.0,2760.0,0.008621,0.400639,0.014-0.016,5568,51.0,0.009159,0.425679,0.042571,华通-2026-04至05
2,3,2784,32.0,2752.0,0.011494,0.534185,0.016-0.017,8352,83.0,0.009938,0.461848,0.069282,华通-2026-04至05
3,4,2784,36.0,2748.0,0.012931,0.600958,0.017-0.019,11136,119.0,0.010686,0.496625,0.099332,华通-2026-04至05
4,5,2783,45.0,2738.0,0.016170,0.751468,0.019-0.020,13919,164.0,0.011782,0.547579,0.136895,华通-2026-04至05
5,6,2784,52.0,2732.0,0.018678,0.868051,0.020-0.021,16703,216.0,0.012932,0.600994,0.180301,华通-2026-04至05
6,7,2784,46.0,2738.0,0.016523,0.767891,0.021-0.022,19487,262.0,0.013445,0.624838,0.218698,华通-2026-04至05
7,8,2784,48.0,2736.0,0.017241,0.801278,0.022-0.024,22271,310.0,0.013919,0.646894,0.258765,华通-2026-04至05
8,9,2783,48.0,2735.0,0.017248,0.801566,0.024-0.025,25054,358.0,0.014289,0.664075,0.298831,华通-2026-04至05
9,10,2784,57.0,2727.0,0.020474,0.951518,0.025-0.026,27838,415.0,0.014908,0.692821,0.346411,华通-2026-04至05


In [114]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/AlTx/')['lift_details'][2]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,563,9.0,554.0,0.015986,0.362613,0.006-0.014,563,9.0,0.015986,0.362613,0.018145,华通-2026-02
1,2,563,9.0,554.0,0.015986,0.362613,0.014-0.016,1126,18.0,0.015986,0.362613,0.036290,华通-2026-02
2,3,562,16.0,546.0,0.028470,0.645793,0.016-0.018,1688,34.0,0.020142,0.456894,0.068548,华通-2026-02
3,4,563,17.0,546.0,0.030195,0.684936,0.018-0.020,2251,51.0,0.022657,0.513930,0.102823,华通-2026-02
4,5,562,17.0,545.0,0.030249,0.686155,0.020-0.021,2813,68.0,0.024173,0.548338,0.137097,华通-2026-02
5,6,563,21.0,542.0,0.037300,0.846097,0.021-0.022,3376,89.0,0.026363,0.597994,0.179435,华通-2026-02
6,7,562,8.0,554.0,0.014235,0.322896,0.022-0.023,3938,97.0,0.024632,0.558734,0.195565,华通-2026-02
7,8,563,16.0,547.0,0.028419,0.644646,0.023-0.025,4501,113.0,0.025106,0.569481,0.227823,华通-2026-02
8,9,562,24.0,538.0,0.042705,0.968689,0.025-0.026,5063,137.0,0.027059,0.613793,0.276210,华通-2026-02
9,10,563,21.0,542.0,0.037300,0.846097,0.026-0.027,5626,158.0,0.028084,0.637040,0.318548,华通-2026-02


In [115]:
sub_probs = ['model_risk_v9_gxiaojin_score','ali_88_prob_no_oot_gap_auc_002']
feas = []
for fea in sub_probs:
    df_lr[f'{fea}_fill'] = df_lr[fea].fillna(0)
    df_lr[f'{fea}_valid'] = df_lr[f'{fea}_fill'].apply(get_valid_prob)
    df_lr[f'{fea}_trans'] = df_lr[f'{fea}_fill'].apply(LogitTransformation)
    feas.append(f'{fea}_trans')
    feas.append(f'{fea}_valid')

tuner = AntiOverfittingTuner(LogisticRegression,'lr')
params_grid_lr = tuner.optimize_anti_overfitting(
    df_lr[feas],
    df_lr['flag'],
    n_trials=30
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [116]:
pd.DataFrame(params_grid_lr['performance_history']).to_csv('params_log_lr_al_tx_2.csv',index=False)

In [117]:
params_grid_lr = pd.read_csv('params_log_lr_al_tx_2.csv')
params_grid_lr = params_grid_lr[params_grid_lr['is_overfitting']==False]
params_grid_lr

,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
0,0,"{'C': 15.741890047456641, 'penalty': 'l1', 'so...",0.612898,0.612751,0.612751,0.612898,0.612751,0.162465,0.170882,0.000148,0.008417,False,0.0,0.002256,0.009074,0.002256,0.009074
1,3,"{'C': 0.0015777663630582469, 'penalty': None, ...",0.612898,0.612751,0.612751,0.612898,0.612751,0.162455,0.170906,0.000148,0.008451,False,0.0,0.002256,0.009074,0.002256,0.009074
2,5,"{'C': 0.0005397956855996448, 'penalty': 'elast...",0.585524,0.585399,0.585399,0.585524,0.585399,0.132843,0.137534,0.000125,0.004691,False,0.0,0.002222,0.008967,0.002222,0.008967
3,10,"{'C': 65.6481761175346, 'penalty': 'l1', 'solv...",0.612898,0.612751,0.612751,0.612898,0.612751,0.162463,0.170882,0.000147,0.008419,False,0.0,0.002256,0.009074,0.002256,0.009074
4,11,"{'C': 83.67826428540977, 'penalty': 'l1', 'sol...",0.612898,0.612751,0.612751,0.612898,0.612751,0.162457,0.170890,0.000147,0.008433,False,0.0,0.002256,0.009074,0.002256,0.009074
5,12,"{'C': 94.42207016006594, 'penalty': 'l1', 'sol...",0.612898,0.612751,0.612751,0.612898,0.612751,0.162457,0.170898,0.000147,0.008441,False,0.0,0.002256,0.009074,0.002256,0.009074
6,13,"{'C': 1.7839064425793556, 'penalty': 'l1', 'so...",0.612869,0.612722,0.612722,0.612869,0.612722,0.162439,0.170882,0.000148,0.008443,False,0.0,0.002254,0.009068,0.002254,0.009068
7,15,"{'C': 0.4046598398272463, 'penalty': 'l2', 'so...",0.612894,0.612747,0.612747,0.612894,0.612747,0.162575,0.170370,0.000147,0.007795,False,0.0,0.002253,0.009038,0.002253,0.009038
8,16,"{'C': 54.92693488469529, 'penalty': 'l1', 'sol...",0.612898,0.612751,0.612751,0.612898,0.612751,0.162463,0.170882,0.000147,0.008419,False,0.0,0.002256,0.009074,0.002256,0.009074
9,17,"{'C': 0.024379283903864225, 'penalty': 'l1', '...",0.612886,0.612693,0.612693,0.612886,0.612693,0.162604,0.170167,0.000193,0.007563,False,0.0,0.002253,0.009043,0.002253,0.009043


In [118]:

import ast
candidate_params = params_grid_lr['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]
candidate_params

[{'C': 15.741890047456641,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 641,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0015777663630582469,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 563,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0005397956855996448,
  'penalty': 'elasticnet',
  'solver': 'saga',
  'max_iter': 546,
  'random_state': 42,
  'l1_ratio': 0.034388521115218396,
  'n_jobs': -1},
 {'C': 65.6481761175346,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 939,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 83.67826428540977,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 975,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 94.42207016006594,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 971,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 1.7839064425793556,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 972,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.4046598398272463,
  'penalty': 'l2',
  'solver': 'saga',
  'max_iter': 187,
 

In [119]:
candidate_params[16]

{'C': 11.046460663472564,
 'penalty': 'l1',
 'solver': 'saga',
 'max_iter': 291,
 'random_state': 42,
 'n_jobs': -1}

In [120]:
model_lr_AlYm = LogisticRegression(**candidate_params[16])
model_lr_AlYm.fit(
    df_lr[feas],
    df_lr['flag'],
)
df_lr['lr_AlTx_prob_2'] = model_lr_AlYm.predict_proba(df_lr[feas])[:,1]

In [121]:
mdrp  = ModelDeliveryReport(
    df_lr,
    'flag',
    'lr_AlTx_prob_2',
    'apply_date_new',
    'partner_name',
)
mdrp.generate_full_report(save_path='report_lr/lr_AlTx_report_2.xlsx')

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_lr/lr_AlTx_report_2.xlsx


{'样本情况':      Month    Good   Bad   Total Bad_Rate
 0  2026-02   10755   496   11251   0.0441
 1  2026-03   59876  1966   61842   0.0318
 2  2026-04   54317  1197   55514   0.0216
 3  2026-05     161     1     162   0.0062
 4       合计  125109  3660  128769   0.0284,
 '模型效果':             样本类型    Good   Bad   Total Bad_Rate      KS     AUC TOP10_lift  \
 0            all  125109  3660  128769   0.0284  0.1612  0.6129     1.9180   
 1     华通-2026-03   59876  1966   61842   0.0318  0.1521  0.6022     1.7902   
 2  华通-2026-04至05   54478  1198   55676   0.0215  0.1587  0.6136     1.9698   
 3     华通-2026-02   10755   496   11251   0.0441  0.1953  0.6313     1.8953   
 
       PSI  
 0  0.0000  
 1  0.0000  
 2  0.0000  
 3  0.0000  ,
 '模型效果分箱':              样本类型                               分箱     好用户   坏用户     总用户  \
 0             all                  (0.0437, 0.166]   12175   702   12877   
 1             all                 (0.0369, 0.0437]   12377   500   12877   
 2             all   

In [122]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/AlTx_2/')['lift_details'][0]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,3093,44.0,3049.0,0.014226,0.447479,0.003-0.014,3093,44.0,0.014226,0.447479,0.022380,华通-2026-03
1,2,3092,60.0,3032.0,0.019405,0.610396,0.014-0.016,6185,104.0,0.016815,0.528924,0.052899,华通-2026-03
2,3,3092,65.0,3027.0,0.021022,0.661262,0.016-0.018,9277,169.0,0.018217,0.573032,0.085961,华通-2026-03
3,4,3092,60.0,3032.0,0.019405,0.610396,0.018-0.019,12369,229.0,0.018514,0.582373,0.116480,华通-2026-03
4,5,3092,62.0,3030.0,0.020052,0.630743,0.019-0.020,15461,291.0,0.018822,0.592046,0.148016,华通-2026-03
5,6,3092,77.0,3015.0,0.024903,0.783342,0.020-0.022,18553,368.0,0.019835,0.623927,0.187182,华通-2026-03
6,7,3092,83.0,3009.0,0.026843,0.844381,0.022-0.023,21645,451.0,0.020836,0.655419,0.229400,华通-2026-03
7,8,3092,75.0,3017.0,0.024256,0.762995,0.023-0.024,24737,526.0,0.021264,0.668865,0.267548,华通-2026-03
8,9,3092,86.0,3006.0,0.027814,0.874901,0.024-0.026,27829,612.0,0.021991,0.691757,0.311292,华通-2026-03
9,10,3092,96.0,2996.0,0.031048,0.976634,0.026-0.027,30921,708.0,0.022897,0.720244,0.360122,华通-2026-03


In [123]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/AlTx/')['lift_details'][1]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,2784,24.0,2760.0,0.008621,0.400639,0.002-0.013,2784,24.0,0.008621,0.400639,0.020033,华通-2026-04至05
1,2,2784,27.0,2757.0,0.009698,0.450719,0.013-0.015,5568,51.0,0.009159,0.425679,0.042571,华通-2026-04至05
2,3,2784,33.0,2751.0,0.011853,0.550879,0.015-0.017,8352,84.0,0.010057,0.467412,0.070117,华通-2026-04至05
3,4,2784,39.0,2745.0,0.014009,0.651038,0.017-0.018,11136,123.0,0.011045,0.513319,0.102671,华通-2026-04至05
4,5,2783,47.0,2736.0,0.016888,0.784867,0.018-0.019,13919,170.0,0.012214,0.567613,0.141903,华通-2026-04至05
5,6,2784,48.0,2736.0,0.017241,0.801278,0.019-0.021,16703,218.0,0.013052,0.606559,0.181970,华通-2026-04至05
6,7,2784,40.0,2744.0,0.014368,0.667732,0.021-0.022,19487,258.0,0.013240,0.615299,0.215359,华通-2026-04至05
7,8,2784,54.0,2730.0,0.019397,0.901438,0.022-0.023,22271,312.0,0.014009,0.651068,0.260434,华通-2026-04至05
8,9,2783,51.0,2732.0,0.018326,0.851664,0.023-0.024,25054,363.0,0.014489,0.673350,0.303005,华通-2026-04至05
9,10,2784,52.0,2732.0,0.018678,0.868051,0.024-0.026,27838,415.0,0.014908,0.692821,0.346411,华通-2026-04至05


In [124]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/AlTx/')['lift_details'][2]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,563,8.0,555.0,0.014210,0.322323,0.003-0.014,563,8.0,0.014210,0.322323,0.016129,华通-2026-02
1,2,563,10.0,553.0,0.017762,0.402904,0.014-0.016,1126,18.0,0.015986,0.362613,0.036290,华通-2026-02
2,3,562,17.0,545.0,0.030249,0.686155,0.016-0.018,1688,35.0,0.020735,0.470333,0.070565,华通-2026-02
3,4,563,12.0,551.0,0.021314,0.483484,0.018-0.019,2251,47.0,0.020880,0.473622,0.094758,华通-2026-02
4,5,562,13.0,549.0,0.023132,0.524707,0.019-0.021,2813,60.0,0.021330,0.483828,0.120968,华通-2026-02
5,6,563,25.0,538.0,0.044405,1.007259,0.021-0.022,3376,85.0,0.025178,0.571118,0.171371,华通-2026-02
6,7,562,17.0,545.0,0.030249,0.686155,0.022-0.023,3938,102.0,0.025901,0.587535,0.205645,华通-2026-02
7,8,563,15.0,548.0,0.026643,0.604355,0.023-0.024,4501,117.0,0.025994,0.589639,0.235887,华通-2026-02
8,9,562,17.0,545.0,0.030249,0.686155,0.024-0.026,5063,134.0,0.026467,0.600352,0.270161,华通-2026-02
9,10,563,24.0,539.0,0.042629,0.966968,0.026-0.027,5626,158.0,0.028084,0.637040,0.318548,华通-2026-02


# tx al ym

In [125]:
df_lr.columns

Index(['serie_no', 'id_card_md5', 'prob_embeddings_v4',
       'ali_88_prob_keep_oot_4_5m', 'ali_88_prob_no_oot',
       'ali_88_prob_no_oot_gap_auc_002', 'bd_lgb_NoOot_prob', 'test_id',
       'flag', 'month', 'partner_name', 'mid_sample_type', 'apply_date_new',
       'bd_lgb_NoOot_prob_fill', 'bd_lgb_NoOot_prob_valid',
       'bd_lgb_NoOot_prob_trans', 'ali_88_prob_no_oot_gap_auc_002_fill',
       'ali_88_prob_no_oot_gap_auc_002_valid',
       'ali_88_prob_no_oot_gap_auc_002_trans', 'lr_BdAl_prob',
       'prob_embeddings_v4_fill', 'prob_embeddings_v4_valid',
       'prob_embeddings_v4_trans', 'lr_BdYm_prob',
       'model_risk_v9_gtongyong_score', 'model_risk_v9_gxiaojin_score',
       'model_risk_v9_gtongyong_score_fill',
       'model_risk_v9_gtongyong_score_valid',
       'model_risk_v9_gtongyong_score_trans', 'lr_AlTx_prob',
       'model_risk_v9_gxiaojin_score_fill',
       'model_risk_v9_gxiaojin_score_valid',
       'model_risk_v9_gxiaojin_score_trans', 'lr_AlTx_prob_2'],
  

In [126]:
sub_probs = ['model_risk_v9_gxiaojin_score','ali_88_prob_no_oot_gap_auc_002','prob_embeddings_v4']
feas = []
for fea in sub_probs:
    df_lr[f'{fea}_fill'] = df_lr[fea].fillna(0)
    df_lr[f'{fea}_valid'] = df_lr[f'{fea}_fill'].apply(get_valid_prob)
    df_lr[f'{fea}_trans'] = df_lr[f'{fea}_fill'].apply(LogitTransformation)
    feas.append(f'{fea}_trans')
    feas.append(f'{fea}_valid')

tuner = AntiOverfittingTuner(LogisticRegression,'lr')
params_grid_lr = tuner.optimize_anti_overfitting(
    df_lr[feas],
    df_lr['flag'],
    n_trials=30
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [127]:
pd.DataFrame(params_grid_lr['performance_history']).to_csv('params_log_lr_AlTxYm.csv',index=False)

In [128]:
params_grid_lr = pd.read_csv('params_log_lr_AlTxYm.csv')
params_grid_lr = params_grid_lr[params_grid_lr['is_overfitting']==False]
params_grid_lr

,trial,params,train_score,val_score,final_score,train_auc,val_auc,train_ks,val_ks,auc_gap,ks_gap,is_overfitting,total_penalty,train_score_std,val_score_std,train_auc_std,val_auc_std
0,0,"{'C': 15.741890047456641, 'penalty': 'l1', 'so...",0.625019,0.624749,0.624749,0.625019,0.624749,0.180894,0.186795,0.000270,0.005901,False,0.0,0.002266,0.009106,0.002266,0.009106
1,3,"{'C': 0.0015777663630582469, 'penalty': None, ...",0.625019,0.624749,0.624749,0.625019,0.624749,0.180898,0.186747,0.000270,0.005849,False,0.0,0.002266,0.009107,0.002266,0.009107
2,5,"{'C': 0.0005397956855996448, 'penalty': 'elast...",0.592196,0.592013,0.592013,0.592196,0.592013,0.135516,0.143391,0.000183,0.007875,False,0.0,0.002159,0.008916,0.002159,0.008916
3,10,"{'C': 0.000769639183009799, 'penalty': None, '...",0.625019,0.624749,0.624749,0.625019,0.624749,0.180886,0.186763,0.000270,0.005877,False,0.0,0.002266,0.009107,0.002266,0.009107
4,11,"{'C': 68.20019829930264, 'penalty': 'l1', 'sol...",0.625019,0.624749,0.624749,0.625019,0.624749,0.180898,0.186763,0.000270,0.005865,False,0.0,0.002266,0.009107,0.002266,0.009107
5,12,"{'C': 4.238271500303412, 'penalty': 'l1', 'sol...",0.625013,0.624740,0.624740,0.625013,0.624740,0.180860,0.186739,0.000272,0.005879,False,0.0,0.002266,0.009102,0.002266,0.009102
6,13,"{'C': 0.0778713991593174, 'penalty': None, 'so...",0.625019,0.624749,0.624749,0.625019,0.624749,0.180900,0.186747,0.000270,0.005847,False,0.0,0.002266,0.009107,0.002266,0.009107
7,15,"{'C': 0.03569251797824189, 'penalty': 'l2', 's...",0.624446,0.624194,0.624194,0.624446,0.624194,0.176238,0.182420,0.000252,0.006183,False,0.0,0.002210,0.008938,0.002210,0.008938
8,16,"{'C': 2.838926264552263, 'penalty': None, 'sol...",0.625019,0.624749,0.624749,0.625019,0.624749,0.180900,0.186747,0.000270,0.005847,False,0.0,0.002266,0.009107,0.002266,0.009107
9,17,"{'C': 1.1045487630580906, 'penalty': None, 'so...",0.625019,0.624749,0.624749,0.625019,0.624749,0.180900,0.186747,0.000270,0.005847,False,0.0,0.002266,0.009107,0.002266,0.009107


In [129]:

import ast
candidate_params = params_grid_lr['params'].to_list()
candidate_params = [ast.literal_eval(s) for s in candidate_params]
candidate_params

[{'C': 15.741890047456641,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 641,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0015777663630582469,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 563,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0005397956855996448,
  'penalty': 'elasticnet',
  'solver': 'saga',
  'max_iter': 546,
  'random_state': 42,
  'l1_ratio': 0.034388521115218396,
  'n_jobs': -1},
 {'C': 0.000769639183009799,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 116,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 68.20019829930264,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 674,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 4.238271500303412,
  'penalty': 'l1',
  'solver': 'saga',
  'max_iter': 570,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.0778713991593174,
  'penalty': None,
  'solver': 'saga',
  'max_iter': 942,
  'random_state': 42,
  'n_jobs': -1},
 {'C': 0.03569251797824189,
  'penalty': 'l2',
  'solver': 'saga',
  'max_iter': 3

In [130]:
candidate_params[13]

{'C': 0.008415909791545785,
 'penalty': None,
 'solver': 'saga',
 'max_iter': 783,
 'random_state': 42,
 'n_jobs': -1}

In [131]:
model_lr_AlYm = LogisticRegression(**candidate_params[13])
model_lr_AlYm.fit(
    df_lr[feas],
    df_lr['flag'],
)
df_lr['lr_AlTxYm_prob'] = model_lr_AlYm.predict_proba(df_lr[feas])[:,1]

In [132]:
mdrp  = ModelDeliveryReport(
    df_lr,
    'flag',
    'lr_AlTxYm_prob',
    'apply_date_new',
    'partner_name',
)
mdrp.generate_full_report(save_path='report_lr/lr_AlTxYm_report.xlsx')

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_lr/lr_AlTxYm_report.xlsx


{'样本情况':      Month    Good   Bad   Total Bad_Rate
 0  2026-02   10755   496   11251   0.0441
 1  2026-03   59876  1966   61842   0.0318
 2  2026-04   54317  1197   55514   0.0216
 3  2026-05     161     1     162   0.0062
 4       合计  125109  3660  128769   0.0284,
 '模型效果':             样本类型    Good   Bad   Total Bad_Rate      KS     AUC TOP10_lift  \
 0            all  125109  3660  128769   0.0284  0.1801  0.6250     2.0027   
 1     华通-2026-03   59876  1966   61842   0.0318  0.1688  0.6132     1.8411   
 2  华通-2026-04至05   54478  1198   55676   0.0215  0.1830  0.6296     2.0115   
 3     华通-2026-02   10755   496   11251   0.0441  0.2092  0.6370     2.2179   
 
       PSI  
 0  0.0000  
 1  0.0000  
 2  0.0000  
 3  0.0000  ,
 '模型效果分箱':              样本类型                                 分箱     好用户   坏用户     总用户  \
 0             all                    (0.0455, 0.199]   12144   733   12877   
 1             all                   (0.0375, 0.0455]   12351   526   12877   
 2             

In [134]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/AlTxYm/')['lift_details'][0]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,3093,37.0,3056.0,0.011962,0.376289,0.001-0.013,3093,37.0,0.011962,0.376289,0.018820,华通-2026-03
1,2,3092,47.0,3045.0,0.015201,0.478144,0.013-0.015,6185,84.0,0.013581,0.427208,0.042726,华通-2026-03
2,3,3092,64.0,3028.0,0.020699,0.651089,0.015-0.017,9277,148.0,0.015953,0.501827,0.075280,华通-2026-03
3,4,3092,79.0,3013.0,0.025550,0.803688,0.017-0.018,12369,227.0,0.018352,0.577286,0.115463,华通-2026-03
4,5,3092,68.0,3024.0,0.021992,0.691782,0.018-0.019,15461,295.0,0.019080,0.600184,0.150051,华通-2026-03
5,6,3092,63.0,3029.0,0.020375,0.640916,0.019-0.021,18553,358.0,0.019296,0.606972,0.182096,华通-2026-03
6,7,3092,65.0,3027.0,0.021022,0.661262,0.021-0.022,21645,423.0,0.019543,0.614728,0.215158,华通-2026-03
7,8,3092,95.0,2997.0,0.030724,0.966461,0.022-0.024,24737,518.0,0.020940,0.658693,0.263479,华通-2026-03
8,9,3092,95.0,2997.0,0.030724,0.966461,0.024-0.025,27829,613.0,0.022027,0.692888,0.311801,华通-2026-03
9,10,3092,81.0,3011.0,0.026197,0.824035,0.025-0.026,30921,694.0,0.022444,0.706002,0.353001,华通-2026-03


In [135]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/AlTxYm/')['lift_details'][1]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,2784,18.0,2766.0,0.006466,0.300479,0.001-0.012,2784,18.0,0.006466,0.300479,0.015025,华通-2026-04至05
1,2,2784,22.0,2762.0,0.007902,0.367252,0.012-0.014,5568,40.0,0.007184,0.333866,0.033389,华通-2026-04至05
2,3,2784,30.0,2754.0,0.010776,0.500799,0.014-0.016,8352,70.0,0.008381,0.389510,0.058431,华通-2026-04至05
3,4,2784,30.0,2754.0,0.010776,0.500799,0.016-0.017,11136,100.0,0.008980,0.417332,0.083472,华通-2026-04至05
4,5,2783,43.0,2740.0,0.015451,0.718069,0.017-0.018,13919,143.0,0.010274,0.477462,0.119366,华通-2026-04至05
5,6,2784,48.0,2736.0,0.017241,0.801278,0.018-0.020,16703,191.0,0.011435,0.531435,0.159432,华通-2026-04至05
6,7,2784,36.0,2748.0,0.012931,0.600958,0.020-0.021,19487,227.0,0.011649,0.541367,0.189482,华通-2026-04至05
7,8,2784,52.0,2732.0,0.018678,0.868051,0.021-0.022,22271,279.0,0.012528,0.582205,0.232888,华通-2026-04至05
8,9,2783,53.0,2730.0,0.019044,0.885062,0.022-0.024,25054,332.0,0.013251,0.615846,0.277129,华通-2026-04至05
9,10,2784,70.0,2714.0,0.025144,1.168530,0.024-0.025,27838,402.0,0.014441,0.671119,0.335559,华通-2026-04至05


In [136]:
mdrp.generate_metric_report_plot(n_bins=20,save_dir='report_lr/AlTxYm/')['lift_details'][2]

,分箱,样本数,正样本数,负样本数,正样本率,Lift,概率范围,累积样本数,累积正样本数,累积正样本率,累积Lift,累积召回率,样本类型
0,1,563,7.0,556.0,0.012433,0.282032,0.001-0.013,563,7.0,0.012433,0.282032,0.014113,华通-2026-02
1,2,563,13.0,550.0,0.023091,0.523775,0.013-0.015,1126,20.0,0.017762,0.402904,0.040323,华通-2026-02
2,3,562,13.0,549.0,0.023132,0.524707,0.015-0.017,1688,33.0,0.019550,0.443456,0.066532,华通-2026-02
3,4,563,15.0,548.0,0.026643,0.604355,0.017-0.018,2251,48.0,0.021324,0.483699,0.096774,华通-2026-02
4,5,562,19.0,543.0,0.033808,0.766879,0.018-0.020,2813,67.0,0.023818,0.540275,0.135081,华通-2026-02
5,6,563,14.0,549.0,0.024867,0.564065,0.020-0.021,3376,81.0,0.023993,0.544242,0.163306,华通-2026-02
6,7,562,15.0,547.0,0.026690,0.605431,0.021-0.023,3938,96.0,0.024378,0.552974,0.193548,华通-2026-02
7,8,563,21.0,542.0,0.037300,0.846097,0.023-0.024,4501,117.0,0.025994,0.589639,0.235887,华通-2026-02
8,9,562,24.0,538.0,0.042705,0.968689,0.024-0.025,5063,141.0,0.027849,0.631714,0.284274,华通-2026-02
9,10,563,15.0,548.0,0.026643,0.604355,0.025-0.027,5626,156.0,0.027728,0.628976,0.314516,华通-2026-02


In [137]:
df_label.columns

Index(['1-阿里反欺诈v2标品', 'B068_prob', 'B069_prob', 'apply_date', 'b058_bh_prob',
       'b058_bh_prob_new_export', 'b066_bh_model_prob', 'b066_prob',
       'b066_score', 'b067_bh_prob', 'b067_prob', 'bh_imbank_proba(B058)',
       'c031_bh_prob', 'enc_method', 'flag', 'high_sample_type',
       'hl_jianpu_score_v1', 'hl_yuxin_score_v3_a', 'hl_yuxin_score_v3_b',
       'hl_yuxin_score_v3_c', 'id_number', 'mid_sample_type', 'mobile',
       'model_risk_v9_gtongyong_score', 'model_risk_v9_gxiaojin_score',
       'partner_code', 'partner_name', 'pric_type_code', 'pric_type_desc',
       'test_id', 'ym_A077', '蜜蜂分_中利率版2_2', '蜜蜂分_中利率版2_2_proba', '蜜蜂分_中利率版6_2',
       '蜜蜂分_中利率版6_2_proba', '蜜蜂分_中利率版6_3_proba', '蜜蜂分_中利率版7_2',
       '蜜蜂分_中利率版7_2_proba', '蜜蜂分_中利率版8_1_proba_a073', '蜜蜂分_中利率版8_2_proba_a074',
       '蜜蜂分_中利率版8_3_proba_a075', '蜜蜂分_中利率版_1', '蜜蜂分_中利率版_1_proba',
       '蜜蜂分_高利率版10_proba(B058)', '蜜蜂分_高利率版5_2', '蜜蜂分_高利率版5_2_proba',
       '蜜蜂分_高利率版6_2', '蜜蜂分_高利率版6_2_proba', '蜜蜂分_高利率版7_2', '

In [138]:
df_label.shape

(128769, 50)

In [157]:
df_label['partner_name'].value_counts()

partner_name
华通银行-星选    128769
Name: count, dtype: int64

# 报告

In [140]:
df_scores = pd.read_csv('/Users/mayongzhi/Job/Job_xyd/数据/客户测建模/华通/华通银行_小雨点测试结果_20260717.csv',encoding='gbk')
df_scores.head()

,serie_no,rn,cif_cust_no,make_loan_date,姓名,身份证,手机号,蜜蜂分B_062,蜜蜂分A_064,蜜蜂分B_054,蜜蜂分B_059,蜜蜂分B_070,xyd_modelHIRR1_1_20250616,xyd_modelMIRR2_20251103
0,BH000001,4,100249100519,2026-03-06,474deb1a1a00f2df6fe1426af7aec00d,1e0968a3c477e2a4e5a5d68fb3712d74,fa4fc24bb7eb64ad38befd4c8bb753ae,523,586,663,587,494,17.011297,11.361520
1,BH000002,106,100109734289,2026-03-23,49121e96fad562a122ebf2d55106a367,3de08d3162b2aed10933cc74165808b7,b821b26916d2c64ff89872f40d858257,585,658,705,649,609,12.167068,6.765089
2,BH000003,114,100253118290,2026-04-05,812d0e8e2ba9281155307138861e674f,89d5fd9ad9ea658b847d6f449908a674,11b515f674b7a9639e1efd47acaddd63,503,510,623,539,595,23.592120,16.525286
3,BH000004,120,100220083085,2026-04-08,acc5115681d105bef92fdfe98b9d0ebb,4e861b04b6721d7c9e85421d2aa86018,4a18e4983795edc5c2d5a5d55f94dfbe,584,685,725,665,617,10.662794,5.963670
4,BH000005,176,100247676250,2026-02-28,04216d6f3d6e724010e5212c56b4a537,a343b5d96e944690c34ada406b395a4f,6079d651e4d5f0b422a5de7ed38b7410,539,646,693,585,553,12.889217,9.128659


In [149]:
df_scores['手机号'].nunique()

128769

In [141]:
df_scores.shape

(128769, 14)

In [147]:
df_lr[['serie_no','flag']].dtypes

serie_no      int64
flag        float64
dtype: object

In [148]:
df_scores['serie_no'].dtypes

dtype('O')

In [154]:
df_scores.merge(df_label[['mobile','flag']],left_on='手机号',right_on='mobile').head()

,serie_no,rn,cif_cust_no,make_loan_date,姓名,身份证,手机号,蜜蜂分B_062,蜜蜂分A_064,蜜蜂分B_054,蜜蜂分B_059,蜜蜂分B_070,xyd_modelHIRR1_1_20250616,xyd_modelMIRR2_20251103,mobile,flag
0,BH000001,4,100249100519,2026-03-06,474deb1a1a00f2df6fe1426af7aec00d,1e0968a3c477e2a4e5a5d68fb3712d74,fa4fc24bb7eb64ad38befd4c8bb753ae,523,586,663,587,494,17.011297,11.361520,fa4fc24bb7eb64ad38befd4c8bb753ae,1.0
1,BH000002,106,100109734289,2026-03-23,49121e96fad562a122ebf2d55106a367,3de08d3162b2aed10933cc74165808b7,b821b26916d2c64ff89872f40d858257,585,658,705,649,609,12.167068,6.765089,b821b26916d2c64ff89872f40d858257,1.0
2,BH000003,114,100253118290,2026-04-05,812d0e8e2ba9281155307138861e674f,89d5fd9ad9ea658b847d6f449908a674,11b515f674b7a9639e1efd47acaddd63,503,510,623,539,595,23.592120,16.525286,11b515f674b7a9639e1efd47acaddd63,1.0
3,BH000004,120,100220083085,2026-04-08,acc5115681d105bef92fdfe98b9d0ebb,4e861b04b6721d7c9e85421d2aa86018,4a18e4983795edc5c2d5a5d55f94dfbe,584,685,725,665,617,10.662794,5.963670,4a18e4983795edc5c2d5a5d55f94dfbe,1.0
4,BH000005,176,100247676250,2026-02-28,04216d6f3d6e724010e5212c56b4a537,a343b5d96e944690c34ada406b395a4f,6079d651e4d5f0b422a5de7ed38b7410,539,646,693,585,553,12.889217,9.128659,6079d651e4d5f0b422a5de7ed38b7410,1.0


In [155]:
df_scores.merge(df_label[['mobile','flag']],left_on='手机号',right_on='mobile').shape

(128769, 16)

In [156]:
df_scores = df_scores.merge(df_label[['mobile','flag']],left_on='手机号',right_on='mobile')

In [158]:
df_lr.columns

Index(['serie_no', 'id_card_md5', 'prob_embeddings_v4',
       'ali_88_prob_keep_oot_4_5m', 'ali_88_prob_no_oot',
       'ali_88_prob_no_oot_gap_auc_002', 'bd_lgb_NoOot_prob', 'test_id',
       'flag', 'month', 'partner_name', 'mid_sample_type', 'apply_date_new',
       'bd_lgb_NoOot_prob_fill', 'bd_lgb_NoOot_prob_valid',
       'bd_lgb_NoOot_prob_trans', 'ali_88_prob_no_oot_gap_auc_002_fill',
       'ali_88_prob_no_oot_gap_auc_002_valid',
       'ali_88_prob_no_oot_gap_auc_002_trans', 'lr_BdAl_prob',
       'prob_embeddings_v4_fill', 'prob_embeddings_v4_valid',
       'prob_embeddings_v4_trans', 'lr_BdYm_prob',
       'model_risk_v9_gtongyong_score', 'model_risk_v9_gxiaojin_score',
       'model_risk_v9_gtongyong_score_fill',
       'model_risk_v9_gtongyong_score_valid',
       'model_risk_v9_gtongyong_score_trans', 'lr_AlTx_prob',
       'model_risk_v9_gxiaojin_score_fill',
       'model_risk_v9_gxiaojin_score_valid',
       'model_risk_v9_gxiaojin_score_trans', 'lr_AlTx_prob_2',
   

In [159]:
df_lr['id_card_md5'].nunique()

128769

In [160]:
df_scores.merge(df_lr[['id_card_md5','month', 'partner_name', 'mid_sample_type']],left_on='身份证',right_on='id_card_md5').shape

(128769, 20)

In [161]:
df_scores = df_scores.merge(df_lr[['id_card_md5','month', 'partner_name', 'mid_sample_type']],left_on='身份证',right_on='id_card_md5')
df_scores.head()

,serie_no,rn,cif_cust_no,make_loan_date,姓名,身份证,手机号,蜜蜂分B_062,蜜蜂分A_064,蜜蜂分B_054,蜜蜂分B_059,蜜蜂分B_070,xyd_modelHIRR1_1_20250616,xyd_modelMIRR2_20251103,mobile,flag,id_card_md5,month,partner_name,mid_sample_type
0,BH000001,4,100249100519,2026-03-06,474deb1a1a00f2df6fe1426af7aec00d,1e0968a3c477e2a4e5a5d68fb3712d74,fa4fc24bb7eb64ad38befd4c8bb753ae,523,586,663,587,494,17.011297,11.361520,fa4fc24bb7eb64ad38befd4c8bb753ae,1.0,1e0968a3c477e2a4e5a5d68fb3712d74,202603,华通-2026-03,1-train
1,BH000002,106,100109734289,2026-03-23,49121e96fad562a122ebf2d55106a367,3de08d3162b2aed10933cc74165808b7,b821b26916d2c64ff89872f40d858257,585,658,705,649,609,12.167068,6.765089,b821b26916d2c64ff89872f40d858257,1.0,3de08d3162b2aed10933cc74165808b7,202603,华通-2026-03,1-train
2,BH000003,114,100253118290,2026-04-05,812d0e8e2ba9281155307138861e674f,89d5fd9ad9ea658b847d6f449908a674,11b515f674b7a9639e1efd47acaddd63,503,510,623,539,595,23.592120,16.525286,11b515f674b7a9639e1efd47acaddd63,1.0,89d5fd9ad9ea658b847d6f449908a674,202604,华通-2026-04至05,3-oot
3,BH000004,120,100220083085,2026-04-08,acc5115681d105bef92fdfe98b9d0ebb,4e861b04b6721d7c9e85421d2aa86018,4a18e4983795edc5c2d5a5d55f94dfbe,584,685,725,665,617,10.662794,5.963670,4a18e4983795edc5c2d5a5d55f94dfbe,1.0,4e861b04b6721d7c9e85421d2aa86018,202604,华通-2026-04至05,3-oot
4,BH000005,176,100247676250,2026-02-28,04216d6f3d6e724010e5212c56b4a537,a343b5d96e944690c34ada406b395a4f,6079d651e4d5f0b422a5de7ed38b7410,539,646,693,585,553,12.889217,9.128659,6079d651e4d5f0b422a5de7ed38b7410,1.0,a343b5d96e944690c34ada406b395a4f,202602,华通-2026-02,1-train


In [162]:
df_scores.drop(columns=['id_card_md5','mobile','mid_sample_type'],inplace=True)

In [163]:
ModelDeliveryReport??

Init signature:
ModelDeliveryReport(
    data: pandas.core.frame.DataFrame,
    target_col: str = 'target',
    score_col: Union[str, List[str]] = 'score',
    date_col: str = 'date',
    sample_type_col: str = 'sample_type',
    score_na_strategy: str = 'intersection',
)
Source:        
class ModelDeliveryReport:
    """模型交付报告生成器"""

    def __init__(self, data: pd.DataFrame, target_col: str = 'target',
                 score_col: Union[str, List[str]] = 'score', date_col: str = 'date',
                 sample_type_col: str = 'sample_type',
                 score_na_strategy: str = 'intersection'):
        """
        初始化报告生成器

        Args:
            data: 包含预测结果的数据框
            target_col: 目标变量列名
            score_col: 模型分数列名，支持两种形式：
                       - str: 单个分数列（原有行为）
                       - List[str]: 多个分数列，对比多个分数的表现
            date_col: 日期列名
            sample_type_col: 样本类型列名 (all, train, test, oot)
            score_na_strategy: 多分数模式下的缺失处理策略：
                       -

In [165]:
"""
模型交付报告生成器
用于生成标准化的模型交付报告，包含样本情况、模型效果、变量分析等内容
"""

import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Tuple, Any, Union
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


class ModelDeliveryReport:
    """模型交付报告生成器"""

    def __init__(self, data: pd.DataFrame, target_col: str = 'target',
                 score_col: Union[str, List[str]] = 'score', date_col: str = 'date',
                 sample_type_col: str = 'sample_type',
                 score_na_strategy: str = 'intersection',
                 score_direction: Union[str, Dict[str, str]] = 'higher_is_bad'):
        """
        初始化报告生成器

        Args:
            data: 包含预测结果的数据框
            target_col: 目标变量列名
            score_col: 模型分数列名，支持两种形式：
                       - str: 单个分数列（原有行为）
                       - List[str]: 多个分数列，对比多个分数的表现
            date_col: 日期列名
            sample_type_col: 样本类型列名 (all, train, test, oot)
            score_na_strategy: 多分数模式下的缺失处理策略：
                       - 'intersection'（默认）: 仅保留所有分数都不为空的样本，
                         生成整体对比报告（宽表，一行一个样本类型，指标列带分数后缀，
                         如 KS_score1、AUC_score1）。
                       - 'separate': 不取交集，按样本类型对每个分数各自用其非空样本
                         计算表现（长表，每个 (样本类型, 分数) 一行，Good/Bad/Total
                         均基于该分数非空样本，并额外给出缺失率列）。
            score_direction: 分数与坏样本风险的方向，支持：
                       - 'higher_is_bad'（默认）: 分数越高风险越高，例如违约概率。
                       - 'higher_is_good': 分数越高风险越低，例如信用评分。
                       - 'auto': 根据全量有效样本的原始AUC自动判断方向。
                       - Dict[str, str]: 多分数模式下按分数列分别配置方向。
        """
        self.data = data.copy()
        self.target_col = target_col

        # 规范化score_col：统一为列表形式，便于多分数处理
        if isinstance(score_col, (list, tuple)):
            self.score_cols = list(score_col)
        else:
            self.score_cols = [score_col]
        self.is_multi_score = len(self.score_cols) > 1
        # 主分数列（向后兼容：单分数方法默认使用第一个分数列）
        self.score_col = self.score_cols[0]

        # 缺失处理策略校验
        if score_na_strategy not in ('intersection', 'separate'):
            raise ValueError(
                f"score_na_strategy 仅支持 'intersection' 或 'separate'，收到: {score_na_strategy}"
            )
        self.score_na_strategy = score_na_strategy

        self.date_col = date_col
        self.sample_type_col = sample_type_col

        # 校验分数列存在性
        missing_cols = [c for c in self.score_cols if c not in self.data.columns]
        if missing_cols:
            raise ValueError(f"以下分数列不存在于数据中: {missing_cols}")

        # intersection 策略：多分数模式下仅保留所有分数都不为空的样本
        if self.is_multi_score and self.score_na_strategy == 'intersection':
            before = len(self.data)
            self.data = self.data.dropna(subset=self.score_cols).reset_index(drop=True)
            after = len(self.data)
            if after < before:
                print(f"多分数模式(intersection)：过滤掉 {before - after} 行存在缺失分数的样本，剩余 {after} 行")

        # 规范化每个分数列的方向配置
        valid_directions = {'higher_is_bad', 'higher_is_good', 'auto'}
        if isinstance(score_direction, dict):
            missing_direction_cols = [c for c in self.score_cols if c not in score_direction]
            if missing_direction_cols:
                raise ValueError(f"score_direction 缺少以下分数列的配置: {missing_direction_cols}")
            direction_config = {c: score_direction[c] for c in self.score_cols}
        else:
            direction_config = {c: score_direction for c in self.score_cols}

        self.score_directions = {}
        for sc, direction in direction_config.items():
            if direction not in valid_directions:
                raise ValueError(
                    "score_direction 仅支持 'higher_is_bad'、'higher_is_good'、'auto'，"
                    f"分数列 {sc} 收到: {direction}"
                )
            if direction == 'auto':
                direction, raw_auc = self._infer_score_direction(
                    self.data[self.target_col], self.data[sc]
                )
                print(
                    f"分数方向自动识别: {sc} -> {direction} "
                    f"(原始AUC={raw_auc:.4f})"
                )
            self.score_directions[sc] = direction

        # 主分数方向（向后兼容及便捷访问）
        self.score_direction = self.score_directions[self.score_col]

        # 确保date列为datetime类型
        if date_col in self.data.columns:
            self.data[date_col] = pd.to_datetime(self.data[date_col])
            self.data['Month'] = self.data[date_col].dt.to_period('M')

    @staticmethod
    def _infer_score_direction(y_true: pd.Series, y_score: pd.Series) -> Tuple[str, float]:
        """根据原始AUC推断分数方向；AUC小于0.5表示分数越高风险越低。"""
        from sklearn.metrics import roc_auc_score

        valid_data = pd.DataFrame({'y_true': y_true, 'y_score': y_score}).dropna()
        if valid_data.empty or valid_data['y_true'].nunique() != 2:
            raise ValueError("score_direction='auto' 需要目标列包含两个类别且存在有效分数")

        raw_auc = roc_auc_score(valid_data['y_true'], valid_data['y_score'])
        direction = 'higher_is_good' if raw_auc < 0.5 else 'higher_is_bad'
        return direction, float(raw_auc)

    def _get_risk_score(self, score: Union[pd.Series, np.ndarray], score_col: str):
        """统一转换为风险分：转换后始终是数值越高，坏样本风险越高。"""
        if self.score_directions[score_col] == 'higher_is_good':
            return -score
        return score

    def _calculate_ks(self, y_true: pd.Series, y_score: pd.Series) -> float:
        """计算KS值"""
        try:
            from sklearn.metrics import roc_curve
            fpr, tpr, _ = roc_curve(y_true, y_score)
            ks = max(tpr - fpr)
            return ks
        except:
            return np.nan

    def _calculate_auc(self, y_true: pd.Series, y_score: pd.Series) -> float:
        """计算AUC值"""
        try:
            from sklearn.metrics import roc_auc_score
            return roc_auc_score(y_true, y_score)
        except:
            return np.nan

    def _calculate_psi(self, expected: pd.Series, actual: pd.Series, bins: int = 10) -> float:
        """计算PSI值"""
        try:
            # 基于expected确定分箱边界
            _, bin_edges = pd.cut(expected, bins=bins, retbins=True, duplicates='drop')

            # 对两个分布进行分箱
            expected_binned = pd.cut(expected, bins=bin_edges, include_lowest=True)
            actual_binned = pd.cut(actual, bins=bin_edges, include_lowest=True)

            # 计算每个箱的占比
            expected_pct = expected_binned.value_counts(normalize=True, sort=False)
            actual_pct = actual_binned.value_counts(normalize=True, sort=False)

            # 确保两个分布有相同的索引
            expected_pct = expected_pct.reindex(actual_pct.index, fill_value=0.001)
            actual_pct = actual_pct.fillna(0.001)

            # 计算PSI
            psi = sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
            return psi
        except:
            return np.nan

    def _create_bins(self, data: pd.Series, bins: int = 10, bin_method: str = 'quantile', 
                     retbins: bool = False, handle_missing: bool = True):
        """
        创建分箱的通用函数，支持缺失值单独划为一箱

        Args:
            data: 需要分箱的数据
            bins: 分箱数量
            bin_method: 分箱方式，'quantile'(等频分箱)或'equal'(等距分箱)
            retbins: 是否返回分箱边界
            handle_missing: 是否将缺失值单独划为一箱（默认True）

        Returns:
            分箱结果和边界（如果retbins=True）
        """
        # 检查是否有缺失值
        has_missing = data.isna().any()
        
        if handle_missing and has_missing:
            # 分离缺失值和非缺失值
            non_missing_mask = data.notna()
            non_missing_data = data[non_missing_mask]
            
            # 对非缺失值进行分箱
            if bin_method == 'quantile':
                try:
                    if retbins:
                        binned_non_missing, bin_edges = pd.qcut(non_missing_data, q=bins, retbins=True, duplicates='drop')
                    else:
                        binned_non_missing = pd.qcut(non_missing_data, q=bins, duplicates='drop')
                        bin_edges = None
                except ValueError:
                    # 如果等频分箱失败，回退到等距分箱
                    if retbins:
                        binned_non_missing, bin_edges = pd.cut(non_missing_data, bins=bins, retbins=True, duplicates='drop')
                    else:
                        binned_non_missing = pd.cut(non_missing_data, bins=bins, duplicates='drop')
                        bin_edges = None
            else:  # equal
                if retbins:
                    binned_non_missing, bin_edges = pd.cut(non_missing_data, bins=bins, retbins=True, duplicates='drop')
                else:
                    binned_non_missing = pd.cut(non_missing_data, bins=bins, duplicates='drop')
                    bin_edges = None
            
            # 创建新的分类，包含缺失值类别
            categories = list(binned_non_missing.cat.categories)
            missing_category = pd.Interval(-np.inf, -np.inf)  # 创建一个特殊的区间表示缺失值
            
            # 使用字符串类别代替区间，方便处理缺失值
            str_categories = [str(cat) for cat in categories] + ['Missing']
            
            # 创建结果Series
            result = pd.Series(index=data.index, dtype='object')
            result[non_missing_mask] = binned_non_missing.astype(str)
            result[~non_missing_mask] = 'Missing'
            
            # 转换为Categorical类型
            result = pd.Categorical(result, categories=str_categories, ordered=True)
            result = pd.Series(result, index=data.index)
            
            if retbins:
                return result, bin_edges
            else:
                return result
        else:
            # 原有逻辑（无缺失值或不处理缺失值）
            if bin_method == 'quantile':
                try:
                    if retbins:
                        return pd.qcut(data, q=bins, retbins=True, duplicates='drop')
                    else:
                        return pd.qcut(data, q=bins, duplicates='drop')
                except ValueError:
                    # 如果等频分箱失败，回退到等距分箱
                    if retbins:
                        return pd.cut(data, bins=bins, retbins=True, duplicates='drop')
                    else:
                        return pd.cut(data, bins=bins, duplicates='drop')
            else:  # equal
                if retbins:
                    return pd.cut(data, bins=bins, retbins=True, duplicates='drop')
                else:
                    return pd.cut(data, bins=bins, duplicates='drop')

    def _calculate_iv(self, feature: pd.Series, target: pd.Series, bins: int = 10) -> float:
        """计算IV值"""
        try:
            # 分箱
            feature_binned = pd.cut(feature, bins=bins, duplicates='drop')

            # 计算每个箱的好坏样本数
            crosstab = pd.crosstab(feature_binned, target)

            if crosstab.shape[1] < 2:
                return np.nan

            good = crosstab[0] if 0 in crosstab.columns else 0
            bad = crosstab[1] if 1 in crosstab.columns else 0

            # 避免除零
            good = good.replace(0, 0.001)
            bad = bad.replace(0, 0.001)

            # 计算占比
            good_pct = good / good.sum()
            bad_pct = bad / bad.sum()

            # 计算WOE和IV
            woe = np.log(bad_pct / good_pct)
            iv = sum((bad_pct - good_pct) * woe)

            return iv
        except:
            return np.nan

    def _calculate_feature_importance(self, features: List[str], 
                                      model: Any = None,
                                      importance_type: str = 'gain') -> Dict[str, float]:
        """
        计算特征重要性

        Args:
            features: 特征列名列表
            model: 可选，用于计算特征重要性的模型
                   - 支持LightGBM, XGBoost, CatBoost, sklearn树模型
                   - 如果为None，则使用IV计算
            importance_type: 特征重要性类型
                   - 对于LightGBM: 'gain', 'split'
                   - 对于XGBoost: 'gain', 'weight', 'cover', 'total_gain', 'total_cover'
                   - 对于sklearn树模型: 仅使用feature_importances_

        Returns:
            特征重要性字典 {feature_name: importance_value}
        """
        feature_gains = {}

        if model is not None:
            # 使用模型计算特征重要性
            try:
                model_features = self._get_model_features(model, features)
                
                # 检查模型类型并获取重要性
                model_type = type(model).__name__
                
                if 'LGBMClassifier' in model_type or 'LGBMRegressor' in model_type or 'Booster' in model_type:
                    # LightGBM模型
                    if hasattr(model, 'booster_'):
                        # sklearn接口
                        importance = model.booster_.feature_importance(importance_type=importance_type)
                        feature_names_model = model.booster_.feature_name()
                    elif hasattr(model, 'feature_importance'):
                        # 原生Booster
                        importance = model.feature_importance(importance_type=importance_type)
                        feature_names_model = model.feature_name()
                    else:
                        # 使用feature_importances_
                        importance = model.feature_importances_
                        feature_names_model = model_features
                        
                elif 'XGBClassifier' in model_type or 'XGBRegressor' in model_type or 'Booster' in model_type:
                    # XGBoost模型
                    if hasattr(model, 'get_booster'):
                        booster = model.get_booster()
                        importance_dict = booster.get_score(importance_type=importance_type)
                        for feat in features:
                            if feat in self.data.columns:
                                feature_gains[feat] = importance_dict.get(feat, 0)
                        return feature_gains
                    elif hasattr(model, 'feature_importances_'):
                        importance = model.feature_importances_
                        feature_names_model = model_features
                    else:
                        raise ValueError(f"无法从XGBoost模型获取特征重要性")
                        
                elif 'CatBoost' in model_type:
                    # CatBoost模型
                    importance = model.get_feature_importance()
                    feature_names_model = model.feature_names_ if hasattr(model, 'feature_names_') else model_features
                    
                elif hasattr(model, 'feature_importances_'):
                    # sklearn树模型 (RandomForest, GradientBoosting, etc.)
                    importance = model.feature_importances_
                    feature_names_model = model_features
                    
                else:
                    raise ValueError(f"不支持的模型类型: {model_type}")

                # 构建特征重要性字典
                importance_dict = dict(zip(feature_names_model, importance))
                for feat in features:
                    if feat in self.data.columns:
                        feature_gains[feat] = importance_dict.get(feat, 0)
                        
            except Exception as e:
                warnings.warn(f"使用模型计算特征重要性失败: {e}，回退到IV计算")
                model = None

        if model is None:
            # 使用IV计算特征重要性
            for feature in features:
                if feature not in self.data.columns:
                    continue
                try:
                    iv = self._calculate_iv(self.data[feature], self.data[self.target_col])
                    feature_gains[feature] = iv if not pd.isna(iv) else 0
                except:
                    feature_gains[feature] = 0

        return feature_gains

    def _get_model_features(self, model: Any, default_features: List[str]) -> List[str]:
        """获取模型使用的特征名称"""
        if hasattr(model, 'feature_name_'):
            return model.feature_name_
        elif hasattr(model, 'feature_names_'):
            return model.feature_names_
        elif hasattr(model, 'feature_names_in_'):
            return list(model.feature_names_in_)
        elif hasattr(model, 'booster_') and hasattr(model.booster_, 'feature_name'):
            return model.booster_.feature_name()
        elif hasattr(model, 'get_booster'):
            return model.get_booster().feature_names
        else:
            return default_features

    def _apply_alternating_fill(self, ws, df: pd.DataFrame, blue_fill, white_fill):
        """
        对工作表应用蓝白交替填充，根据分组列区分不同类别
        
        Args:
            ws: openpyxl worksheet对象
            df: 数据框
            blue_fill: 蓝色填充样式
            white_fill: 白色填充样式
        """
        # 定义需要进行分组填充的列（按优先级排序）
        group_columns = ['样本类型', '特征', '指标英文','指标中文']
        
        # 找到第一个存在的分组列
        group_col = None
        group_col_idx = None
        for col in group_columns:
            if col in df.columns:
                group_col = col
                group_col_idx = df.columns.get_loc(col) + 1  # openpyxl列索引从1开始
                break
        
        if group_col is None:
            return  # 没有需要分组的列
        
        # 获取该列的值，追踪分组变化
        col_values = df[group_col].tolist()
        
        # 计算每行应该使用的填充颜色
        current_fill = blue_fill
        prev_value = None
        row_fills = []
        
        for value in col_values:
            if prev_value is not None and value != prev_value:
                # 值发生变化，切换颜色
                current_fill = white_fill if current_fill == blue_fill else blue_fill
            row_fills.append(current_fill)
            prev_value = value
        
        # 应用填充颜色到所有单元格（跳过表头，从第2行开始）
        max_col = len(df.columns)
        for row_idx, fill in enumerate(row_fills, start=2):  # Excel行从1开始，表头占第1行
            for col_idx in range(1, max_col + 1):
                ws.cell(row=row_idx, column=col_idx).fill = fill

    def _convert_numeric_strings(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        将DataFrame中的数字字符串转换为真正的数字类型
        
        Args:
            df: 输入的数据框
            
        Returns:
            转换后的数据框
        """
        df_copy = df.copy()
        
        for col in df_copy.columns:
            # 跳过明显不是数字的列
            if col in ['分箱', 'score_bin', '概率范围', 'Month']:
                continue
                
            # 尝试将列转换为数字
            try:
                # 先检查是否为字符串类型的列
                if df_copy[col].dtype == 'object':
                    # 尝试转换为数字，无法转换的保持原样
                    converted = pd.to_numeric(df_copy[col], errors='coerce')
                    # 只有当大部分值都能成功转换时才替换
                    if converted.notna().sum() > 0:
                        # 保留原始的非数字值（如 "N/A"）
                        mask = converted.notna()
                        df_copy.loc[mask, col] = converted[mask]
            except Exception:
                pass
                
        return df_copy

    def generate_sample_summary(self) -> pd.DataFrame:
        """
        生成样本情况统计表

        Returns:
            样本情况汇总表
        """
        if 'Month' not in self.data.columns:
            raise ValueError("需要日期列来生成按月统计")

        # 按月统计
        monthly_stats = []
        for month in sorted(self.data['Month'].unique()):
            month_data = self.data[self.data['Month'] == month]
            good = (month_data[self.target_col] == 0).sum()
            bad = (month_data[self.target_col] == 1).sum()
            total = len(month_data)
            bad_rate = bad / total if total > 0 else 0

            monthly_stats.append({
                'Month': str(month),
                'Good': good,
                'Bad': bad,
                'Total': total,
                'Bad_Rate': f"{bad_rate:.4f}"
            })

        # 总计行
        total_good = (self.data[self.target_col] == 0).sum()
        total_bad = (self.data[self.target_col] == 1).sum()
        total_total = len(self.data)
        total_bad_rate = total_bad / total_total if total_total > 0 else 0

        monthly_stats.append({
            'Month': '合计',
            'Good': total_good,
            'Bad': total_bad,
            'Total': total_total,
            'Bad_Rate': f"{total_bad_rate:.4f}"
        })

        return pd.DataFrame(monthly_stats)

    def generate_model_performance(self, base_type: str = 'all', n_bins: int = 10) -> pd.DataFrame:
        """
        生成模型效果统计表

        单分数模式下，输出列为 KS / AUC / TOP{100/n_bins}_lift / PSI；
        多分数模式下，每个分数对应一组指标列，列名带分数后缀，
        如 n_bins=20 时输出 KS_score1 / AUC_score1 / TOP5_lift_score1 / PSI_score1。

        Args:
            base_type: PSI计算的基准样本类型
            n_bins: Lift分箱数量，例如20分箱对应TOP5_lift

        Returns:
            模型效果汇总表
        """
        if not isinstance(n_bins, int) or isinstance(n_bins, bool) or n_bins <= 0:
            raise ValueError(f"n_bins 必须为正整数，收到: {n_bins}")

        top_pct_label = f"{100 / n_bins:g}"
        top_lift_column = f"TOP{top_pct_label}_lift"
        performance_stats = []

        # separate 策略：每个分数各自用其非空样本计算表现，并额外给出缺失率列
        is_separate = self.is_multi_score and self.score_na_strategy == 'separate'

        # 获取各分数的基准数据用于PSI计算
        if base_type in self.data[self.sample_type_col].values:
            base_data = self.data[self.data[self.sample_type_col] == base_type]
            # separate 模式下基准分布也仅基于该分数非空样本
            if is_separate:
                base_scores_map = {sc: base_data[sc].dropna() for sc in self.score_cols}
            else:
                base_scores_map = {sc: base_data[sc] for sc in self.score_cols}
        else:
            base_scores_map = {sc: None for sc in self.score_cols}

        # 按样本类型统计
        sample_types = ['all'] + [t for t in self.data[self.sample_type_col].unique() if t != 'all']

        for sample_type in sample_types:
            if sample_type == 'all':
                sample_data = self.data
            else:
                sample_data = self.data[self.data[self.sample_type_col] == sample_type]

            if len(sample_data) == 0:
                continue

            good = (sample_data[self.target_col] == 0).sum()
            bad = (sample_data[self.target_col] == 1).sum()
            total = len(sample_data)
            bad_rate = bad / total if total > 0 else 0

            row = {
                '样本类型': sample_type,
                'Good': good,
                'Bad': bad,
                'Total': total,
                'Bad_Rate': f"{bad_rate:.4f}",
            }

            # 对每个分数列分别计算指标
            for sc in self.score_cols:
                # 仅多分数模式添加后缀，单分数保持原列名
                suffix = f"_{sc}" if self.is_multi_score else ""

                if is_separate:
                    # separate 模式：仅用该分数非空样本计算指标，并记录缺失率
                    score_mask = sample_data[sc].notna()
                    score_data = sample_data[score_mask]
                    miss_rate = 1 - len(score_data) / total if total > 0 else 0
                    row[f'缺失率{suffix}'] = f"{miss_rate:.4f}"
                else:
                    score_data = sample_data

                # 该分数在当前样本类型下无有效样本（如全部缺失），指标置 N/A，避免空数组分箱报错
                if len(score_data) == 0:
                    row[f'KS{suffix}'] = "N/A"
                    row[f'AUC{suffix}'] = "N/A"
                    row[f'{top_lift_column}{suffix}'] = "N/A"
                    row[f'PSI{suffix}'] = "N/A"
                    continue

                # 统一转换为“数值越高风险越高”，避免高分代表好时指标方向倒置
                risk_score = self._get_risk_score(score_data[sc], sc)
                ks = self._calculate_ks(score_data[self.target_col], risk_score)
                auc = self._calculate_auc(score_data[self.target_col], risk_score)

                base_scores = base_scores_map.get(sc)
                if base_scores is not None and sample_type != base_type:
                    psi = self._calculate_psi(base_scores, score_data[sc])
                else:
                    psi = 0.0

                df_lift, baseline_rate = self._calculate_lift_for_plot(
                    score_data[self.target_col], risk_score, n_bins=n_bins,
                    display_score=score_data[sc]
                )

                row[f'KS{suffix}'] = f"{ks:.4f}" if not pd.isna(ks) else "N/A"
                row[f'AUC{suffix}'] = f"{auc:.4f}" if not pd.isna(auc) else "N/A"
                row[f'{top_lift_column}{suffix}'] = f"{df_lift.iloc[0]['Lift']:.4f}" if len(df_lift) > 0 else "N/A"
                row[f'PSI{suffix}'] = f"{psi:.4f}" if not pd.isna(psi) else "N/A"

            performance_stats.append(row)

        return pd.DataFrame(performance_stats)

    def generate_metric_report_plot(self, n_bins: int = 10, save_dir: str = None) -> Dict[str, Any]:
        """
        生成模型评估指标图表（类似metric_report_plot）

        Args:
            n_bins: 分箱数量
            save_dir: 图片保存目录（临时目录）

        Returns:
            包含图片路径和汇总数据的字典
        """
        import matplotlib.pyplot as plt
        from sklearn.metrics import roc_curve, auc, roc_auc_score
        import tempfile
        import os

        if not isinstance(n_bins, int) or isinstance(n_bins, bool) or n_bins <= 0:
            raise ValueError(f"n_bins 必须为正整数，收到: {n_bins}")
        top_pct_label = f"{100 / n_bins:g}"

        # 设置中文字体支持
        plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "STHeiti"]
        plt.rcParams["axes.unicode_minus"] = False

        sample_types = [t for t in self.data[self.sample_type_col].unique() if t != 'all']

        if save_dir is None:
            save_dir = tempfile.mkdtemp()

        result = {
            'images': {},
            'metrics_summary': [],
            'lift_details': []
        }

        for sc in self.score_cols:
            score_suffix = f" - {sc}" if self.is_multi_score else ""
            for sample_type in sample_types:
                sample_data = self.data[self.data[self.sample_type_col] == sample_type]

                if len(sample_data) == 0:
                    continue

                y_true = sample_data[self.target_col].values
                y_score = sample_data[sc].values
                risk_score = self._get_risk_score(y_score, sc)

                # KS、Lift和ROC统一使用“越高越坏”的风险方向
                ks_value, df_ks, ks_index = self._calculate_ks_for_plot(y_true, risk_score)
                df_lift, baseline_rate = self._calculate_lift_for_plot(
                    y_true, risk_score, n_bins, display_score=y_score
                )

                # 计算AUC
                try:
                    fpr, tpr, _ = roc_curve(y_true, risk_score)
                    roc_auc = auc(fpr, tpr)
                except:
                    fpr, tpr = [0, 1], [0, 1]
                    roc_auc = np.nan

                # 创建综合图表
                fig, axes = plt.subplots(2, 3, figsize=(20, 12))
                fig.suptitle(f'{sample_type}{score_suffix} 模型性能综合分析 (KS={ks_value:.3f})', fontsize=16, weight='bold')

                # 1. KS曲线
                x_axis = np.arange(len(df_ks)) / len(df_ks)
                axes[0, 0].plot(x_axis, df_ks['cum_good_rate'], 'b-', label='累积好样本率')
                axes[0, 0].plot(x_axis, df_ks['cum_bad_rate'], 'r-', label='累积坏样本率')
                axes[0, 0].fill_between(x_axis, df_ks['cum_good_rate'], df_ks['cum_bad_rate'],
                                        alpha=0.3, color='green')
                max_ks_x = ks_index / len(df_ks)
                axes[0, 0].axvline(x=max_ks_x, color='orange', linestyle='--', alpha=0.8)
                axes[0, 0].set_title(f'KS曲线 (KS={ks_value:.3f})')
                axes[0, 0].legend()
                axes[0, 0].grid(True, alpha=0.3)

                # 2. Lift曲线（从最高风险样本开始累计）
                axes[0, 1].plot(df_lift['累积样本占比'], df_lift['累积Lift'],
                                'b-o', linewidth=2, markersize=6)
                axes[0, 1].axhline(y=1, color='red', linestyle='--', alpha=0.7)
                axes[0, 1].set_xlabel('累积样本占比')
                axes[0, 1].set_ylabel('累积Lift')
                axes[0, 1].set_title('累积Lift曲线')
                axes[0, 1].grid(True, alpha=0.3)

                # 3. ROC曲线
                axes[0, 2].plot(fpr, tpr, 'b-', label=f'ROC (AUC={roc_auc:.3f})')
                axes[0, 2].plot([0, 1], [0, 1], 'k--', alpha=0.5)
                axes[0, 2].set_xlabel('假正率')
                axes[0, 2].set_ylabel('真正率')
                axes[0, 2].set_title('ROC曲线')
                axes[0, 2].legend()
                axes[0, 2].grid(True, alpha=0.3)

                # 4. 分箱Lift
                axes[1, 0].bar(df_lift['分箱'], df_lift['Lift'],
                               color='skyblue', alpha=0.7, edgecolor='navy')
                axes[1, 0].axhline(y=1, color='red', linestyle='--', alpha=0.7)
                axes[1, 0].set_xlabel('分箱')
                axes[1, 0].set_ylabel('Lift值')
                axes[1, 0].set_title('各分箱Lift值')
                axes[1, 0].grid(True, alpha=0.3)

                # 5. 原始分数分布（保留业务分数刻度）
                axes[1, 1].hist(y_score[y_true == 0], bins=30, alpha=0.6, label='负样本',
                                color='blue', density=True)
                axes[1, 1].hist(y_score[y_true == 1], bins=30, alpha=0.6, label='正样本',
                                color='red', density=True)
                axes[1, 1].set_xlabel('模型分数')
                axes[1, 1].set_ylabel('密度')
                axes[1, 1].set_title('样本分数分布')
                axes[1, 1].legend()
                axes[1, 1].grid(True, alpha=0.3)

                # 6. 增益图（横轴为累计样本占比，纵轴为累计坏样本召回率）
                sample_pct = df_lift['累积样本占比']
                recall = df_lift['累积召回率']
                axes[1, 2].plot(sample_pct, recall,
                                'b-o', linewidth=2, label='模型增益')
                axes[1, 2].plot([0, 1], [0, 1], 'r--', alpha=0.7, label='随机模型')
                axes[1, 2].fill_between(sample_pct, sample_pct,
                                        recall, alpha=0.3, color='green')
                axes[1, 2].set_xlabel('样本比例')
                axes[1, 2].set_ylabel('捕获正样本比例')
                axes[1, 2].set_title('增益图')
                axes[1, 2].legend()
                axes[1, 2].grid(True, alpha=0.3)

                plt.tight_layout()

                # 保存图片（多分数模式下文件名和key包含分数名）
                image_key = f'{sample_type}_{sc}' if self.is_multi_score else sample_type
                img_path = os.path.join(save_dir, f'metric_report_{image_key}.png')
                fig.savefig(img_path, dpi=150, bbox_inches='tight')
                plt.close(fig)

                result['images'][image_key] = img_path

                # 保存汇总指标
                metrics_row = {}
                if self.is_multi_score:
                    metrics_row['分数'] = sc
                metrics_row.update({
                    '样本类型': sample_type,
                    '样本量': len(sample_data),
                    '正样本数': int(y_true.sum()),
                    '正样本率': f"{baseline_rate:.4f}",
                    'KS': f"{ks_value:.4f}" if not pd.isna(ks_value) else "N/A",
                    'AUC': f"{roc_auc:.4f}" if not pd.isna(roc_auc) else "N/A",
                    f'Top{top_pct_label}%_Lift': f"{df_lift.iloc[0]['Lift']:.4f}" if len(df_lift) > 0 else "N/A"
                })
                result['metrics_summary'].append(metrics_row)

                # 保存Lift详情
                df_lift_copy = df_lift.copy()
                if self.is_multi_score:
                    df_lift_copy['分数'] = sc
                df_lift_copy['样本类型'] = sample_type
                result['lift_details'].append(df_lift_copy)

        return result

    def _calculate_ks_for_plot(self, y_true: np.ndarray, y_prob: np.ndarray) -> Tuple[float, pd.DataFrame, int]:
        """计算KS统计量用于绘图"""
        df = pd.DataFrame({'y_true': y_true, 'y_prob': y_prob})
        df = df.sort_values('y_prob', ascending=False).reset_index(drop=True)

        df['bad'] = df['y_true']
        df['good'] = 1 - df['y_true']

        total_good = df['good'].sum()
        total_bad = df['bad'].sum()

        if total_good == 0 or total_bad == 0:
            return np.nan, df, 0

        df['cum_good'] = df['good'].cumsum()
        df['cum_bad'] = df['bad'].cumsum()

        df['cum_good_rate'] = df['cum_good'] / total_good
        df['cum_bad_rate'] = df['cum_bad'] / total_bad

        df['ks'] = df['cum_bad_rate'] - df['cum_good_rate']

        ks_value = df['ks'].max()
        ks_index = df['ks'].idxmax()

        return ks_value, df, ks_index

    def _calculate_lift_for_plot(self, y_true: np.ndarray, y_prob: np.ndarray,
                                  n_bins: int = 10,
                                  display_score: Optional[np.ndarray] = None) -> Tuple[pd.DataFrame, float]:
        """
        计算Lift统计表用于绘图。

        y_prob 必须是已统一方向的风险分（数值越高风险越高）；display_score
        用于在分箱范围中保留原始业务分数刻度。
        """
        if display_score is None:
            display_score = y_prob

        df = pd.DataFrame({
            'y_true': y_true,
            'y_prob': y_prob,
            'display_score': display_score,
        })
        df = df.dropna(subset=['y_prob', 'display_score'])

        # 无有效样本时直接返回空表，避免空数组分箱报错
        if len(df) == 0:
            return pd.DataFrame(), 0.0

        total_samples = len(df)
        total_positive = df['y_true'].sum()
        baseline_rate = total_positive / total_samples if total_samples > 0 else 0

        # 按风险分分箱：标签越大表示风险越高
        try:
            df['risk_bin'] = pd.qcut(
                df['y_prob'], q=n_bins, labels=False, duplicates='drop'
            ) + 1
        except Exception:
            df['risk_bin'] = pd.cut(
                df['y_prob'], bins=n_bins, labels=False, duplicates='drop'
            ) + 1

        # 从最高风险箱开始排列和累计，保证Lift/增益曲线方向正确
        lift_summary = []
        risk_bins = sorted(df['risk_bin'].dropna().unique(), reverse=True)
        for display_bin, risk_bin in enumerate(risk_bins, start=1):
            bin_data = df[df['risk_bin'] == risk_bin]

            bin_stats = {
                '分箱': display_bin,
                '样本数': len(bin_data),
                '正样本数': bin_data['y_true'].sum(),
                '负样本数': len(bin_data) - bin_data['y_true'].sum(),
                '正样本率': bin_data['y_true'].mean(),
                'Lift': bin_data['y_true'].mean() / baseline_rate if baseline_rate > 0 else 0,
                '概率范围': (
                    f"{bin_data['display_score'].min():.3f}-"
                    f"{bin_data['display_score'].max():.3f}"
                ),
            }
            lift_summary.append(bin_stats)

        df_lift = pd.DataFrame(lift_summary)

        # 累计统计均从最高风险样本开始
        df_lift['累积样本数'] = df_lift['样本数'].cumsum()
        df_lift['累积样本占比'] = df_lift['累积样本数'] / total_samples
        df_lift['累积正样本数'] = df_lift['正样本数'].cumsum()
        df_lift['累积正样本率'] = df_lift['累积正样本数'] / df_lift['累积样本数']
        df_lift['累积Lift'] = df_lift['累积正样本率'] / baseline_rate if baseline_rate > 0 else 0
        df_lift['累积召回率'] = df_lift['累积正样本数'] / total_positive if total_positive > 0 else 0

        return df_lift, baseline_rate

    def generate_score_bins_analysis(self, bins: int = 10, bin_method: str = 'quantile') -> pd.DataFrame:
        """
        生成模型效果分箱分析

        Args:
            bins: 分箱数量
            bin_method: 分箱方式，'quantile'(等频分箱)或'equal'(等距分箱)

        Returns:
            分箱分析结果
        """
        bin_analysis = []

        sample_types = ['all'] + [t for t in self.data[self.sample_type_col].unique() if t != 'all']

        # 基于all样本确定分箱边界
        if 'all' in self.data[self.sample_type_col].unique():
            all_data = self.data
        else:
            all_data = self.data

        # 对每个分数列分别进行分箱分析
        for sc in self.score_cols:
            # 根据分箱方式选择分箱函数
            _, bin_edges = self._create_bins(all_data[sc], bins=bins, bin_method=bin_method, retbins=True)

            for sample_type in sample_types:
                if sample_type == 'all':
                    sample_data = self.data
                else:
                    sample_data = self.data[self.data[self.sample_type_col] == sample_type]

                if len(sample_data) == 0:
                    continue

                # 分箱
                sample_data_copy = sample_data.copy()
                sample_data_copy['score_bin'] = pd.cut(sample_data_copy[sc],
                                                      bins=bin_edges, include_lowest=True)

                # 按分箱统计
                bin_stats = sample_data_copy.groupby('score_bin').agg({
                    self.target_col: ['count', 'sum', 'mean']
                }).round(4)

                bin_stats.columns = ['total_users', 'bad_users', 'bad_rate']
                bin_stats['good_users'] = bin_stats['total_users'] - bin_stats['bad_users']
                bin_stats['total_pct'] = bin_stats['total_users'] / len(sample_data)
                bin_stats['bad_pct'] = bin_stats['bad_users'] / bin_stats['bad_users'].sum()

                # 从高风险箱开始累计：概率分降序，信用评分升序
                high_score_is_good = self.score_directions[sc] == 'higher_is_good'
                bin_stats = bin_stats.sort_index(ascending=high_score_is_good)
                bin_stats['cum_bad_rate'] = (bin_stats['bad_users'].cumsum() /
                                           bin_stats['total_users'].cumsum())
                bin_stats['cum_bad_pct'] = bin_stats['bad_users'].cumsum() / bin_stats['bad_users'].sum()

                # 计算KS和LIFT
                bin_stats['ks'] = np.abs(bin_stats['cum_bad_pct'] -
                                       (bin_stats['total_users'].cumsum() / bin_stats['total_users'].sum()))
                bin_stats['lift'] = bin_stats['bad_rate'] / (bin_stats['bad_users'].sum() / bin_stats['total_users'].sum())

                # PSI计算（与all样本对比）
                if sample_type != 'all':
                    all_sample_data = self.data.copy()
                    all_sample_data['score_bin'] = pd.cut(all_sample_data[sc],
                                                        bins=bin_edges, include_lowest=True)
                    all_bin_pct = all_sample_data.groupby('score_bin').size() / len(all_sample_data)
                    current_bin_pct = bin_stats['total_pct']

                    # 确保索引一致
                    all_bin_pct = all_bin_pct.reindex(current_bin_pct.index, fill_value=0.001)
                    current_bin_pct = current_bin_pct.fillna(0.001)

                    psi_values = (current_bin_pct - all_bin_pct) * np.log(current_bin_pct / all_bin_pct)
                else:
                    psi_values = pd.Series([0] * len(bin_stats), index=bin_stats.index)

                bin_stats['psi'] = psi_values

                # 重新排序列
                bin_stats = bin_stats[['good_users', 'bad_users', 'total_users', 'total_pct',
                                     'bad_pct', 'bad_rate', 'ks', 'lift', 'cum_bad_rate',
                                     'cum_bad_pct', 'psi']]

                # 添加样本类型信息
                for idx, row in bin_stats.iterrows():
                    record = {}
                    if self.is_multi_score:
                        record['分数'] = sc
                    record.update({
                        '样本类型': sample_type,
                        '分箱': str(idx),
                        '好用户': int(row['good_users']),
                        '坏用户': int(row['bad_users']),
                        '总用户': int(row['total_users']),
                        '总用户占比': f"{row['total_pct']:.4f}",
                        '坏用户占比': f"{row['bad_pct']:.4f}",
                        '坏用户率': f"{row['bad_rate']:.4f}",
                        'KS': f"{row['ks']:.4f}",
                        'LIFT': f"{row['lift']:.4f}",
                        '累计坏用户率': f"{row['cum_bad_rate']:.4f}",
                        '累计坏用户占比': f"{row['cum_bad_pct']:.4f}",
                        'PSI稳定性': f"{row['psi']:.4f}"
                    })
                    bin_analysis.append(record)

                # 添加总计行
                total_good = bin_stats['good_users'].sum()
                total_bad = bin_stats['bad_users'].sum()
                total_total = bin_stats['total_users'].sum()

                total_record = {}
                if self.is_multi_score:
                    total_record['分数'] = sc
                total_record.update({
                    '样本类型': sample_type,
                    '分箱': '总计',
                    '好用户': int(total_good),
                    '坏用户': int(total_bad),
                    '总用户': int(total_total),
                    '总用户占比': "1.0000",
                    '坏用户占比': "1.0000",
                    '坏用户率': f"{total_bad/total_total:.4f}" if total_total > 0 else "0.0000",
                    'KS': f"{bin_stats['ks'].max():.4f}",
                    'LIFT': "N/A",
                    '累计坏用户率': f"{total_bad/total_total:.4f}" if total_total > 0 else "0.0000",
                    '累计坏用户占比': "1.0000",
                    'PSI稳定性': f"{bin_stats['psi'].sum():.4f}"
                })
                bin_analysis.append(total_record)

        return pd.DataFrame(bin_analysis)

    def generate_top_features_effectiveness(self, features: List[str],
                                          feature_names: Dict[str, str] = None,
                                          top_n: int = 10,
                                          bins: int = 5,
                                          bin_method: str = 'quantile',
                                          model: Any = None,
                                          importance_type: str = 'gain') -> pd.DataFrame:
        """
        生成TOP10变量有效性分析 - 分箱级别统计

        Args:
            features: 特征列名列表
            feature_names: 特征中文名映射
            top_n: 返回前N个特征
            bins: 分箱数量
            bin_method: 分箱方式，'quantile'(等频分箱)或'equal'(等距分箱)
            model: 可选，用于计算特征重要性的模型（支持LightGBM/XGBoost/sklearn模型）
            importance_type: 特征重要性类型，'gain'/'split'/'weight'（仅对树模型有效）

        Returns:
            变量有效性分析结果 - 每行一个分箱
        """
        if feature_names is None:
            feature_names = {f: f for f in features}

        effectiveness_results = []

        # 计算每个特征的总体有效性(gain)
        feature_gains = self._calculate_feature_importance(
            features, model=model, importance_type=importance_type
        )

        # 选择TOP N特征
        top_features = sorted(feature_gains.items(), key=lambda x: x[1], reverse=True)[:top_n]

        for feature, total_gain in top_features:
            if feature not in self.data.columns:
                continue

            try:
                # 对特征进行分箱
                self.data[f'{feature}_bin'] = self._create_bins(self.data[feature], bins=bins, bin_method=bin_method)
                bin_labels = self.data[f'{feature}_bin'].cat.categories

                # 按分箱统计
                for bin_label in bin_labels:
                    bin_data = self.data[self.data[f'{feature}_bin'] == bin_label]
                    if len(bin_data) == 0:
                        continue

                    bin_result = {
                        '指标英文': feature,
                        '指标中文': feature_names.get(feature, feature),
                        '数据类型': str(self.data[feature].dtype),
                        '分箱': str(bin_label),
                        'total_gain': f"{total_gain:.4f}"
                    }

                    # 动态获取样本类型（排除'all'）
                    sample_types = [t for t in self.data[self.sample_type_col].unique() if t != 'all']
                    for sample_type in sample_types:
                        sample_bin_data = bin_data[bin_data[self.sample_type_col] == sample_type]

                        if len(sample_bin_data) == 0:
                            bin_result[f'N_{sample_type}'] = 0
                            bin_result[f'分布占比_{sample_type}'] = "0.0000"
                            bin_result[f'坏样本数量_{sample_type}'] = 0
                            bin_result[f'逾期率_{sample_type}'] = "0.0000"
                            continue

                        n_sample = len(sample_bin_data)
                        bad_count = (sample_bin_data[self.target_col] == 1).sum()
                        bad_rate = bad_count / n_sample if n_sample > 0 else 0

                        # 计算在该样本类型中的分布占比
                        total_sample_type = len(self.data[self.data[self.sample_type_col] == sample_type])
                        distribution_ratio = n_sample / total_sample_type if total_sample_type > 0 else 0

                        bin_result[f'N_{sample_type}'] = n_sample
                        bin_result[f'分布占比_{sample_type}'] = f"{distribution_ratio:.4f}"
                        bin_result[f'坏样本数量_{sample_type}'] = bad_count
                        bin_result[f'逾期率_{sample_type}'] = f"{bad_rate:.4f}"

                    effectiveness_results.append(bin_result)

            except Exception as e:
                print(f"特征 {feature} 分箱失败: {e}")
                continue

        return pd.DataFrame(effectiveness_results)

    def generate_top_features_effectiveness_monthly(self, features: List[str],
                                                   feature_names: Dict[str, str] = None,
                                                   sample_type: str = 'test',
                                                   top_n: int = 10,
                                                   bins: int = 5,
                                                   bin_method: str = 'quantile',
                                                   model: Any = None,
                                                   importance_type: str = 'gain') -> pd.DataFrame:
        """
        生成按月拆分的变量有效性分析 - 分箱级别按月统计

        Args:
            features: 特征列名列表
            feature_names: 特征中文名映射
            sample_type: 分析的样本类型
            top_n: 返回前N个特征
            bins: 分箱数量
            bin_method: 分箱方式，'quantile'(等频分箱)或'equal'(等距分箱)
            model: 可选，用于计算特征重要性的模型
            importance_type: 特征重要性类型

        Returns:
            按月拆分的变量有效性分析结果 - 每行一个分箱
        """
        if 'Month' not in self.data.columns:
            raise ValueError("需要日期列来生成按月统计")

        if feature_names is None:
            feature_names = {f: f for f in features}

        # 筛选指定样本类型的数据
        sample_data = self.data[self.data[self.sample_type_col] == sample_type]
        if len(sample_data) == 0:
            return pd.DataFrame()

        # 获取月份列表
        months = sorted(sample_data['Month'].unique())
        month_strs = [str(m) for m in months]

        # 计算特征重要性并选择TOP N
        feature_gains = self._calculate_feature_importance(
            features, model=model, importance_type=importance_type
        )

        top_features = sorted(feature_gains.items(), key=lambda x: x[1], reverse=True)[:top_n]

        monthly_results = []

        for feature, total_gain in top_features:
            if feature not in sample_data.columns:
                continue

            try:
                # 对特征进行分箱（基于全量该类型样本）
                sample_data_copy = sample_data.copy()
                sample_data_copy[f'{feature}_bin'] = self._create_bins(sample_data_copy[feature], bins=bins, bin_method=bin_method)
                bin_labels = sample_data_copy[f'{feature}_bin'].cat.categories

                # 按分箱统计
                for bin_label in bin_labels:
                    bin_data = sample_data_copy[sample_data_copy[f'{feature}_bin'] == bin_label]
                    if len(bin_data) == 0:
                        continue

                    bin_result = {
                        '指标英文': feature,
                        '指标中文': feature_names.get(feature, feature),
                        '数据类型': str(sample_data[feature].dtype),
                        '分箱': str(bin_label),
                        'total_gain': f"{total_gain:.4f}"
                    }

                    # 按月统计每个分箱
                    for month in months:
                        month_bin_data = bin_data[bin_data['Month'] == month]
                        month_str = str(month)

                        if len(month_bin_data) == 0:
                            bin_result[f'N_{sample_type}_{month_str}'] = 0
                            bin_result[f'分布占比_{sample_type}_{month_str}'] = "0.0000"
                            bin_result[f'坏样本数量_{sample_type}_{month_str}'] = 0
                            bin_result[f'逾期率_{sample_type}_{month_str}'] = "0.0000"
                            continue

                        n_month = len(month_bin_data)
                        bad_count = (month_bin_data[self.target_col] == 1).sum()
                        bad_rate = bad_count / n_month if n_month > 0 else 0

                        # 计算在该月该样本类型中的分布占比
                        total_month_sample = len(sample_data_copy[sample_data_copy['Month'] == month])
                        distribution_ratio = n_month / total_month_sample if total_month_sample > 0 else 0

                        bin_result[f'N_{sample_type}_{month_str}'] = n_month
                        bin_result[f'分布占比_{sample_type}_{month_str}'] = f"{distribution_ratio:.4f}"
                        bin_result[f'坏样本数量_{sample_type}_{month_str}'] = bad_count
                        bin_result[f'逾期率_{sample_type}_{month_str}'] = f"{bad_rate:.4f}"

                    monthly_results.append(bin_result)

            except Exception as e:
                print(f"特征 {feature} 按月分箱失败: {e}")
                continue

        return pd.DataFrame(monthly_results)

    def generate_top_features_stability(self, features: List[str],
                                       feature_names: Dict[str, str] = None,
                                       top_n: int = 10,
                                       bins: int = 5,
                                       bin_method: str = 'quantile',
                                       model: Any = None,
                                       importance_type: str = 'gain') -> pd.DataFrame:
        """
        生成TOP10变量稳定性分析 - 分箱级别统计

        Args:
            features: 特征列名列表
            feature_names: 特征中文名映射
            top_n: 返回前N个特征
            bins: 分箱数量
            bin_method: 分箱方式，'quantile'(等频分箱)或'equal'(等距分箱)
            model: 可选，用于计算特征重要性的模型
            importance_type: 特征重要性类型

        Returns:
            变量稳定性分析结果 - 每行一个分箱
        """
        if feature_names is None:
            feature_names = {f: f for f in features}

        # 计算特征重要性并选择TOP N
        feature_gains = self._calculate_feature_importance(
            features, model=model, importance_type=importance_type
        )

        top_features = sorted(feature_gains.items(), key=lambda x: x[1], reverse=True)[:top_n]

        stability_results = []

        # 动态获取样本类型并选择第一个作为基准
        sample_types_all = [t for t in self.data[self.sample_type_col].unique() if t != 'all']
        base_sample_type_global = sample_types_all[0] if sample_types_all else 'train'
        base_data = self.data[self.data[self.sample_type_col] == base_sample_type_global]

        for feature, total_gain in top_features:
            if feature not in self.data.columns:
                continue

            try:
                # 基于基准数据确定分箱边界，确保一致性
                if len(base_data) > 0:
                    _, bin_edges = pd.cut(base_data[feature], bins=bins, retbins=True, duplicates='drop')
                else:
                    _, bin_edges = pd.cut(self.data[feature], bins=bins, retbins=True, duplicates='drop')

                # 对所有数据应用相同分箱
                self.data[f'{feature}_bin'] = pd.cut(self.data[feature], bins=bin_edges, include_lowest=True)
                bin_labels = self.data[f'{feature}_bin'].cat.categories

                # 按分箱统计
                for bin_label in bin_labels:
                    bin_data = self.data[self.data[f'{feature}_bin'] == bin_label]
                    if len(bin_data) == 0:
                        continue

                    bin_result = {
                        '指标英文': feature,
                        '指标中文': feature_names.get(feature, feature),
                        '数据类型': str(self.data[feature].dtype),
                        '分箱': str(bin_label),
                        'total_gain': f"{total_gain:.4f}"
                    }

                    # 动态获取样本类型（排除'all'）
                    sample_types = [t for t in self.data[self.sample_type_col].unique() if t != 'all']
                    
                    # 获取第一个样本类型作为基准（通常是train）
                    base_sample_type = sample_types[0] if sample_types else 'train'

                    # 先计算基准样本分箱的分布
                    base_bin_data = bin_data[bin_data[self.sample_type_col] == base_sample_type]
                    base_total = len(self.data[self.data[self.sample_type_col] == base_sample_type])
                    base_bin_ratio = len(base_bin_data) / base_total if base_total > 0 else 0

                    for sample_type in sample_types:
                        sample_bin_data = bin_data[bin_data[self.sample_type_col] == sample_type]
                        sample_total = len(self.data[self.data[self.sample_type_col] == sample_type])

                        if len(sample_bin_data) == 0 or sample_total == 0:
                            bin_result[f'N_{sample_type}'] = 0
                            bin_result[f'{sample_type}_iv'] = "0.0000"
                            bin_result[f'psi_{sample_type}'] = "0.0000"
                            continue

                        n_sample = len(sample_bin_data)
                        bin_result[f'N_{sample_type}'] = n_sample

                        # 计算该分箱的IV贡献
                        try:
                            good_count = (sample_bin_data[self.target_col] == 0).sum()
                            bad_count = (sample_bin_data[self.target_col] == 1).sum()

                            # 计算总体的好坏样本数
                            sample_type_data = self.data[self.data[self.sample_type_col] == sample_type]
                            total_good = (sample_type_data[self.target_col] == 0).sum()
                            total_bad = (sample_type_data[self.target_col] == 1).sum()

                            if total_good > 0 and total_bad > 0 and good_count > 0 and bad_count > 0:
                                good_rate = good_count / total_good
                                bad_rate = bad_count / total_bad
                                woe = np.log(bad_rate / good_rate)
                                iv_contrib = (bad_rate - good_rate) * woe
                                bin_result[f'{sample_type}_iv'] = f"{iv_contrib:.4f}"
                            else:
                                bin_result[f'{sample_type}_iv'] = "0.0000"
                        except:
                            bin_result[f'{sample_type}_iv'] = "0.0000"

                        # 计算PSI（以第一个样本类型为基准）
                        if sample_type == base_sample_type:
                            bin_result[f'psi_{sample_type}'] = "0.0000"
                        else:
                            try:
                                current_bin_ratio = n_sample / sample_total
                                if base_bin_ratio > 0 and current_bin_ratio > 0:
                                    psi_contrib = (current_bin_ratio - base_bin_ratio) * np.log(current_bin_ratio / base_bin_ratio)
                                    bin_result[f'psi_{sample_type}'] = f"{psi_contrib:.4f}"
                                else:
                                    bin_result[f'psi_{sample_type}'] = "0.0000"
                            except:
                                bin_result[f'psi_{sample_type}'] = "0.0000"

                    stability_results.append(bin_result)

            except Exception as e:
                print(f"特征 {feature} 稳定性分析失败: {e}")
                continue

        return pd.DataFrame(stability_results)

    def generate_feature_time_distribution(self, features: List[str],
                                         feature_names: Dict[str, str] = None,
                                         top_n: int = 10,
                                         bins: int = 5,
                                         bin_method: str = 'quantile',
                                         model: Any = None,
                                         importance_type: str = 'gain') -> pd.DataFrame:
        """
        生成单特征时间分布TOP10分析

        Args:
            features: 特征列名列表
            feature_names: 特征中文名映射
            top_n: 返回前N个特征
            bins: 分箱数量
            bin_method: 分箱方式，'quantile'(等频分箱)或'equal'(等距分箱)
            model: 可选，用于计算特征重要性的模型
            importance_type: 特征重要性类型

        Returns:
            特征时间分布分析结果
        """
        if 'Month' not in self.data.columns:
            raise ValueError("需要日期列来生成时间分布分析")

        if feature_names is None:
            feature_names = {f: f for f in features}

        # 获取月份列表
        months = sorted(self.data['Month'].unique())
        month_strs = [str(m) for m in months]

        # 计算特征重要性并选择TOP N
        feature_gains = self._calculate_feature_importance(
            features, model=model, importance_type=importance_type
        )

        top_features = sorted(feature_gains.items(), key=lambda x: x[1], reverse=True)[:top_n]

        distribution_results = []

        for feature, _ in top_features:
            if feature not in self.data.columns:
                continue

            # 对特征进行分箱
            try:
                self.data[f'{feature}_bin'] = self._create_bins(self.data[feature], bins=bins, bin_method=bin_method)
                bin_labels = self.data[f'{feature}_bin'].cat.categories
            except:
                continue

            # 按分箱和月份统计
            for bin_label in bin_labels:
                bin_data = self.data[self.data[f'{feature}_bin'] == bin_label]
                if len(bin_data) == 0:
                    continue

                # 创建结果字典，按指定顺序
                result = {}

                # 1. 数据类型(特征)和分箱
                result['特征'] = feature_names.get(feature, feature)
                result['score_bin'] = str(bin_label)

                # 按月统计数据
                monthly_data = {}
                total_num = 0
                total_bad = 0

                for month in months:
                    month_data = bin_data[bin_data['Month'] == month]
                    month_str = str(month)

                    num = len(month_data)
                    bad_num = (month_data[self.target_col] == 1).sum()
                    bad_ratio = bad_num / num if num > 0 else 0
                    lift = (bad_ratio / (self.data[self.target_col].mean())) if num > 0 else 0

                    monthly_data[month_str] = {
                        'num': num,
                        'bad_num': bad_num,
                        'bad_ratio': bad_ratio,
                        'lift': lift
                    }

                    total_num += num
                    total_bad += bad_num

                # 2. 各月的num (n列)
                for month in months:
                    month_str = str(month)
                    result[f'num_{month_str}'] = monthly_data[month_str]['num']

                # 3. 合计数量
                result['合计数量'] = total_num

                # 4. 各月的ratio (n列) - 该分箱在该月的样本数占全部样本总数的比例
                for month in months:
                    month_str = str(month)
                    ratio = monthly_data[month_str]['num'] / len(self.data) if len(self.data) > 0 else 0
                    result[f'ratio_{month_str}'] = f"{ratio:.4f}"

                # 5. 合计占比
                result['合计占比'] = f"{total_num / len(self.data):.4f}"

                # 6. 各月的bad_ratio (n列)
                for month in months:
                    month_str = str(month)
                    result[f'bad_ratio_{month_str}'] = f"{monthly_data[month_str]['bad_ratio']:.4f}"

                # 7. 各月的bad_num (n列)
                for month in months:
                    month_str = str(month)
                    result[f'bad_num_{month_str}'] = monthly_data[month_str]['bad_num']

                # 8. 各月的lift (n列)
                for month in months:
                    month_str = str(month)
                    result[f'lift_{month_str}'] = f"{monthly_data[month_str]['lift']:.4f}"

                distribution_results.append(result)

        return pd.DataFrame(distribution_results)

    def generate_model_score_distribution(self, bins: int = 10, bin_method: str = 'quantile') -> pd.DataFrame:
        """
        生成模型分时间分布分析（复用特征时间分布逻辑）

        Args:
            bins: 分箱数量
            bin_method: 分箱方式，'quantile'(等频分箱)或'equal'(等距分箱)

        Returns:
            模型分时间分布分析结果
        """
        if 'Month' not in self.data.columns:
            raise ValueError("需要日期列来生成时间分布分析")

        # 获取月份列表
        months = sorted(self.data['Month'].unique())
        month_strs = [str(m) for m in months]

        distribution_results = []

        # 对每个分数列分别进行分箱分析
        for sc in self.score_cols:
            # 对模型分进行分箱
            try:
                bin_col = f'{sc}_score_bin'
                self.data[bin_col] = self._create_bins(self.data[sc], bins=bins, bin_method=bin_method)
                bin_labels = self.data[bin_col].cat.categories
            except:
                continue

            # 多分数模式下，特征名用分数列名区分
            feature_label = sc if self.is_multi_score else '模型分'

            # 按分箱统计
            for bin_label in bin_labels:
                bin_data = self.data[self.data[bin_col] == bin_label]
                if len(bin_data) == 0:
                    continue

                # 创建结果字典，按指定顺序
                result = {}

                # 1. 数据类型和分箱
                result['特征'] = feature_label
                result['score_bin'] = str(bin_label)

                # 按月统计数据
                monthly_data = {}
                total_num = 0
                total_bad = 0

                for month in months:
                    month_data = bin_data[bin_data['Month'] == month]
                    month_str = str(month)

                    num = len(month_data)
                    bad_num = (month_data[self.target_col] == 1).sum()
                    bad_ratio = bad_num / num if num > 0 else 0
                    lift = (bad_ratio / (self.data[self.target_col].mean())) if num > 0 else 0

                    monthly_data[month_str] = {
                        'num': num,
                        'bad_num': bad_num,
                        'bad_ratio': bad_ratio,
                        'lift': lift
                    }

                    total_num += num
                    total_bad += bad_num

                # 2. 各月的num (n列)
                for month in months:
                    month_str = str(month)
                    result[f'num_{month_str}'] = monthly_data[month_str]['num']

                # 3. 合计数量
                result['合计数量'] = total_num

                # 4. 各月的ratio (n列) - 该分箱在该月的样本数占全部样本总数的比例
                for month in months:
                    month_str = str(month)
                    ratio = monthly_data[month_str]['num'] / len(self.data) if len(self.data) > 0 else 0
                    result[f'ratio_{month_str}'] = f"{ratio:.4f}"

                # 5. 合计占比
                result['合计占比'] = f"{total_num / len(self.data):.4f}"

                # 6. 各月的bad_ratio (n列)
                for month in months:
                    month_str = str(month)
                    result[f'bad_ratio_{month_str}'] = f"{monthly_data[month_str]['bad_ratio']:.4f}"

                # 7. 各月的bad_num (n列)
                for month in months:
                    month_str = str(month)
                    result[f'bad_num_{month_str}'] = monthly_data[month_str]['bad_num']

                # 8. 各月的lift (n列)
                for month in months:
                    month_str = str(month)
                    result[f'lift_{month_str}'] = f"{monthly_data[month_str]['lift']:.4f}"

                distribution_results.append(result)

        return pd.DataFrame(distribution_results)

    def generate_full_report(self, features: List[str] = None,
                           feature_names: Dict[str, str] = None,
                           save_path: str = None,
                           bin_method: str = 'quantile',
                           model: Any = None,
                           importance_type: str = 'gain',
                           experiment_meta: Optional[Dict[str, Any]] = None,
                           n_bins: int = 10) -> Dict[str, pd.DataFrame]:
        """
        生成完整的模型交付报告

        Args:
            features: 特征列名列表
            feature_names: 特征中文名映射
            save_path: 保存路径（可选）
            bin_method: 分箱方式，'quantile'(等频分箱)或'equal'(等距分箱)
            model: 可选，用于计算特征重要性的模型（支持LightGBM/XGBoost/sklearn模型）
            importance_type: 特征重要性类型，'gain'/'split'/'weight'（仅对树模型有效）
            experiment_meta: 实验元信息（可选），用于记录实验ID、参数、路径等
            n_bins: 模型分及Lift分箱数量，例如20分箱对应TOP5_lift

        Returns:
            包含所有报告表格的字典
        """
        if not isinstance(n_bins, int) or isinstance(n_bins, bool) or n_bins <= 0:
            raise ValueError(f"n_bins 必须为正整数，收到: {n_bins}")

        report = {}

        # 0. 实验元信息（用于参数-评估追踪）
        if experiment_meta:
            meta_rows = []
            for key, value in experiment_meta.items():
                if isinstance(value, (dict, list, tuple)):
                    value = str(value)
                meta_rows.append({'字段': str(key), '值': value})
            report['实验信息'] = pd.DataFrame(meta_rows)

        try:
            # 1. 样本情况
            report['样本情况'] = self.generate_sample_summary()
            print("✓ 样本情况统计完成")
        except Exception as e:
            print(f"✗ 样本情况统计失败: {e}")
            report['样本情况'] = pd.DataFrame()

        try:
            # 2. 模型效果
            report['模型效果'] = self.generate_model_performance(n_bins=n_bins)
            print("✓ 模型效果分析完成")
        except Exception as e:
            print(f"✗ 模型效果分析失败: {e}")
            report['模型效果'] = pd.DataFrame()

        try:
            # 3. 模型效果分箱
            report['模型效果分箱'] = self.generate_score_bins_analysis(
                bins=n_bins, bin_method=bin_method
            )
            print("✓ 模型效果分箱分析完成")
        except Exception as e:
            print(f"✗ 模型效果分箱分析失败: {e}")
            report['模型效果分箱'] = pd.DataFrame()

        try:
            # 3.5 模型评估指标报告（metric_report_plot的图表版本）
            metric_plot_result = self.generate_metric_report_plot(n_bins=n_bins)
            # report['评估指标汇总'] = pd.DataFrame(metric_plot_result['metrics_summary'])
            if metric_plot_result['lift_details']:
                report['Lift分箱详情'] = pd.concat(metric_plot_result['lift_details'], ignore_index=True)
            else:
                report['Lift分箱详情'] = pd.DataFrame()
            # 保存图片路径供后续插入Excel
            report['_metric_images'] = metric_plot_result['images']
            print("✓ 评估指标报告生成完成")
        except Exception as e:
            print(f"✗ 评估指标报告生成失败: {e}")
            # report['评估指标汇总'] = pd.DataFrame()
            report['Lift分箱详情'] = pd.DataFrame()
            report['_metric_images'] = {}

        if features:
            try:
                # 4. TOP10变量有效性
                report['TOP10变量有效性'] = self.generate_top_features_effectiveness(
                    features, feature_names, bin_method=bin_method,
                    model=model, importance_type=importance_type
                )
                print("✓ TOP10变量有效性分析完成")
            except Exception as e:
                print(f"✗ TOP10变量有效性分析失败: {e}")
                report['TOP10变量有效性'] = pd.DataFrame()

            try:
                # 4-1. 按月拆分的变量有效性（以test为例）
                report['变量有效性按月拆分'] = self.generate_top_features_effectiveness_monthly(
                    features, feature_names, bin_method=bin_method,
                    model=model, importance_type=importance_type
                )
                print("✓ 变量有效性按月拆分分析完成")
            except Exception as e:
                print(f"✗ 变量有效性按月拆分分析失败: {e}")
                report['变量有效性按月拆分'] = pd.DataFrame()

            try:
                # 5. TOP10变量稳定性
                report['TOP10变量稳定性'] = self.generate_top_features_stability(
                    features, feature_names, bin_method=bin_method,
                    model=model, importance_type=importance_type
                )
                print("✓ TOP10变量稳定性分析完成")
            except Exception as e:
                print(f"✗ TOP10变量稳定性分析失败: {e}")
                report['TOP10变量稳定性'] = pd.DataFrame()

            try:
                # 6. 单特征时间分布TOP10
                report['单特征时间分布TOP10'] = self.generate_feature_time_distribution(
                    features, feature_names, bin_method=bin_method,
                    model=model, importance_type=importance_type
                )
                print("✓ 单特征时间分布分析完成")
            except Exception as e:
                print(f"✗ 单特征时间分布分析失败: {e}")
                report['单特征时间分布TOP10'] = pd.DataFrame()

        try:
            # 7. 模型分表现
            report['模型分表现'] = self.generate_model_score_distribution(
                bins=n_bins, bin_method=bin_method
            )
            print("✓ 模型分表现分析完成")
        except Exception as e:
            print(f"✗ 模型分表现分析失败: {e}")
            report['模型分表现'] = pd.DataFrame()

        # 保存报告
        if save_path:
            try:
                from openpyxl import Workbook
                from openpyxl.drawing.image import Image as OpenpyxlImage
                from openpyxl.styles import PatternFill
                import os

                # 定义蓝白交替填充颜色
                blue_fill = PatternFill(start_color="CCE5FF", end_color="CCE5FF", fill_type="solid")
                white_fill = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")

                # 获取图片信息
                metric_images = report.pop('_metric_images', {})

                with pd.ExcelWriter(save_path, engine='openpyxl') as writer:
                    # 先写入所有数据表格
                    for sheet_name, df in report.items():
                        if not df.empty and not sheet_name.startswith('_'):
                            # 转换数字字符串为真正的数字
                            df_converted = self._convert_numeric_strings(df)
                            df_converted.to_excel(writer, sheet_name=sheet_name, index=False)
                            
                            # 对涉及分箱的表格应用蓝白交替填充
                            self._apply_alternating_fill(
                                writer.sheets[sheet_name], 
                                df, 
                                blue_fill, 
                                white_fill
                            )

                    # 为每个样本类型的图片创建单独的sheet
                    for sample_type, img_path in metric_images.items():
                        if os.path.exists(img_path):
                            sheet_name = f'{sample_type}_评估图表'
                            # 创建一个空的sheet
                            empty_df = pd.DataFrame({'': ['']})
                            empty_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False)

                            # 获取worksheet并插入图片
                            ws = writer.sheets[sheet_name]
                            img = OpenpyxlImage(img_path)
                            # 调整图片大小（可选）
                            img.width = 1200
                            img.height = 720
                            ws.add_image(img, 'A1')

                print(f"✓ 报告已保存至: {save_path}")

                # 清理临时图片文件
                for img_path in metric_images.values():
                    try:
                        if os.path.exists(img_path):
                            os.remove(img_path)
                    except:
                        pass

            except Exception as e:
                print(f"✗ 报告保存失败: {e}")
                import traceback
                traceback.print_exc()

        # 移除内部使用的图片路径信息
        report.pop('_metric_images', None)

        return report


def create_sample_data():
    """创建示例数据用于测试"""
    np.random.seed(42)

    n_samples = 10000
    months = pd.date_range('2023-01-01', '2023-12-01', freq='MS')

    data = []
    for i in range(n_samples):
        month = np.random.choice(months)
        sample_type = np.random.choice(['train', 'test', 'oot'], p=[0.6, 0.2, 0.2])

        # 生成特征
        feature1 = np.random.normal(0, 1)
        feature2 = np.random.normal(0, 1)
        feature3 = np.random.normal(0, 1)

        # 生成目标变量（与特征相关）
        prob = 1 / (1 + np.exp(-(0.5 * feature1 + 0.3 * feature2 - 0.2 * feature3)))
        target = np.random.binomial(1, prob)

        # 生成模型分数
        score = 0.5 * feature1 + 0.3 * feature2 - 0.2 * feature3 + np.random.normal(0, 0.1)

        data.append({
            'date': month,
            'sample_type': sample_type,
            'target': target,
            'score': score,
            'feature1': feature1,
            'feature2': feature2,
            'feature3': feature3
        })

    return pd.DataFrame(data)


In [ ]:
self, data: pd.DataFrame, target_col: str = 'target',
                 score_col: Union[str, List[str]] = 'score', date_col: str = 'date',
                 sample_type_col: str = 'sample_type',
                 score_na_strategy: str = 'intersection',
                 score_direction: Union[str, Dict[str, str]] = 'higher_is_bad'):
        """

In [167]:
df_scores.columns

Index(['serie_no', 'rn', 'cif_cust_no', 'make_loan_date', '姓名', '身份证', '手机号',
       '蜜蜂分B_062', '蜜蜂分A_064', '蜜蜂分B_054', '蜜蜂分B_059', '蜜蜂分B_070',
       'xyd_modelHIRR1_1_20250616', 'xyd_modelMIRR2_20251103', 'flag', 'month',
       'partner_name'],
      dtype='object')

In [168]:
df_scores['part_flag'] = '华通'
df_scores.head()

,serie_no,rn,cif_cust_no,make_loan_date,姓名,身份证,手机号,蜜蜂分B_062,蜜蜂分A_064,蜜蜂分B_054,蜜蜂分B_059,蜜蜂分B_070,xyd_modelHIRR1_1_20250616,xyd_modelMIRR2_20251103,flag,month,partner_name,part_flag
0,BH000001,4,100249100519,2026-03-06,474deb1a1a00f2df6fe1426af7aec00d,1e0968a3c477e2a4e5a5d68fb3712d74,fa4fc24bb7eb64ad38befd4c8bb753ae,523,586,663,587,494,17.011297,11.361520,1.0,202603,华通-2026-03,华通
1,BH000002,106,100109734289,2026-03-23,49121e96fad562a122ebf2d55106a367,3de08d3162b2aed10933cc74165808b7,b821b26916d2c64ff89872f40d858257,585,658,705,649,609,12.167068,6.765089,1.0,202603,华通-2026-03,华通
2,BH000003,114,100253118290,2026-04-05,812d0e8e2ba9281155307138861e674f,89d5fd9ad9ea658b847d6f449908a674,11b515f674b7a9639e1efd47acaddd63,503,510,623,539,595,23.592120,16.525286,1.0,202604,华通-2026-04至05,华通
3,BH000004,120,100220083085,2026-04-08,acc5115681d105bef92fdfe98b9d0ebb,4e861b04b6721d7c9e85421d2aa86018,4a18e4983795edc5c2d5a5d55f94dfbe,584,685,725,665,617,10.662794,5.963670,1.0,202604,华通-2026-04至05,华通
4,BH000005,176,100247676250,2026-02-28,04216d6f3d6e724010e5212c56b4a537,a343b5d96e944690c34ada406b395a4f,6079d651e4d5f0b422a5de7ed38b7410,539,646,693,585,553,12.889217,9.128659,1.0,202602,华通-2026-02,华通


In [169]:
mdrp  = ModelDeliveryReport(
    df_scores,
    'flag',
    ['蜜蜂分B_062',	'蜜蜂分A_064',	'蜜蜂分B_054',	'蜜蜂分B_059',	'蜜蜂分B_070'],
    'make_loan_date',
    'part_flag',
    score_direction = 'higher_is_good'
)
mdrp.generate_full_report(save_path='report_lr/all_report_new.xlsx',n_bins=20)

✓ 样本情况统计完成
✓ 模型效果分析完成
✓ 模型效果分箱分析完成
✓ 评估指标报告生成完成
✓ 模型分表现分析完成
✓ 报告已保存至: report_lr/all_report_new.xlsx


{'样本情况':      Month    Good   Bad   Total Bad_Rate
 0  2026-02   10755   496   11251   0.0441
 1  2026-03   59876  1966   61842   0.0318
 2  2026-04   54317  1197   55514   0.0216
 3  2026-05     161     1     162   0.0062
 4       合计  125109  3660  128769   0.0284,
 '模型效果':   样本类型    Good   Bad   Total Bad_Rate KS_蜜蜂分B_062 AUC_蜜蜂分B_062  \
 0  all  125109  3660  128769   0.0284      0.1156       0.5789   
 1   华通  125109  3660  128769   0.0284      0.1156       0.5789   
 
   TOP5_lift_蜜蜂分B_062 PSI_蜜蜂分B_062 KS_蜜蜂分A_064  ... TOP5_lift_蜜蜂分B_054  \
 0             1.7411       0.0000      0.1271  ...             1.8247   
 1             1.7411       0.0000      0.1271  ...             1.8247   
 
   PSI_蜜蜂分B_054 KS_蜜蜂分B_059 AUC_蜜蜂分B_059 TOP5_lift_蜜蜂分B_059 PSI_蜜蜂分B_059  \
 0       0.0000      0.1197       0.5827             1.8188       0.0000   
 1       0.0000      0.1197       0.5827             1.8188       0.0000   
 
   KS_蜜蜂分B_070 AUC_蜜蜂分B_070 TOP5_lift_蜜蜂分B_070 PSI_蜜蜂分B_070  
 0    